# TOPIX500：BB・FT/PTインフレ感応度とFama–French型ファクター

**原入力 → BB/FT/PT再推定 → 個別株感応度 → 時点別ソート → ファクター会計 → 形成後検証を、この1本で実行します。**

元の `main(1).ipynb` のBB/FTPT関数とHAC・偏相関・BH補正を流用しました。Pythonコードはすべて本Notebookに内包しています。元Notebookの実行、別の自作Pythonモジュール、外部データAPIは不要です。原データは手元のファイルから読み込みます。

## 最初に行うこと

1. **「1. 設定」**の `PROJECT_DIR` を、既存の `inputs/` があるプロジェクトへ設定します。
2. `STOCK_CFG` で株式リターン、時価総額、TOPIX500構成履歴のファイル名・列名を指定します。CSV / Excel / Parquet、縦持ち / 横持ちに対応します。
3. 必要に応じ市場・現金・業種・マクロ公表カレンダーを指定します。入力を省略した場合の挙動は下表のとおりです。
4. カーネルを再起動し、上から全セルを実行します。結果は `RESULT` とNotebook内の図表に残ります。

| 項目 | 初期設定・実装上の意味 |
|---|---|
| マクロ | 元の日本BB・FT/PTの式とNewsOnly定義を維持。CPIの項目定義も元入力に従う |
| 主ファクター | 総合、コストプッシュ、労働需給。主仕様は縮小β、比較は生β・偏相関 |
| 感応度 | 同時12か月リターンを四半期末で観測。最低40四半期、拡大型OLS、HAC 4/8四半期 |
| 時点管理 | 各過去四半期まで原系列を切断してSTL・BBを再推定。現在ビンテージによる疑似時点分析 |
| マクロ公表日 | 未指定時は四半期末から3か月後の月末を仮定。実際の全入力利用可能日をCSV等で置換可能 |
| 銘柄分類 | 新しい四半期モデルの利用開始月末に、サイズ中央値×感応度30/70%点の独立2×3 |
| ウェイト | 分類を次回まで保持し、月末時価総額と既知の構成銘柄により月次更新。新規構成銘柄の追加は次回のβ形成時 |
| 会計 | 前月末情報で翌月を保有する月次研究会計。日次入力でも1〜2営業日後の約定は再現しない |
| 投資額 | 担保資本1、ロング100%・ショート100%（グロス200%、ネット0%） |
| 売買費用 | 片道10bp × ドリフト後からの売買金額。初期建玉を含み、費用は月末に現金控除する近似 |
| 借株料 | 初期値は年率0bpの仮定。`borrow_bps_annual` で変更。終端での一括清算費用は含まない |
| 現金 | 未入力時は `zero_explicit` により月次0を明示仮定。金利を反映する場合は現金系列を指定 |
| 市場 | 未入力時は市場診断を省略。市場中立版を有効にするには市場系列が必要 |
| 保存 | 新しいrunフォルダにCSV・入力ハッシュ・設定を出力。元データ・既存Notebook・元のLATEST.jsonを変更しない |

**この納品物の検証は合成データによるものです。実際のBB入力・TOPIX500データは今回添付されていないため、実証的な損益や感応度の数値は収録していません。**

## 0. 実行環境

Python 3.10以降。基本依存は `numpy pandas scipy statsmodels matplotlib ipython`、Jupyter実行には `ipykernel` が必要です。Excelは `openpyxl`（古い `.xls` は `xlrd`）、Parquetは `pyarrow` を追加します。

必要な場合のみ、ご自身の実行環境で以下を実行してください。本Notebookは自動でインストールしません。

```bash
uv pip install numpy pandas scipy statsmodels matplotlib ipykernel openpyxl pyarrow
```

動作検証環境：Python 3.12。numpy 2.3 / pandas 2.2系、およびnumpy 2.5 / pandas 3.1 / scipy 1.18 / statsmodels 0.15系で主要処理を確認しています。

In [ ]:
import importlib.util
required = ['numpy', 'pandas', 'scipy', 'statsmodels', 'matplotlib', 'IPython']
missing = [p for p in required if importlib.util.find_spec(p) is None]
if missing:
    raise ImportError('不足パッケージ: '+', '.join(missing)+'。上の実行環境の説明に従って追加してください。')


## 1. 設定

ファイルパス・列名・頻度・単位を最初に合わせてください。推定窓、コスト、任意の中立化もこのセルで指定します。

In [ ]:
from pathlib import Path
import copy

# ===== 最初に変更する箇所 =====
PROJECT_DIR = Path.cwd()  # 例: Path('/Users/yourname/research/BB_PA_QualityGate_Ablation')
AS_OF = None             # None = 日本の実行日。固定する場合は '2026-11-11' 等
STOCK_DIR = PROJECT_DIR / 'inputs' / 'stocks'
OUTPUT_ROOT = PROJECT_DIR / 'outputs_topix500_inflation'

RUN_SELF_CHECKS = True
RUN_ANALYSIS = True
SAVE_RESULTS = True
WRITE_EMPTY_INPUT_TEMPLATES = False  # Trueにした場合だけヘッダーのみ作成。既存ファイルは上書き不可

MACRO_CFG = {
    'input_dir': str(PROJECT_DIR / 'inputs'),
    'input_mode': 'provider',       # 'provider' / 'canonical_csv'
    'as_of': AS_OF,
    'max_end_quarter': None,
    'bb_start': '1996Q2',
    'news_history_start': '1997Q1',
    'news_mean_start': '1990Q1',
    'approved_cpi_base': '2025',
    'bb_cfg': {},                   # 元BB設定から変える場合だけ指定
    'provider_files': {},
    'release_calendar': None,       # 下の例で実際の利用可能日を指定可能
    'assumed_release_lag_months': 3,
    'max_model_age_months': 6,
}
# 例：全原料を使った四半期モデルの利用可能日
# MACRO_CFG['release_calendar'] = {
#     'path': str(STOCK_DIR / 'macro_release_calendar.csv'),
#     'columns': {'quarter': 'quarter', 'available_at': 'available_at'},
# }

STOCK_CFG = {
    'as_of': AS_OF,
    'returns': {
        'path': str(STOCK_DIR / 'returns.csv'),
        'format': 'auto', 'layout': 'long',
        'frequency': 'monthly',      # 'daily' も可。損益計算は月次に集計
        'kind': 'returns',           # 'returns' / 'tr_levels'
        'unit': 'decimal',           # 0.01=1%。'percent'なら1=1%
        'currency': 'JPY', 'return_basis': 'total_return_including_dividends',
        'encoding': 'utf-8-sig', 'sheet_name': 0,
        'columns': {'date': 'date', 'security_id': 'security_id', 'value': 'return'},
    },
    'caps': {
        'path': str(STOCK_DIR / 'market_caps.csv'),
        'format': 'auto', 'layout': 'long',
        'columns': {'date': 'date', 'security_id': 'security_id', 'value': 'market_cap',
                    'available_at': 'available_at'},
        'max_age_days': 62, 'availability_lag_days': 0,
        'definition': 'total_market_cap', 'currency': 'JPY',
    },
    'membership': {
        'path': str(STOCK_DIR / 'membership.csv'),
        'format': 'auto', 'mode': 'intervals',
        'columns': {'security_id': 'security_id', 'effective_from': 'effective_from',
                    'effective_to': 'effective_to', 'available_at': 'available_at'},
    },
    'sectors': None,
    'market': None,
    'cash': None,
    'cash_missing_policy': 'zero_explicit', # 実金利系列を使用するならcashを設定
    'trading_calendar': None,
    'id_pad_length': None,
}
# 任意系列の設定例（0.01=1%の月次リターン）
# STOCK_CFG['market'] = {'path': str(STOCK_DIR / 'market.csv'), 'layout': 'long',
#     'frequency': 'monthly', 'kind': 'returns', 'unit': 'decimal',
#     'columns': {'date': 'date', 'value': 'return'}}
# STOCK_CFG['cash'] = {'path': str(STOCK_DIR / 'cash.csv'), 'layout': 'long',
#     'frequency': 'monthly', 'kind': 'returns', 'unit': 'decimal',
#     'columns': {'date': 'date', 'value': 'return'}}
# STOCK_CFG['sectors'] = {'path': str(STOCK_DIR / 'sectors.csv'), 'mode': 'dated',
#     'columns': {'date': 'date', 'security_id': 'security_id', 'sector': 'sector',
#                 'available_at': 'available_at'}}

SENSITIVITY_CFG = {
    'min_quarters': 40,
    'window_quarters': None,         # None=拡大型、40=直近40四半期
    'sample_start': None,
    'estimator': 'ols',              # 'ols' / 'wls'
    'half_life_quarters': 20,        # WLSのみ
    'return_basis': 'total',         # 元15節のlevel_returnを継承。'excess_cash'も可
    'control_market': False,
    'min_cross_section': 5,
    'max_last_observation_gap_quarters': 1,
    'condition_warning': 100,
    'min_residual_news_sd_pp': 1e-8,
}
FACTOR_SPECS = {
    'total': {'target': 'news_total', 'controls': ['news_growth']},
    'cost_push': {'target': 'news_cost_push', 'controls': ['news_labor', 'news_growth']},
    'labor': {'target': 'news_labor', 'controls': ['news_cost_push', 'news_growth']},
}
# 詳細要因を追加する場合はtargetとcontrolsを定義します。
# CPとenergy/food/shortage、laborとlabor_ft/labor_ptを同時に全て投入しないでください。

FACTOR_CFG = {
    'start_month': None, 'end_month': None,
    'score_methods': ['beta_shrunk', 'beta_raw', 'partial_corr'],
    'score_quantiles': [0.30, 0.70],
    'min_cell_size': 5,
    'gross_target': 2.0,
    'cost_bps_one_way': 10.0,
    'borrow_bps_annual': 0.0,
    'neutral_variants': [],          # 任意: ['sector', 'market', 'sector_market']
    'market_beta_window_months': 60,
    'market_beta_min_months': 36,
}
DIAGNOSTICS_CFG = {'min_quarters': 12, 'bootstrap_reps': 399,
                   'block_length': 8, 'seed': 20261111}
CONFIG = {'macro': MACRO_CFG, 'stocks': STOCK_CFG, 'sensitivity': SENSITIVITY_CFG,
          'factor': FACTOR_CFG, 'factor_specs': FACTOR_SPECS,
          'diagnostics': DIAGNOSTICS_CFG, 'data_label': 'user_provided'}


## 2. 入力ガイド

### 2.1 マクロ入力

## BB・FT/PTマクロ入力と再利用する計算

このノートブックには、添付の`main(1).ipynb`から再利用した原系列の読み込み、STL、制約付きBB推定、FT/PTの連立計算、歴史分解、NewsOnly作成を内包しています。元ノートブックの事前実行や別の自作Pythonファイルは不要です。原データの`inputs/`フォルダを同じ構成で用意し、`MACRO_CFG["input_dir"]`に指定してください。入力ファイルを変更・上書きする処理はありません。

### 必要な入力

両モードで、`source_metadata.csv`、`static/canonical_schema.json`、`static/google_calibration_target.csv`を使用します。CPIの基準年は2025、Google系列の`shortage.complete_through`は最後の完了月を`YYYY-MM`で指定します。

- `provider`：`provider/`内の原15ファイルを使用します。CPI3本、毎月勤労2本、労働力調査2本、求人・就職2本、GDP4本、企業インフレ期待、Google供給不足です。具体的なファイル名は`BB_PROVIDER_FILES`にあり、設定で変更できます。
- `canonical_csv`：`canonical_schema.json`に定義された各groupの`csv/<group>.csv`を使用します。元プログラムが出力する原系列CSVと同じ列・頻度を指定してください。計算済みのBBニュースを原系列の代わりに入れる形式ではありません。

`macro_required_paths(MACRO_CFG)`で必要なパスを確認できます。欠損月・不完全四半期・系列定義の不一致を検査し、未知の欠損をゼロで埋めません。

### 形成時点ごとのモデル版

```python
engine = load_macro_inputs(MACRO_CFG)
version = build_macro_version(engine, engine.latest_quarter)
news = version["news"]
```

`engine.earliest_quarter`から`latest_quarter`まで、指定四半期で原系列を切断し、STLからFT/PTまで再推定します。ニュースは「要因寄与の4期平均の4期差」で、調査予想サプライズは加えません。`model_version`と`provenance`に推定終期・使用月・出典を保存します。統計の公表日時と売買日時は、別の形成スケジュールで管理します。

元仕様からの時点処理上の変更は、税調整CPIのアンカーを過去モデルの終月以前に制限した点と、Google尺度較正に2004Q1–2010Q4を使うため最早モデル期を2010Q4以降に制限した点です。FT/PTの人数ウェイト、回帰式、寄与の加法性は維持しています。

### 検証範囲と解釈

過去の切断時点でも、入力する原系列は現在の改定済みデータです。将来の観測期間は使いませんが、過去の公表値を復元した厳密なリアルタイム検証ではありません。CPI基準改定やGoogle尺度などの影響も残るため、出力に`recursive_current_vintage_not_real_time`を記録します。

合成原系列で、読み込みからニュース作成まで、元関数との数値一致、将来期間の変更に対する過去版の不変性、入力の非変更を検証しています。利用者の実ファイルは未添付のため、実際の原統計ファイルの表形式と実証値は未検証です。

### 2.2 株式入力

## 株式ファイルの指定と入力仕様

冒頭の `STOCK_CFG` でファイルパスと列名を指定します。元データは変更しません。株式リターン、時価総額、構成履歴の3ファイルが必須です。空のテンプレートは入力仕様を示すだけで、分析用データにはなりません。

| ファイル | 標準列と意味 |
|---|---|
| `returns.csv` | `date, security_id, return`。リターンは `unit='decimal'` なら0.01＝1%、`percent`なら1＝1%。配当込みの系列を指定します。 |
| `market_caps.csv` | `date, security_id, market_cap, available_at`。観測日・銘柄・正の時価総額・利用可能日。総時価総額か浮動株時価総額か、通貨を統一します。 |
| `membership.csv` | `security_id, effective_from, effective_to, available_at`。開始日は含み、終了日は含みません。継続中の終了日は空欄です。 |
| `membership_snapshots.csv` | 代替形式の `date, security_id, member, available_at`。各日付の完全な構成銘柄一覧が必要です。 |
| `sectors.csv` | 任意。`date, security_id, sector, available_at`。歴史的業種変更を持つ場合は `mode='dated'`。 |
| `market.csv`, `cash.csv` | 任意。`date, return`。TOPIX配当込み収益率と現金収益率など。 |

`columns` は「標準名: 手元の列名」です。例えば列名が `年月, 銘柄コード, 配当込み指数` なら、次のように指定します。

```python
STOCK_CFG['returns'].update({
    'path': str(PROJECT_DIR / 'inputs/stocks/stock_data.xlsx'),
    'sheet_name': 'TR', 'layout': 'long', 'frequency': 'monthly',
    'kind': 'tr_levels',
    'columns': {'date': '年月', 'security_id': '銘柄コード', 'value': '配当込み指数'},
})
```

`kind='returns'` は単純リターン、`tr_levels` は配当込み指数水準です。銘柄が列に並ぶ横持ち表は `layout='wide'` とし、日付列を指定します。対象外の列は `ignore_columns` に列名を列挙します。IDは文字列として扱い、Excel由来の `1234.0` は `1234` に統一します。元から数値として保存されたIDの桁数が必要なら `id_pad_length` を指定してください。

時価総額は、形成日以前に観測され、かつ `available_at` も形成日以前の最新値を使います。既定では62日より古い値を使いません。利用可能日を省略した場合の仮定は監査結果へ記録します。snapshot形式を使う場合は `mode='snapshots', complete_snapshots=True` を指定し、全行が利用可能になるまで新しいsnapshotを使いません。

`available_at` を省略する場合は列自体を削除してください。列を残す場合は全行に利用可能日が必要で、空欄の行はエラーになります。列がない場合、時価総額は `date + availability_lag_days`、構成区間は `effective_from`、snapshotは `date` を利用可能日と仮定します。

日次入力は `frequency='daily'` で月次化できます。正確な欠損・完了月判定には `trading_calendar` で取引日一覧を指定してください。省略時は観測日の和集合を使うため、全銘柄に共通する欠落日は検出できません。リターンやTR指数を前方補完せず、月中欠損を監査します。実際の大幅変動を±50%等で自動削除せず、−100%の終端損失も保持します。

CSVは追加依存不要です。Excelには `openpyxl`（古いxlsは `xlrd`）、Parquetには `pyarrow` または `fastparquet` が必要です。設定したファイルがなければ、不足ファイルをまとめて表示して停止します。現金を省略して `cash_missing_policy='zero_explicit'` とした場合は、ゼロ金利の研究上の仮定として監査に残ります。市場リターンの未入力はNaNで保持し、市場診断では使用しません。

## 3. 推定・構築・会計の定義

### 3.1 既存15節との対応

BB要因別寄与を $c_{k,q}$ とすると、既存のNewsOnlyは

$$N_{k,q}=\frac14\sum_{j=0}^{3}c_{k,q-j}-\frac14\sum_{j=0}^{3}c_{k,q-4-j}.$$

各銘柄の過去12か月複利リターンを四半期末で観測し、総合は `news_total + news_growth`、CPと労働需給は `news_cost_push + news_labor + news_growth` に回帰します。`stock_sensitivity_main`には、元`dual_main`に対応する6要因の個別回帰、`stock_sensitivity_joint`には主ファクター用の条件付きβを格納します。FT/PTは共通労働需給入力の伝播経路であり、独立ショックとは呼びません。

元の `ols_hac`, `partial_corr`, `bh_adjust` を名前衝突を避ける `sf_` 接頭辞付きで流用しています。欠損四半期をまたいで隣接行を隣接期と扱わないHACも維持します。個別株については銘柄別標本を使い、全銘柄共通dropnaや±50%超の自動除外は適用しません。

### 3.2 縮小βと比較仕様

$$\widetilde\beta_i=\mu+\frac{v_{CS}}{v_{CS}+SE_{i,HAC8}^2}(\widehat\beta_i-\mu).$$

$\mu$ と $v_{CS}$ は形成時点の適格銘柄の生βの横断平均・標本分散です。これは単純なVasicek型経験的priorであり、相関した推定誤差を除去して真のβ分散を識別したものではありません。生β、偏相関も同じ構築ルールで比較します。個別p値で銘柄を足切りしません。表示するp値・HAC区間・BH補正は生の回帰係数に対するものです。

### 3.3 独立2×3ソート

サイズは適格TOPIX500内の時価総額中央値、スコアは同じ全適格集合の30/70%点です。各セルは時価総額加重です。これはTOPIX500向けのFF型設計であり、French公式日本系列の完全再現ではありません。

$$F_m=\frac12(R_{SH,m}-R_{SL,m})+\frac12(R_{BH,m}-R_{BL,m}).$$

同値スコアを銘柄コード順で人工的に分けません。6セルの最低銘柄数に達しない期間は、開始前なら理由を付けて未形成、開始後なら停止します。形成済みセルは次回モデル更新まで固定し、月次には既知の構成除外・時価総額を反映します。新規構成銘柄は次回のβ形成で採用対象になります。

### 3.4 担保・費用・12か月リターン

期首NAVを1とした株式ウェイト $w_{i,m}$ に対し、

$$R^{gross}_m=r^f_m+\sum_i w_{i,m}(r_{i,m}-r^f_m),$$
$$R^{net}_m=R^{gross}_m-c\sum_i|w_{i,m}-w^{drift}_{i,m}|-b\,\frac{\sum_i[-w_{i,m}]_+}{12}.$$

$c$は片道売買費用率、$b$は年率借株料です。ドリフトは前月の費用後NAVから計算します。費用を月末に現金から払う月次近似であり、ブローカーの証拠金・貸株リベート・日中売買は再現しません。12か月超過収益は、

$$R^{e,12m}_{F,m}=\prod_{j=0}^{11}(1+R^{net}_{m-j})-\prod_{j=0}^{11}(1+r^f_{m-j}).$$

単純に $\prod(1+F_m)-1$ とする系列や、長短各レッグの12か月複利差とは区別します。

### 3.5 形成後の検証

形成後のリターンだけを接続し、形成に使ったものと対応するマクロ統制で12か月感応度を推定します。最新ビンテージのニュースによる事後評価です。HAC 4/8四半期と、欠損区間をまたがない8四半期ブロックの条件付きbootstrapを出します。bootstrapはBB・銘柄選別自体を再推定するものではありません。全戦略共通期間の損益、Low/Neutral/Highの形成後β、市場β、因子間相関も確認します。

任意の業種・市場中立版は、原型ウェイトに対する最小距離の制約射影です。元のFF型とは別系列であり、事前の推定市場β・業種露出をゼロにしても、形成後の市場βやインフレβを保証しません。

## 4. BB・FT/PTエンジン

元関数を独立したクラスへ移植しています。長い定義セルは折りたたんであります。入力ファイルへの書き込みや、別Notebookの実行はありません。

In [ ]:
# BB / FTPT engine, reused from the attached main(1).ipynb.
# No file execution, network access, or writes to the original inputs directory.
from pathlib import Path
from dataclasses import dataclass
from io import BytesIO
import copy
import csv
import hashlib
import io
import json
import posixpath
import re
import zipfile
import xml.etree.ElementTree as ET
import numpy as np
import pandas as pd
from scipy import linalg, stats
from statsmodels.tsa.seasonal import STL


BB_SOURCE_NOTEBOOK_SHA256 = "dbec3aee67311226c92ddf117e931cca0c0c6947e5d6d5fc9af22b36662d3061"

BB_DEFAULTS = {
    "wage_trend":13, "stl_seasonal":13, "other_trend":19,
    "growth":"log", "price_productivity":True, "expectations":"FIRM5",
    "tax_anchor":"2019-12", "google_calibration_start":"2004Q1",
    "google_calibration_end":"2010Q4", "google_earthquake_start":"2011-03",
    "google_earthquake_end":"2011-06", "irf_horizon":20,
}
BB_PROVIDER_FILES = {
    "cpi_nsa":"cpi_nsa.csv", "cpi_sa":"cpi_sa.csv", "cpi_tax":"cpi_tax.xlsx",
    "labour_actual":"labour_actual.csv", "labour_index":"labour_index.csv",
    "lfs_main":"labour_force.xlsx", "lfs_nonagri":"labour_force_nonagri.xlsx",
    "job_vacancies":"job_vacancies.xlsx", "job_placements":"job_placements.xlsx",
    "gdp_real_sa":"gdp_real_sa.csv", "gdp_real_nsa":"gdp_real_nsa.csv",
    "gdp_nominal_sa":"gdp_nominal_sa.csv", "gdp_nominal_nsa":"gdp_nominal_nsa.csv",
    "expectations":"einf.csv", "shortage":"shortage.csv",
}


@dataclass(frozen=True)
class _BBEquationSpec:
    name: str
    terms: tuple
    homogeneity_groups: tuple
    constant: bool


def macro_required_paths(config):
    """Required source paths; read-only. Returns missing paths as well."""
    base = Path(config.get("input_dir", "inputs")).expanduser().resolve()
    paths = [base/"source_metadata.csv", base/"static/canonical_schema.json",
             base/"static/google_calibration_target.csv"]
    mode = config.get("input_mode", "provider")
    if mode == "provider":
        names = {**BB_PROVIDER_FILES, **config.get("provider_files", {})}
        paths += [base/"provider"/v for v in names.values()]
    elif mode == "canonical_csv":
        if paths[1].is_file():
            schema = json.loads(paths[1].read_text(encoding="utf-8-sig"))
            paths += [base/"csv"/(g+".csv") for g in schema]
    else:
        raise ValueError("macro.input_mode must be provider or canonical_csv")
    return paths


class BBFTPTMacroEngine:
    """Original BB equations and FT/PT transmission with prefix refitting.

    Historical prefixes use the currently supplied vintage of the raw data.
    They are NOT real-time historical-vintage estimates. All constants and
    original functions are isolated on this object, avoiding notebook globals.
    """

    NS = {'m':'http://schemas.openxmlformats.org/spreadsheetml/2006/main',
          'r':'http://schemas.openxmlformats.org/officeDocument/2006/relationships'}
    ENDOG = ["gw","gcpi","cf10","cf1"]
    EXOG = ["magpty","grpe","grpf","shortage","vu","dummyq2_2020","dummyq3_2020"]
    BASE_VARS = ["gw","gcpi","cf1","cf10","magpty","grpe","grpf","shortage","vu","diffcpicf"]
    PARTS = ["initial_conditions","energy","food","shortage","labor","productivity","covid_q2","covid_q3"] + ["residual_"+v for v in ENDOG]
    FACTORS = ["news_total","news_cost_push","news_labor","news_energy","news_food","news_shortage"]
    FTPT_EQ = ["gw_ft","gw_pt","gcpi","cf10","cf1"]
    FTPT_STATE = ["gw_ft","gw_pt","gw","gcpi","cf10","cf1"]
    FTPT_PARTS = ["initial_conditions","energy","food","shortage","labor","productivity","covid_q2","covid_q3"] + ["residual_"+v for v in FTPT_EQ]

    def __init__(self, config):
        self.config = copy.deepcopy(config)
        self.CFG = {**BB_DEFAULTS, **config.get("bb_cfg", {})}
        self.INPUT_DIR = Path(config.get("input_dir", "inputs")).expanduser().resolve()
        self.PROVIDER_FILES = {**BB_PROVIDER_FILES, **config.get("provider_files", {})}
        self.BB_START = str(config.get("bb_start", "1996Q2"))
        self.NEWS_HISTORY_START = str(config.get("news_history_start", "1997Q1"))
        self.NEWS_MEAN_START = str(config.get("news_mean_start", "1990Q1"))
        self.approved_cpi_base = str(config.get("approved_cpi_base", "2025"))
        if self.approved_cpi_base != "2025":
            raise ValueError("Reused CPI columns are 2025-base. A different base requires an explicit source-mapping update.")
        if self.CFG["expectations"] not in ("FIRM5", "FIRM10"):
            raise ValueError("Use FIRM5 or FIRM10; the original CIE path is not silently substituted.")
        self.long_year = 5 if self.CFG["expectations"]=="FIRM5" else 10
        as_of = config.get("as_of")
        self.as_of = pd.Timestamp(as_of) if as_of else pd.Timestamp.now(tz="Asia/Tokyo")
        if self.as_of.tzinfo is not None:
            self.as_of = self.as_of.tz_convert("Asia/Tokyo").tz_localize(None)
        self.as_of = self.as_of.normalize()
        self.last_month = self.as_of.to_period("M")-1
        self.last_quarter = self.last_month.asfreq("Q")
        if self.last_quarter.asfreq("M","end")>self.last_month:
            self.last_quarter -= 1
        self.PARSER_FLAGS=[]
        self.SOURCE_MAP={}
        self.alerts=[]
        self.checks=[]
        self.input_manifest={}
        self._versions={}
        self._load_inputs()

    def check(self, name, condition, detail=""):
        self.checks.append({"test":name,"passed":bool(condition),"detail":str(detail)})
        if not condition:
            raise ValueError(f"BB input/model check failed: {name}: {detail}")

    def alert(self, code, detail, severity="warning"):
        self.alerts.append({"code":code,"detail":str(detail),"severity":severity})

    def read_bytes(self, path, role):
        path=Path(path).resolve()
        if not path.is_file():
            raise FileNotFoundError(f"{role}: {path}")
        blob=path.read_bytes()
        self.input_manifest[role]={"path":str(path),"bytes":len(blob),
                                  "sha256":hashlib.sha256(blob).hexdigest()}
        return blob

    def _load_inputs(self):
        missing=[str(p) for p in macro_required_paths(self.config) if not p.is_file()]
        if missing:
            raise FileNotFoundError("BB raw inputs are missing. Set macro.input_dir to the original inputs/ directory.\n"+"\n".join(missing))
        metadata=pd.read_csv(BytesIO(self.read_bytes(self.INPUT_DIR/"source_metadata.csv","source_metadata")),
                             dtype=str, keep_default_na=False)
        if "role" not in metadata or metadata.role.duplicated().any():
            raise ValueError("source_metadata.csv needs a unique role column")
        self.source_meta=metadata.set_index("role").to_dict("index")
        for role in ("cpi_nsa","cpi_sa","cpi_tax"):
            if self.source_meta.get(role,{}).get("base_year")!=self.approved_cpi_base:
                raise ValueError(f"{role}: base_year must equal {self.approved_cpi_base}")
        for role, entry in self.source_meta.items():
            if entry.get("publication_date"):
                stamp=pd.Timestamp(entry["publication_date"])
                if stamp.tzinfo is not None:
                    stamp=stamp.tz_convert("Asia/Tokyo").tz_localize(None)
                if stamp.normalize()>self.as_of:
                    raise ValueError(f"{role}: source publication date is after macro.as_of")
        self.schema=json.loads(self.read_bytes(self.INPUT_DIR/"static/canonical_schema.json","canonical_schema").decode("utf-8-sig"))
        if not isinstance(self.schema,dict) or not self.schema:
            raise ValueError("canonical_schema.json must contain the original non-empty mapping")
        for group, entry in self.schema.items():
            if not isinstance(entry,dict) or not {"frequency","date_column","columns"}.issubset(entry):
                raise ValueError(f"Invalid canonical schema entry: {group}")
            if entry["frequency"] not in ("M","Q"):
                raise ValueError(f"Invalid canonical frequency: {group}")
            if Path(group).name!=group:
                raise ValueError("Canonical group names must be simple file basenames")
        mode=self.config.get("input_mode","provider")
        raw_m,raw_q=self.load_provider() if mode=="provider" else self.load_canonical(self.schema)
        if raw_m.empty or raw_q.empty:
            raise ValueError("Macro monthly/quarterly input table is empty")
        raw_m=raw_m.reindex(pd.period_range(raw_m.index.min(),raw_m.index.max(),freq="M",name="month"))
        raw_q=raw_q.reindex(pd.period_range(raw_q.index.min(),raw_q.index.max(),freq="Q",name="quarter"))
        self.raw_monthly=raw_m
        self.raw_quarterly=raw_q
        self.M=raw_m.loc[:self.last_month].copy()
        self.Q=raw_q.loc[:self.last_quarter].copy()
        complete=self.source_meta.get("shortage",{}).get("complete_through","")
        if not re.fullmatch(r"\d{4}-\d{2}",complete):
            raise ValueError("source_metadata shortage.complete_through requires YYYY-MM")
        self.google_last=min(pd.Period(complete,freq="M"),self.last_month)
        self.M.loc[self.M.index>self.google_last,"google_shortage"]=np.nan
        self._validate_raw_inputs()
        self.target=pd.read_csv(BytesIO(self.read_bytes(self.INPUT_DIR/"static/google_calibration_target.csv","google_calibration_target")),index_col=0)
        self.target.index=pd.PeriodIndex(self.target.index.astype(str),freq="Q")
        if self.target.index.has_duplicates or "reference_shortage" not in self.target:
            raise ValueError("Google calibration target needs unique quarterly rows and reference_shortage")
        cal_start=pd.Period(self.CFG["google_calibration_start"],freq="Q")
        cal_end=pd.Period(self.CFG["google_calibration_end"],freq="Q")
        if (str(cal_start),str(cal_end))!=("2004Q1","2010Q4"):
            raise ValueError("This reused reconstruction fixes Google calibration to 2004Q1–2010Q4")
        # Do not permit historical estimation before the calibration window ends.
        self.earliest_quarter=max(pd.Period(self.BB_START,freq="Q")+39, cal_end,
                                  pd.Period(self.NEWS_HISTORY_START,freq="Q")+7)
        if self.latest_quarter<self.earliest_quarter:
            raise ValueError("Not enough macro history after the fixed Google calibration interval")
        gq=self.qmean_complete(self.M.google_shortage.loc[:self.latest_quarter.asfreq("M","end")])
        pair=pd.concat({"new":gq,"old":self.target.reference_shortage},axis=1).loc[cal_start:cal_end].dropna()
        if len(pair)!=28 or not pair.index.equals(pd.period_range(cal_start,cal_end,freq="Q")):
            raise ValueError("Google calibration requires all 28 quarters in 2004Q1–2010Q4")
        den=float(pair.new@pair.new)
        if den<=0: raise ValueError("Google calibration observations are all zero")
        self.GOOGLE_SCALE=float((pair.new@pair.old)/den)
        self.GOOGLE_FILL=float(self.M.google_shortage.loc["2004-01":"2010-12"].mean())
        if not np.isfinite(self.GOOGLE_SCALE) or not np.isfinite(self.GOOGLE_FILL):
            raise ValueError("Google calibration produced a non-finite parameter")
        self.calibration=pair
        self.alert("current_vintage", "Prefix refits remove future periods, but CPI rebasing, revised raw data, Google scaling and the static calibration target remain current-vintage. These are not real-time backtests.","info")

    def _validate_raw_inputs(self):
        required_m=[
            "cpi_2025_nsa_all","cpi_2025_nsa_food","cpi_2025_nsa_energy","cpi_2025_sa_all",
            "labour_ft_scheduled_pay_index","labour_pt_scheduled_pay_index","labour_pt_scheduled_hours_index",
            "labour_all_scheduled_pay_index","labour_all_employment_index","labour_all_total_hours_index",
            "labour_ft_employees_start","labour_ft_employees_end","labour_pt_employees_start","labour_pt_employees_end",
            "lfs_sa_unemployed","lfs_sa_employees","jobs_sa_active_vacancies","jobs_sa_placements","google_shortage"]
        required_q=["gdp_real_sa_annualized","firm_expectation_1y",f"firm_expectation_{self.long_year}y"]
        if "cpi_taxadjusted_2025_all" not in self.M:
            raise ValueError("Missing tax-adjusted CPI series")
        rows=[]; ends=[]
        for freq,frame,required in (("M",self.M,required_m),("Q",self.Q,required_q)):
            for col in required:
                if col not in frame or frame[col].dropna().empty:
                    raise ValueError(f"Missing or empty raw macro series: {col}")
                series=frame[col]
                completed=self.qmean_complete(series) if freq=="M" else series
                last=completed.last_valid_index()
                if last is None: raise ValueError(f"No complete quarter: {col}")
                ends.append(last)
                values=series.dropna()
                rows.append({"series":col,"frequency":freq,"source_role":self.SOURCE_MAP.get(col,"unknown"),
                             "first":str(values.index[0]),"last":str(values.index[-1]),
                             "last_complete_quarter":str(last),"n":len(values)})
        self.latest_quarter=min([self.last_quarter]+ends)
        request=self.config.get("max_end_quarter")
        if request:
            request=pd.Period(request,freq="Q")
            if request>self.latest_quarter:
                raise ValueError(f"macro.max_end_quarter {request} exceeds available {self.latest_quarter}")
            self.latest_quarter=request
        self.BB_END=self.latest_quarter
        self.coverage=pd.DataFrame(rows)
        self.coverage["used_end"]=str(self.latest_quarter)
        for col in required_m:
            start=max(self.M[col].first_valid_index(),pd.Period("1993-01",freq="M"))
            sample=self.M.loc[start:self.latest_quarter.asfreq("M","end"),col]
            if sample.isna().any():
                raise ValueError(f"{col}: internal monthly gaps: {sample.index[sample.isna()].astype(str).tolist()[:12]}")
        for col in required_q:
            start=max(self.Q[col].first_valid_index(),pd.Period("1994Q1",freq="Q"))
            if self.Q.loc[start:self.latest_quarter,col].isna().any():
                raise ValueError(f"{col}: internal quarterly gaps")
        for col in [c for c in required_m if c!="google_shortage"]+["cpi_taxadjusted_2025_all"]:
            self.check("positive_"+col, self.M[col].dropna().gt(0).all())
        self.check("google_0_100", self.M.google_shortage.dropna().between(0,100).all())
        self.check("jobs_unfilled_positive",(self.M.jobs_sa_active_vacancies-self.M.jobs_sa_placements).dropna().gt(0).all())
        for edge in ("start","end"):
            all_col="labour_all_employees_"+edge
            if all_col in self.M:
                pair=pd.concat([self.M[all_col],self.M["labour_ft_employees_"+edge]+self.M["labour_pt_employees_"+edge]],axis=1).dropna()
                self.check("FT_PT_headcount_"+edge,not pair.empty and np.allclose(pair.iloc[:,0],pair.iloc[:,1],atol=1))
        for col in ("cpi_2025_nsa_all","cpi_2025_nsa_food","cpi_2025_nsa_energy"):
            values=self.raw_monthly.loc["2025-01":"2025-12",col].dropna()
            if len(values)==12:
                self.check("CPI_base_year_"+col,abs(values.mean()-100)<.15,str(values.mean()))
            else:
                self.alert("CPI_base_year_not_numerically_verified",col)

    def build_version(self, cutoff_q):
        q=pd.Period(cutoff_q,freq="Q")
        if not self.earliest_quarter<=q<=self.latest_quarter:
            raise ValueError(f"Macro cutoff {q} is outside {self.earliest_quarter}–{self.latest_quarter}")
        key=str(q)
        if key in self._versions:
            return self._versions[key]
        local=copy.copy(self)
        local.M=self.M.loc[:q.asfreq("M","end")].copy()
        local.Q=self.Q.loc[:q].copy()
        local.CFG=dict(self.CFG)
        local.CFG["tax_anchor"]=str(min(pd.Period(self.CFG["tax_anchor"],freq="M"),q.asfreq("M","end")))
        local.BB_END=q
        data,monthly,quarterly,construction_meta=local.build_reconstruction(str(q),growth_method=self.CFG["growth"])
        data["cf1"]=local.Q.firm_expectation_1y.reindex(data.index)
        data["cf10"]=local.Q[f"firm_expectation_{self.long_year}y"].reindex(data.index)
        data["diffcpicf"]=data.gcpi.rolling(4,min_periods=4).mean()-data.cf1.shift(4)
        data=data[self.BASE_VARS+["dummyq2_2020","dummyq3_2020"]].join(quarterly[["gw_ft","gw_pt","weight_ft_lagged"]])
        expected=pd.period_range(self.BB_START,str(q),freq="Q")
        if data.reindex(expected).isna().any().any():
            raise ValueError(f"{q}: complete BB estimation inputs are unavailable")
        identity=data.gw_ft*data.weight_ft_lagged+data.gw_pt*(1-data.weight_ft_lagged)
        pair=pd.concat([data.gw,identity],axis=1).dropna()
        local.check("prefix_wage_identity",np.allclose(pair.iloc[:,0],pair.iloc[:,1],atol=1e-12),str(q))
        fits=local.fit_ftpt_system(data,start=self.BB_START,end=str(q),price_productivity=self.CFG["price_productivity"])
        hd,history_meta=local.ftpt_history(fits,data,self.NEWS_HISTORY_START,str(q),self.NEWS_MEAN_START,split_channels=True)
        gdp=local.Q[["gdp_real_sa_annualized"]].rename(columns={"gdp_real_sa_annualized":"real_gdp"})
        gdp["growth_yoy_pct"]=100*(gdp.real_gdp/gdp.real_gdp.shift(4)-1)
        gdp["news_growth"]=gdp.growth_yoy_pct.diff(4)
        news=local.ftpt_news(hd["gcpi"],gdp,"FIRM_1_"+str(self.long_year))
        valid=news.dropna(subset=self.FACTORS+["news_growth"])
        if valid.empty:
            raise ValueError(f"{q}: no complete BB news rows")
        local.check("prefix_news_additivity",np.allclose(valid.news_total,valid.news_reconstructed,atol=1e-8),str(q))
        local.check("prefix_news_groups",np.allclose(valid.news_total,valid.news_cost_push+valid.news_labor+valid.news_other,atol=1e-8),str(q))
        local.check("prefix_FT_PT_channels",np.allclose(valid.news_labor,valid.news_labor_ft+valid.news_labor_pt,atol=1e-8),str(q))
        rows=[{"equation":eq,"term":term,"beta":float(value)} for eq,fit in fits.items() for term,value in fit["beta"].items()]
        coefficients=pd.DataFrame(rows)
        fingerprint=hashlib.sha256(coefficients.to_csv(index=False,float_format="%.16g").encode()).hexdigest()[:12]
        version=f"FTPT_{q}_{fingerprint}"
        compiled=local.compile_ftpt_system(fits)
        roots=local.ftpt_frozen_roots(compiled,float(data.loc[q,"weight_ft_lagged"]))
        diagnostics={"fit_metrics":local.fit_metrics(fits),"roots":roots,
                     "coefficient_sums":local.coefficient_summaries(fits,"hac"),
                     "history_additivity_error":history_meta["max_reconstruction_error"],
                     "direct_simulation_error":history_meta["direct_simulation_error"]}
        provenance={"input_vintage":"recursive_current_vintage_not_real_time",
                    "production_eligible":False,"fit_start":self.BB_START,"fit_end":str(q),
                    "last_month_used":str(q.asfreq("M","end")),
                    "tax_anchor":local.CFG["tax_anchor"],
                    "google_calibration_end":self.CFG["google_calibration_end"],
                    "publication_timing":"handled separately by formation/release calendar",
                    "news_definition":"4-quarter mean of BB contributions minus its 4-quarter lag",
                    "inference_scope":"BB decomposition is not an orthogonal structural-shock identification",
                    "source_notebook":"main(1).ipynb (functions copied; no runtime dependency)",
                    "source_notebook_sha256":BB_SOURCE_NOTEBOOK_SHA256,
                    "source_cells_0_based":[5,6,10,13,16,32,36,38],
                    "construction":construction_meta,"history":history_meta}
        result={"news":news,"model_version":version,"macro_end":q,"provenance":provenance,
                "coefficients":coefficients,"diagnostics":diagnostics,
                "bb_input":data,"inflation_contributions":hd["gcpi"]}
        self._versions[key]=result
        return result

    # Reused from source cell 5: xlsx_values
    def xlsx_values(self, blob: bytes, sheet_name: str) -> dict[str, object]:

        def text(si):
            parts = []
            for child in si:
                if child.tag == '{' + self.NS['m'] + '}t':
                    parts.append(child.text or '')
                elif child.tag == '{' + self.NS['m'] + '}r':
                    parts.extend((t.text or '' for t in child.findall('m:t', self.NS)))
            return ''.join(parts)
        with zipfile.ZipFile(BytesIO(blob)) as z:
            strings = []
            if 'xl/sharedStrings.xml' in z.namelist():
                strings = [text(si) for si in ET.fromstring(z.read('xl/sharedStrings.xml'))]
            book = ET.fromstring(z.read('xl/workbook.xml'))
            rel = {n.attrib['Id']: n.attrib['Target'] for n in ET.fromstring(z.read('xl/_rels/workbook.xml.rels'))}
            sheets = {s.attrib['name']: s for s in book.find('m:sheets', self.NS)}
            if sheet_name not in sheets:
                raise KeyError(f'シートがありません: {sheet_name}; 候補={list(sheets)}')
            s = sheets[sheet_name]
            target = rel[s.attrib['{' + self.NS['r'] + '}id']]
            target = target.lstrip('/') if target.startswith('/') else posixpath.normpath('xl/' + target)
            result = {}
            for c in ET.fromstring(z.read(target)).findall('.//m:sheetData/m:row/m:c', self.NS):
                node = c.find('m:v', self.NS)
                raw = node.text if node is not None else None
                kind = c.attrib.get('t')
                if kind == 's':
                    value = strings[int(raw)] if raw is not None else ''
                elif kind == 'inlineStr':
                    value = text(c.find('m:is', self.NS))
                elif kind == 'e':
                    value = np.nan
                elif kind == 'b':
                    value = raw == '1'
                elif kind == 'str':
                    value = raw or ''
                elif raw is None:
                    value = None
                else:
                    try:
                        value = float(raw)
                    except ValueError:
                        value = raw
                result[c.attrib['r']] = value
            return result

    # Reused from source cell 6: decode_csv
    def decode_csv(self, b, preferred='cp932'):
        tried = []
        for enc in dict.fromkeys([preferred, 'utf-8-sig', 'cp932']):
            try:
                return b.decode(enc)
            except UnicodeError:
                tried.append(enc)
        raise UnicodeError(f'CSV文字コードを解釈できません: {tried}')

    # Reused from source cell 6: num
    def num(self, v):
        if v is None:
            return np.nan
        if isinstance(v, (float, int, np.number)):
            return float(v)
        s = str(v).strip().replace(',', '')
        if s in ['', '-', '--', '***', '…', '...', 'nan', '#N/A']:
            return np.nan
        if s == '<1':
            self.PARSER_FLAGS.append({'type': 'google_less_than_1', 'rule': '0.5; explicit approximation'})
            return 0.5
        try:
            return float(s)
        except ValueError:
            return np.nan

    # Reused from source cell 6: put
    def put(self, target, name, dates, values, freq, role):
        idx = pd.PeriodIndex(dates, freq=freq)
        if idx.has_duplicates:
            raise ValueError(f'{role}: {name}の日付が重複しています')
        if len(idx) == 0:
            raise ValueError(f'{role}: {name}を抽出できません。表見出し・形式を確認してください。')
        s = pd.Series(np.asarray(values, dtype=float), index=idx, name=name, dtype=float).sort_index()
        if np.isinf(s.to_numpy()).any():
            raise ValueError(f'{role}: 無限大があります')
        target[name] = s
        self.SOURCE_MAP[name] = role

    # Reused from source cell 6: era_year
    def era_year(self, v):
        if isinstance(v, (float, int)) and 1900 <= v <= 2100:
            return int(v)
        s = str(v)
        m = re.search('(19\\d{2}|20\\d{2})', s)
        if m:
            return int(m[0])
        for era, offset in [('昭和', 1925), ('平成', 1988), ('令和', 2018)]:
            m = re.search(era + '(\\d+|元)', s)
            if m:
                return offset + (1 if m[1] == '元' else int(m[1]))
        return None

    # Reused from source cell 6: excel_col
    def excel_col(self, n):
        out = ''
        while n:
            n, r = divmod(n - 1, 26)
            out = chr(65 + r) + out
        return out

    # Reused from source cell 6: provider_blob
    def provider_blob(self, role):
        return self.read_bytes(self.INPUT_DIR / 'provider' / self.PROVIDER_FILES[role], role)

    # Reused from source cell 6: load_provider
    def load_provider(self):
        monthly = {}
        quarterly = {}

        def cpi(role, codes):
            rows = list(csv.reader(io.StringIO(self.decode_csv(self.provider_blob(role)))))
            code_rows = [r for r in rows[:25] if all((v in r for v in codes.values()))]
            if len(code_rows) != 1:
                raise ValueError(f'{role}: CPI項目コードを一意に特定できません')
            hdr = code_rows[0]
            locations = {k: hdr.index(c) for k, c in codes.items()}
            dates = []
            values = {k: [] for k in codes}
            for row in rows:
                if not row or not re.fullmatch('\\d{6}', row[0].strip()):
                    continue
                y, m = divmod(int(row[0]), 100)
                if not 1 <= m <= 12:
                    continue
                if len(row) <= max(locations.values()):
                    raise ValueError(f'{role}: 行の列数不足 {row[0]}')
                dates.append(pd.Period(year=y, month=m, freq='M'))
                for k, c in locations.items():
                    values[k].append(self.num(row[c]))
            for k, v in values.items():
                self.put(monthly, k, dates, v, 'M', role)
        cpi('cpi_nsa', {'cpi_2025_nsa_all': '0001', 'cpi_2025_nsa_food': '0002', 'cpi_2025_nsa_energy': '0167'})
        cpi('cpi_sa', {'cpi_2025_sa_all': '0901'})
        tax = self.xlsx_values(self.provider_blob('cpi_tax'), 'zmi')
        dates = []
        values = []
        for a, v in tax.items():
            if re.fullmatch('A\\d+', a) and np.isfinite(self.num(v)) and (190001 <= self.num(v) <= 210012):
                y, m = divmod(int(self.num(v)), 100)
                if 1 <= m <= 12:
                    dates.append(pd.Period(year=y, month=m, freq='M'))
                    values.append(self.num(tax.get('B' + a[1:])))
        self.put(monthly, 'cpi_taxadjusted_2025_all', dates, values, 'M', 'cpi_tax')
        fields = {'所定内給与': 'scheduled_pay', '総実労働時間': 'total_hours', '所定内労働時間': 'scheduled_hours', '前月末労働者数': 'employees_start', '本月末労働者数': 'employees_end'}
        index_fields = {'所定内給与': 'scheduled_pay_index', '総実労働時間': 'total_hours_index', '所定内労働時間': 'scheduled_hours_index', '常用雇用': 'employment_index', 'パートタイム労働者比率': 'parttime_ratio_published'}
        for role, kind, cols in [('labour_actual', '実数', fields), ('labour_index', '指数', index_fields)]:
            reader = csv.DictReader(io.StringIO(self.decode_csv(self.provider_blob(role))))
            required = {'種別', '年', '月', '産業分類', '規模', '就業形態', *cols}
            if not required.issubset(reader.fieldnames or []):
                raise ValueError(f'{role}: 必須見出し不足 {required - set(reader.fieldnames or [])}')
            store = {}
            for row in reader:
                if row['種別'] != kind or row['産業分類'].strip() != 'TL' or row['規模'].strip() != 'T':
                    continue
                et = row['就業形態'].strip()
                if et not in ['0', '1', '2'] or not row['月'].strip().isdigit():
                    continue
                mo = int(row['月'])
                yr = int(row['年'])
                if not 1 <= mo <= 12:
                    continue
                et = {'0': 'all', '1': 'ft', '2': 'pt'}[et]
                for jp, en in cols.items():
                    val = self.num(row[jp])
                    if not np.isfinite(val):
                        continue
                    key = f'labour_{et}_{en}'
                    store.setdefault(key, []).append((pd.Period(year=yr, month=mo, freq='M'), val))
            for k, v in store.items():
                self.put(monthly, k, [x[0] for x in v], [x[1] for x in v], 'M', role)

        def lfs(role, mapping):
            b = self.provider_blob(role)
            for sheet, tag in [('季節調整値', 'sa'), ('原数値', 'nsa')]:
                cc = self.xlsx_values(b, sheet)
                last = max((int(re.search('\\d+$', a)[0]) for a in cc))
                year = None
                store = {k: [] for k in mapping}
                for r in range(1, last + 1):
                    y = self.era_year(cc.get(f'A{r}'))
                    if y is not None:
                        year = y
                    mat = re.fullmatch('(\\d{1,2})月', str(cc.get(f'B{r}', '')))
                    if not mat:
                        continue
                    mo = int(mat[1])
                    if mo == 1:
                        ny = self.era_year(cc.get(f'A{r + 1}'))
                        if ny is not None:
                            year = ny
                    if year is None or not 1 <= mo <= 12:
                        continue
                    date = pd.Period(year=year, month=mo, freq='M')
                    for k, col in mapping.items():
                        val = self.num(cc.get(f'{col}{r}'))
                        if np.isfinite(val):
                            store[k].append((date, val))
                for k, v in store.items():
                    self.put(monthly, f'lfs_{tag}_{k}', [x[0] for x in v], [x[1] for x in v], 'M', role)
        lfs('lfs_main', {'labor_force': 'E', 'employed': 'H', 'employees': 'K', 'unemployed': 'N', 'unemployment_rate': 'T'})
        lfs('lfs_nonagri', {'employed_check': 'E', 'agri_employed': 'H', 'nonagri_employed': 'K'})
        for role, sheet, var in [('job_vacancies', '第６表ー１（パート含む）', 'active_vacancies'), ('job_placements', '第８表ー１（パート含む）', 'placements')]:
            cc = self.xlsx_values(self.provider_blob(role), sheet)
            maxr = max((int(re.search('\\d+$', a)[0]) for a in cc))
            for tag, offset in [('nsa', 2), ('sa', 20)]:
                dates = []
                values = []
                for r in range(1, maxr + 1):
                    mat = re.fullmatch('(\\d{4})年', str(cc.get(f'A{r}', '')))
                    if not mat:
                        continue
                    y = int(mat[1])
                    for mo in range(1, 13):
                        val = self.num(cc.get(f'{self.excel_col(offset + mo)}{r}'))
                        if np.isfinite(val):
                            dates.append(pd.Period(year=y, month=mo, freq='M'))
                            values.append(val)
                self.put(monthly, f'jobs_{tag}_{var}', dates, values, 'M', role)
        for role, key in [('gdp_real_sa', 'gdp_real_sa_annualized'), ('gdp_real_nsa', 'gdp_real_nsa'), ('gdp_nominal_sa', 'gdp_nominal_sa_annualized'), ('gdp_nominal_nsa', 'gdp_nominal_nsa')]:
            rows = list(csv.reader(io.StringIO(self.decode_csv(self.provider_blob(role)))))
            header = ' '.join((' '.join(r) for r in rows[:8]))
            expected = '実質' if 'real' in role else '名目'
            if expected not in header or '国内総生産' not in header:
                raise ValueError(f'{role}: 実額GDPの見出しが一致しません')
            if role.endswith('_sa') and '季節調整' not in header:
                raise ValueError(f'{role}: 季調系列ではありません')
            dates = []
            values = []
            year = None
            for row in rows:
                if len(row) < 2:
                    continue
                mat = re.fullmatch('\\s*(?:(\\d{4})/\\s*)?(\\d{1,2})-\\s*(\\d{1,2})\\.\\s*', row[0])
                if not mat:
                    continue
                if mat[1]:
                    year = int(mat[1])
                if year is None:
                    continue
                mo = int(mat[2])
                val = self.num(row[1])
                if mo not in [1, 4, 7, 10] or int(mat[3]) != mo + 2:
                    continue
                if not np.isfinite(val):
                    continue
                dates.append(pd.Period(year=year, quarter=(mo - 1) // 3 + 1, freq='Q'))
                values.append(val)
            self.put(quarterly, key, dates, values, 'Q', role)
        lines = self.decode_csv(self.provider_blob('expectations'), 'utf-8-sig').splitlines()
        head = next((i for i, l in enumerate(lines) if l.lstrip('\ufeff').startswith('Quarter,')), None)
        if head is None:
            raise ValueError('einf.csvのQuarterヘッダーがありません')
        fr = list(csv.DictReader(io.StringIO('\n'.join(lines[head:]))))
        for h in range(1, 11):
            if f'{h}Year' not in fr[0]:
                raise ValueError(f'einf.csv: {h}Year列なし')
            dates = []
            values = []
            for row in fr:
                label = row['Quarter'].strip()
                if not re.fullmatch('\\d{6}', label):
                    continue
                y, mo = divmod(int(label), 100)
                if mo not in [3, 6, 9, 12]:
                    raise ValueError(f'einf観測日が四半期末月でありません: {label}')
                val = self.num(row[f'{h}Year'])
                if np.isfinite(val):
                    dates.append(pd.Period(year=y, quarter=mo // 3, freq='Q'))
                    values.append(val)
            self.put(quarterly, f'firm_expectation_{h}y', dates, values, 'Q', 'expectations')
        gt = list(csv.reader(io.StringIO(self.decode_csv(self.provider_blob('shortage'), 'utf-8-sig'))))
        heading = ' '.join((' '.join(r) for r in gt[:8]))
        if '供給不足' not in heading:
            raise ValueError('Google CSVの検索語が供給不足ではありません')
        dates = []
        values = []
        for row in gt:
            if len(row) >= 2 and re.fullmatch('\\d{4}-\\d{2}', row[0]):
                dates.append(pd.Period(row[0], freq='M'))
                values.append(self.num(row[1]))
        self.put(monthly, 'google_shortage', dates, values, 'M', 'shortage')
        return (pd.DataFrame(monthly).sort_index(), pd.DataFrame(quarterly).sort_index())

    # Reused from source cell 6: load_canonical
    def load_canonical(self, schema):
        monthly = {}
        quarterly = {}
        for group, entry in schema.items():
            b = self.read_bytes(self.INPUT_DIR / 'csv' / f'{group}.csv', 'canonical_' + group)
            df = pd.read_csv(BytesIO(b), encoding='utf-8-sig')
            key = entry['date_column']
            if key not in df:
                raise ValueError(f'{group}.csv: {key}列が必要です')
            dates = df.pop(key).astype(str)
            missing = set(entry['columns']) - set(df)
            if missing:
                raise ValueError(f'{group}.csv: 原系列列が不足 {sorted(missing)}')
            for c in entry['columns']:
                values = pd.to_numeric(df[c], errors='raise')
                self.put(monthly if entry['frequency'] == 'M' else quarterly, c, dates, values, entry['frequency'], 'canonical_' + group)
        return (pd.DataFrame(monthly).sort_index(), pd.DataFrame(quarterly).sort_index())

    # Reused from source cell 10: qmean_complete
    def qmean_complete(self, series: pd.Series) -> pd.Series:
        if not isinstance(series.index, pd.PeriodIndex) or series.index.freqstr != 'M':
            raise TypeError('月次PeriodIndexを指定してください')
        grouped = series.groupby(series.index.asfreq('Q'))
        return grouped.mean().where(grouped.count() == 3)

    # Reused from source cell 10: growth
    def growth(self, series: pd.Series, method='log') -> pd.Series:
        ratio = series / series.shift(1)
        if method == 'log':
            return 400 * np.log(ratio)
        if method == 'simple':
            return 400 * (ratio - 1)
        if method == 'compound':
            return 100 * (ratio ** 4 - 1)
        raise ValueError(method)

    # Reused from source cell 10: log_stl
    def log_stl(self, series: pd.Series, trend=19, seasonal=13):
        s = series.dropna().astype(float)
        if not (s > 0).all():
            raise ValueError('STL input must be positive')
        if not s.index.equals(pd.period_range(s.index[0], s.index[-1], freq='M', name=s.index.name)):
            raise ValueError('STL input has internal gaps')
        result = STL(np.log(s.to_numpy()), period=12, seasonal=seasonal, trend=trend, robust=True).fit()
        return pd.DataFrame({'raw': s.to_numpy(), 'sa': np.exp(np.log(s.to_numpy()) - result.seasonal), 'tc': np.exp(result.trend), 'log_seasonal': result.seasonal}, index=s.index)

    # Reused from source cell 10: build_reconstruction
    def build_reconstruction(self, end_quarter, *, wage_trend=None, wage_component='tc', wage_source='index', pt_hours='scheduled', price_sa='official_hybrid', productivity_source='mls_index', shortage_adjustment='mean', growth_method='log', cpi_base=2025):
        qend = pd.Period(end_quarter, freq='Q')
        mend = qend.asfreq('M', 'end')
        m = self.M.loc[:'%s' % mend].copy()
        qi = pd.period_range('1990Q1', qend, freq='Q', name='quarter')
        panel = pd.DataFrame(index=qi)
        details_m = pd.DataFrame(index=m.index)
        details_q = pd.DataFrame(index=qi)
        if cpi_base == 2020:
            if 'cpi_2020_nsa_all' not in m:
                raise ValueError('旧基準CPIは診断用キャッシュでのみ利用')
        p0 = m['cpi_2025_nsa_all']
        tax = m['cpi_taxadjusted_2025_all'].copy()
        anchor = pd.Period(self.CFG['tax_anchor'], freq='M')
        ratio = float(tax.loc[anchor] / p0.loc[anchor])
        after = m.index > anchor
        tax.loc[after] = p0.loc[after] * ratio
        tax = tax.loc['1990-01':]
        cpistl = self.log_stl(tax, trend=self.CFG['other_trend'], seasonal=self.CFG['stl_seasonal'])
        official_factor = m.cpi_2025_sa_all / p0
        cpisa = cpistl.sa.copy()
        if price_sa == 'official_hybrid':
            ix = cpisa.index.intersection(official_factor.dropna().index)
            cpisa.loc[ix] = tax.loc[ix] * official_factor.loc[ix]
        elif price_sa != 'stl':
            raise ValueError(price_sa)
        panel['gcpi'] = self.growth(self.qmean_complete(cpisa), growth_method)
        details_m['cpi_adjusted_nsa'] = tax
        details_m['cpi_adjusted_sa'] = cpisa
        details_q['cpi_level_sa'] = self.qmean_complete(cpisa)
        suffix = '_index' if wage_source == 'index' else ''
        ft = m['labour_ft_scheduled_pay' + suffix]
        denom = 'labour_pt_' + ('scheduled_hours' if pt_hours == 'scheduled' else 'total_hours') + suffix
        pt = m['labour_pt_scheduled_pay' + suffix] / m[denom]
        ftcomp = self.log_stl(ft, trend=wage_trend or self.CFG['wage_trend'], seasonal=self.CFG['stl_seasonal'])
        ptcomp = self.log_stl(pt, trend=wage_trend or self.CFG['wage_trend'], seasonal=self.CFG['stl_seasonal'])
        n_ft = (m.labour_ft_employees_start + m.labour_ft_employees_end) / 2
        n_pt = (m.labour_pt_employees_start + m.labour_pt_employees_end) / 2
        weights = self.qmean_complete(n_ft / (n_ft + n_pt)).shift(1)
        gft = self.growth(self.qmean_complete(ftcomp[wage_component]), growth_method)
        gpt = self.growth(self.qmean_complete(ptcomp[wage_component]), growth_method)
        panel['gw'] = weights * gft + (1 - weights) * gpt
        details_q['gw_ft'] = gft
        details_q['gw_pt'] = gpt
        details_q['weight_ft_lagged'] = weights
        details_m['ft_wage_component'] = ftcomp[wage_component]
        details_m['pt_hourly_component'] = ptcomp[wage_component]
        wage_den = m.labour_all_scheduled_pay_index
        for var, key in [('grpe', 'energy'), ('grpf', 'food')]:
            rel = m['cpi_2025_nsa_' + key] / wage_den
            relsa = self.log_stl(rel, trend=self.CFG['other_trend'], seasonal=self.CFG['stl_seasonal']).sa
            panel[var] = self.growth(self.qmean_complete(relsa), growth_method)
            details_m[var + '_level_sa'] = relsa
        A = m.jobs_sa_active_vacancies
        J = m.jobs_sa_placements
        U = m.lfs_sa_unemployed * 10000
        E = m.lfs_sa_employees * 10000
        vu = (A - J) / (E + A - J) / (U / (E + U))
        panel['vu'] = self.qmean_complete(vu)
        details_m['vu_rate_ratio'] = vu
        if productivity_source == 'mls_index':
            labor = m.labour_all_employment_index * m.labour_all_total_hours_index
        elif productivity_source == 'lfs_nonagri':
            labor = m.lfs_nsa_nonagri_employed * m.labour_all_total_hours
        elif productivity_source == 'mls_level':
            labor = (m.labour_all_employees_start + m.labour_all_employees_end) / 2 * m.labour_all_total_hours
        else:
            raise ValueError(productivity_source)
        labor_sa = self.log_stl(labor, trend=self.CFG['other_trend'], seasonal=self.CFG['stl_seasonal']).sa
        prod = self.Q.gdp_real_sa_annualized.loc[:qend] / 4 / self.qmean_complete(labor_sa)
        p_growth = self.growth(prod, growth_method)
        panel['magpty'] = p_growth.rolling(8, min_periods=8).mean()
        details_q['productivity_level'] = prod
        details_q['productivity_growth'] = p_growth
        details_m['labor_input_sa'] = labor_sa
        g = m.google_shortage.copy()
        if shortage_adjustment == 'mean':
            g.loc[self.CFG['google_earthquake_start']:self.CFG['google_earthquake_end']] = self.GOOGLE_FILL
        elif shortage_adjustment == 'legacy_zero':
            g.loc['2011-01':'2011-06'] = 0.0
        elif shortage_adjustment != 'none':
            raise ValueError(shortage_adjustment)
        g.loc[(g.index < pd.Period('2004-01')) & (g.index >= pd.Period('1990-01'))] = self.GOOGLE_FILL
        panel['shortage'] = self.qmean_complete(g) * self.GOOGLE_SCALE
        details_m['google_adjusted_legacy_units'] = g * self.GOOGLE_SCALE
        for q in [2, 3]:
            panel[f'dummyq{q}_2020'] = (panel.index == pd.Period(f'2020Q{q}')).astype(float)
        meta = {'construction_end': str(qend), 'last_month_used': str(mend), 'wage_trend': wage_trend or self.CFG['wage_trend'], 'wage_component': wage_component, 'wage_source': wage_source, 'pt_hours': pt_hours, 'price_sa': price_sa, 'productivity_source': productivity_source, 'shortage_adjustment': shortage_adjustment, 'growth': growth_method, 'tax_extension_anchor': self.CFG['tax_anchor'], 'tax_level_ratio': ratio, 'google_scale': self.GOOGLE_SCALE, 'google_pre2004_fill_raw': self.GOOGLE_FILL}
        return (panel, details_m, details_q, meta)

    # Reused from source cell 13: equation_specs
    def equation_specs(self, price_productivity: bool=True, covid_dummies: bool=True):
        lag = lambda v, a, b: [(v, j) for j in range(a, b + 1)]
        wage = lag('gw', 1, 4) + lag('cf1', 1, 4) + [('magpty', 1)] + lag('vu', 1, 4) + lag('diffcpicf', 1, 4)
        if covid_dummies:
            wage += [('dummyq2_2020', 0), ('dummyq3_2020', 0)]
        price = [('magpty', 0)] if price_productivity else []
        price += lag('gcpi', 1, 4) + lag('gw', 0, 4) + lag('grpe', 0, 4) + lag('grpf', 0, 4) + lag('shortage', 0, 4)
        return {'gw': _BBEquationSpec('gw', tuple(wage), ('gw', 'cf1'), True), 'gcpi': _BBEquationSpec('gcpi', tuple(price), ('gcpi', 'gw'), True), 'cf10': _BBEquationSpec('cf10', tuple(lag('cf10', 1, 4) + lag('gcpi', 0, 4)), ('cf10', 'gcpi'), False), 'cf1': _BBEquationSpec('cf1', tuple(lag('cf1', 1, 4) + lag('cf10', 0, 4) + lag('gcpi', 0, 4)), ('cf1', 'cf10', 'gcpi'), False)}

    # Reused from source cell 13: term_label
    def term_label(self, v: str, lag: int) -> str:
        return v if lag == 0 else f'L.{v}' if lag == 1 else f'L{lag}.{v}'

    # Reused from source cell 13: design
    def design(self, data: pd.DataFrame, spec: _BBEquationSpec, start='1989Q1', end='2023Q2'):
        X = pd.DataFrame({self.term_label(v, l): data[v].shift(l) for v, l in spec.terms}, index=data.index)
        if spec.constant:
            X['_cons'] = 1.0
        target = data[spec.name]
        mask = target.notna() & X.notna().all(axis=1)
        mask &= (data.index >= pd.Period(start)) & (data.index <= pd.Period(end))
        X, y = (X.loc[mask].astype(float), target.loc[mask].astype(float))
        if not len(y):
            raise ValueError(f'{spec.name}: 利用可能な観測がありません')
        active = []
        for v, l in spec.terms:
            name = self.term_label(v, l)
            if v.startswith('dummy') and np.all(X[name].to_numpy() == 0):
                X = X.drop(columns=name)
            else:
                active.append((v, l))
        effective_spec = _BBEquationSpec(spec.name, tuple(active), spec.homogeneity_groups, spec.constant)
        R = np.array([[float(v in spec.homogeneity_groups) for v, l in active] + ([0.0] if spec.constant else [])])
        return (y, X, R, np.ones(1), effective_spec)

    # Reused from source cell 13: wald_test
    def wald_test(self, beta, cov, A, value, df_resid):
        A = np.atleast_2d(A)
        delta = A @ beta - np.broadcast_to(value, A.shape[0])
        V = A @ cov @ A.T
        V = (V + V.T) / 2
        rank = np.linalg.matrix_rank(V, tol=max(1e-16, np.linalg.norm(V, 2) * 1e-10))
        if rank == 0:
            return (np.nan, np.nan, 0)
        W = float(delta @ np.linalg.pinv(V, rcond=1e-10) @ delta)
        F = max(0.0, W / rank)
        return (F, float(stats.f.sf(F, rank, df_resid)), rank)

    # Reused from source cell 13: restricted_fit
    def restricted_fit(self, y: pd.Series, X: pd.DataFrame, R: np.ndarray, r: np.ndarray, spec: _BBEquationSpec, variance_df: str='restricted'):
        a, b = (X.to_numpy(), y.to_numpy())
        n, k = a.shape
        if np.linalg.matrix_rank(a) < k:
            raise ValueError(f'{spec.name}: Xがランク落ちしています（{n}x{k}）。黙って擬似逆行列推定しません。')
        Z = linalg.null_space(R)
        b0 = np.linalg.lstsq(R, r, rcond=None)[0]
        A = a @ Z
        theta = np.linalg.lstsq(A, b - a @ b0, rcond=None)[0]
        beta = b0 + Z @ theta
        fitted = a @ beta
        resid = b - fitted
        kfree = Z.shape[1]
        df_resid = n - kfree if variance_df == 'restricted' else n - k
        if df_resid <= 0:
            raise ValueError('残差自由度が非正です')
        Ap = np.linalg.pinv(A)
        bread = Ap @ Ap.T
        sigma2 = float(resid @ resid / df_resid)
        cov = sigma2 * Z @ bread @ Z.T
        scores = A * resid[:, None]
        meat = scores.T @ scores
        for lag in range(1, min(4, n - 1) + 1):
            cross = scores[lag:].T @ scores[:-lag]
            meat += (1 - lag / 5) * (cross + cross.T)
        cov_hac = n / (n - kfree) * Z @ bread @ meat @ bread @ Z.T
        bu = np.linalg.lstsq(a, b, rcond=None)[0]
        eu = b - a @ bu
        inv = np.linalg.pinv(a)
        vu = float(eu @ eu / (n - k)) * inv @ inv.T
        Fh, ph, _ = self.wald_test(bu, vu, R, r, n - k)
        corr2 = float(np.corrcoef(b, fitted)[0, 1] ** 2)
        ss = float(np.sum((b - b.mean()) ** 2))
        return dict(spec=spec, beta=pd.Series(beta, index=X.columns), cov=cov, cov_hac=cov_hac, y=y, X=X, R=R, r=r, fitted=pd.Series(fitted, index=y.index), residual=pd.Series(resid, index=y.index), n=n, k=k, k_free=kfree, df_resid=df_resid, rss=float(resid @ resid), rmse=float(np.sqrt(np.mean(resid ** 2))), r2_corr=corr2, r2_centered=1 - float(resid @ resid) / ss, homogeneity_p=ph, homogeneity_F=Fh, condition_number=float(np.linalg.cond(A)), restriction_error=float(np.max(np.abs(R @ beta - r))))

    # Reused from source cell 13: fit_system
    def fit_system(self, data: pd.DataFrame, price_productivity: bool=True, start='1989Q1', end='2023Q2', covid_dummies: bool=True, common_sample: bool=False, variance_df: str='restricted'):
        specs = self.equation_specs(price_productivity, covid_dummies)
        blocks = {name: self.design(data, spec, start, end) for name, spec in specs.items()}
        if common_sample:
            idx = None
            for y, _, _, _, _ in blocks.values():
                idx = y.index if idx is None else idx.intersection(y.index)
            blocks = {name: (y.loc[idx], X.loc[idx], R, r, sp) for name, (y, X, R, r, sp) in blocks.items()}
        return {name: self.restricted_fit(*block, variance_df=variance_df) for name, block in blocks.items()}

    # Reused from source cell 13: coefficient_summaries
    def coefficient_summaries(self, model: dict, covariance='iid') -> pd.DataFrame:
        rows = []
        for eq, f in model.items():
            beta = f['beta'].to_numpy()
            cov = f['cov'] if covariance == 'iid' else f['cov_hac']
            for group in dict.fromkeys((v for v, l in f['spec'].terms)):
                indices = [j for j, (v, l) in enumerate(f['spec'].terms) if v == group]
                Asum = np.zeros((1, len(beta)))
                Asum[0, indices] = 1
                Ajoint = np.eye(len(beta))[indices, :]
                fs, ps, _ = self.wald_test(beta, cov, Asum, 0, f['df_resid'])
                fj, pj, qj = self.wald_test(beta, cov, Ajoint, 0, f['df_resid'])
                se = float(np.sqrt(max(0.0, (Asum @ cov @ Asum.T).item())))
                rows.append(dict(equation=eq, group=group, coef_sum=float((Asum @ beta).item()), se_sum=se, p_sum=ps, p_joint=pj, F_sum=fs, F_joint=fj, covariance=covariance, n=f['n'], start=str(f['y'].index.min()), end=str(f['y'].index.max())))
        return pd.DataFrame(rows)

    # Reused from source cell 13: fit_metrics
    def fit_metrics(self, model: dict) -> pd.DataFrame:
        return pd.DataFrame([dict(equation=name, start=str(f['y'].index.min()), end=str(f['y'].index.max()), **{k: f[k] for k in ['n', 'k', 'k_free', 'df_resid', 'r2_corr', 'r2_centered', 'rmse', 'rss', 'homogeneity_p', 'condition_number', 'restriction_error']}) for name, f in model.items()])

    # Reused from source cell 16: change_only
    def change_only(self, s):
        return s.rolling(4, min_periods=4).mean().diff(4)

    # Reused from source cell 16: make_news
    def make_news(self, hd, growth, expectation_source='CIE_1_10'):
        out = pd.DataFrame(index=hd.index)
        for c in self.PARTS:
            out['news_' + c] = self.change_only(hd[c])
        out['news_total'] = self.change_only(hd['actual'])
        out['news_cost_push'] = out[['news_energy', 'news_food', 'news_shortage']].sum(axis=1, min_count=3)
        out['news_other'] = out[['news_' + c for c in self.PARTS if c not in ['energy', 'food', 'shortage', 'labor']]].sum(axis=1, min_count=8)
        out['news_reconstructed'] = out[['news_' + c for c in self.PARTS]].sum(axis=1, min_count=len(self.PARTS))
        out['news_growth'] = growth.news_growth.reindex(out.index)
        out['bb_inflation_4q_average'] = hd.actual.rolling(4, min_periods=4).mean()
        out['cie_pre2007_reference_period'] = out.index.year <= 2006 if expectation_source == 'CIE_1_10' else False
        out['expectation_source'] = expectation_source
        out['fixed_vintage_ex_post'] = True
        return out

    # Reused from source cell 32: ftpt_specs
    def ftpt_specs(self, price_productivity=True, covid_dummies=True):
        base = self.equation_specs(price_productivity, covid_dummies)
        out = {}
        for dep in ['gw_ft', 'gw_pt']:
            wage = base['gw']
            terms = tuple(((dep if v == 'gw' else v, l) for v, l in wage.terms))
            out[dep] = _BBEquationSpec(dep, terms, (dep, 'cf1'), True)
        out.update({k: base[k] for k in ['gcpi', 'cf10', 'cf1']})
        return out

    # Reused from source cell 32: fit_ftpt_system
    def fit_ftpt_system(self, data, start='1996Q2', end=None, price_productivity=None, covid_dummies=True):
        end = str(self.BB_END) if end is None else str(end)
        prod = self.CFG['price_productivity'] if price_productivity is None else price_productivity
        blocks = {k: self.design(data, sp, start, end) for k, sp in self.ftpt_specs(prod, covid_dummies).items()}
        common = None
        for y, X, R, r, sp in blocks.values():
            common = y.index if common is None else common.intersection(y.index)
        expected = pd.period_range(start, end, freq='Q')
        if not common.equals(expected):
            raise ValueError('FT/PT共通標本に欠損があります。黙って推定開始期や対象行を変えません。')
        return {k: self.restricted_fit(y.loc[common], X.loc[common], R, r, sp) for k, (y, X, R, r, sp) in blocks.items()}

    # Reused from source cell 36: compile_ftpt_system
    def compile_ftpt_system(self, fits):
        n = len(self.FTPT_STATE)
        p = 8
        q = 4
        H = np.eye(n)
        AL = np.zeros((p, n, n))
        BL = np.zeros((q + 1, n, len(self.EXOG)))
        c = np.zeros(n)

        def add_y(row, var, lag, value):
            col = self.FTPT_STATE.index(var)
            if lag == 0:
                H[row, col] -= value
            else:
                AL[lag - 1, row, col] += value
        for eq, f in fits.items():
            row = self.FTPT_STATE.index(eq)
            if f['spec'].constant:
                c[row] = f['beta']['_cons']
            for v, l in f['spec'].terms:
                b = float(f['beta'][self.term_label(v, l)])
                if v == 'diffcpicf':
                    for j in range(4):
                        add_y(row, 'gcpi', l + j, b / 4)
                    add_y(row, 'cf1', l + 4, -b)
                elif v in self.FTPT_STATE:
                    add_y(row, v, l, b)
                elif v in self.EXOG:
                    BL[l, row, self.EXOG.index(v)] += b
                else:
                    raise KeyError(v)
        return {'H': H, 'AL': AL, 'BL': BL, 'c': c, 'p': p, 'q': q, '_cache': {}}

    # Reused from source cell 36: ftpt_matrices
    def ftpt_matrices(self, comp, weight):
        weight = float(weight)
        if not 0 < weight < 1:
            raise ValueError('FT/PT集計ウェイトが(0,1)範囲外です')
        if weight not in comp['_cache']:
            H = comp['H'].copy()
            g = self.FTPT_STATE.index('gw')
            H[g, self.FTPT_STATE.index('gw_ft')] = -weight
            H[g, self.FTPT_STATE.index('gw_pt')] = -(1 - weight)
            invH = np.linalg.inv(H)
            A = np.einsum('ij,ljk->lik', invH, comp['AL'])
            B = np.einsum('ij,ljk->lik', invH, comp['BL'])
            comp['_cache'][weight] = (A, B, invH @ comp['c'], invH)
        return comp['_cache'][weight]

    # Reused from source cell 36: simulate_ftpt
    def simulate_ftpt(self, comp, data, x, start, end, innovations=None, zero_history=False, constant=True, weights=None):
        dates = pd.period_range(start, end, freq='Q')
        all_dates = pd.period_range(dates[0] - comp['p'], dates[-1], freq='Q')
        p, q = (comp['p'], comp['q'])
        n = len(self.FTPT_STATE)
        y = np.zeros((len(all_dates), n))
        if not zero_history:
            y[:p] = data.reindex(all_dates[:p])[self.FTPT_STATE].to_numpy()
            if not np.isfinite(y[:p]).all():
                raise ValueError('FT/PT初期8四半期の内生履歴が不足')
        xv = x.reindex(all_dates)[self.EXOG].to_numpy()
        if not np.isfinite(xv[p - q:]).all():
            raise ValueError('外生入力/ラグに欠損')
        ww = (data.weight_ft_lagged if weights is None else weights).reindex(dates)
        if ww.isna().any():
            raise ValueError('FT/PT集計ウェイトが不足')
        u = np.zeros((len(dates), n))
        if innovations is not None:
            uu = innovations.reindex(dates)
            if not set(self.FTPT_EQ).issubset(uu.columns):
                raise ValueError('FT/PT残差の5列が必要')
            if uu[self.FTPT_EQ].isna().any().any():
                raise ValueError('FT/PT残差に欠損')
            for eq in self.FTPT_EQ:
                u[:, self.FTPT_STATE.index(eq)] = uu[eq]
            if 'gw' in uu and (not np.allclose(uu.gw, 0)):
                raise ValueError('集計gwは恒等式。独立した残差は入力できません')
        for j, t in enumerate(range(p, len(all_dates))):
            A, B, c, imp = self.ftpt_matrices(comp, ww.iloc[j])
            val = c.copy() if constant else np.zeros(n)
            val += np.einsum('lij,lj->i', A, y[t - np.arange(1, p + 1)])
            val += np.einsum('lij,lj->i', B, xv[t - np.arange(q + 1)])
            val += imp @ u[j]
            if not np.isfinite(val).all():
                raise FloatingPointError('FT/PTシミュレーションが発散')
            y[t] = val
        out = pd.DataFrame(y[p:], index=dates, columns=self.FTPT_STATE)
        identity = ww * out.gw_ft + (1 - ww) * out.gw_pt
        if not np.allclose(out.gw, identity, atol=1e-09, rtol=1e-10):
            raise AssertionError('再帰計算でFT/PT集計恒等式が壊れています')
        return out

    # Reused from source cell 36: ftpt_innovations
    def ftpt_innovations(self, fits, data, start, end):
        dates = pd.period_range(start, end, freq='Q')
        out = {}
        for eq, f in fits.items():
            y, X, *_ = self.design(data, f['spec'], start, end)
            out[eq] = (y - X.reindex(columns=f['beta'].index) @ f['beta']).reindex(dates)
        result = pd.DataFrame(out, index=dates)
        if result.isna().any().any():
            raise ValueError('FT/PT残差を全期間で計算できません')
        return result

    # Reused from source cell 36: ftpt_labor_entry
    def ftpt_labor_entry(self, comp, leg):
        if leg not in ['FT', 'PT']:
            raise ValueError(leg)
        cc = {k: v for k, v in comp.items() if k != '_cache'}
        cc['BL'] = comp['BL'].copy()
        cc['_cache'] = {}
        other = 'gw_pt' if leg == 'FT' else 'gw_ft'
        cc['BL'][:, self.FTPT_STATE.index(other), self.EXOG.index('vu')] = 0.0
        return cc

    # Reused from source cell 36: ftpt_history
    def ftpt_history(self, fits, data, start, end, mean_start='1990Q1', vu_reference=None, split_channels=True):
        comp = self.compile_ftpt_system(fits)
        dates = pd.period_range(start, end, freq='Q')
        anchor = dates[0] - 1
        x = data[self.EXOG].copy()
        base = x.copy()
        means = data.loc[mean_start:str(anchor), ['magpty', 'shortage']].mean()
        base.loc[dates, ['grpe', 'grpf']] = 0.0
        base.loc[dates, 'vu'] = float(data.loc[anchor, 'vu'] if vu_reference is None else vu_reference)
        for v in ['magpty', 'shortage']:
            base.loc[dates, v] = float(means[v])
        base.loc[dates, ['dummyq2_2020', 'dummyq3_2020']] = 0.0
        initial = self.simulate_ftpt(comp, data, base, start, end)
        parts = {'initial_conditions': initial}
        groups = {'energy': ['grpe'], 'food': ['grpf'], 'shortage': ['shortage'], 'labor': ['vu'], 'productivity': ['magpty'], 'covid_q2': ['dummyq2_2020'], 'covid_q3': ['dummyq3_2020']}
        for name, cols in groups.items():
            sc = base.copy()
            sc.loc[dates, cols] = x.loc[dates, cols]
            parts[name] = self.simulate_ftpt(comp, data, sc, start, end) - initial
        U = self.ftpt_innovations(fits, data, start, end)
        for eq in self.FTPT_EQ:
            u = U * 0.0
            u[eq] = U[eq]
            parts['residual_' + eq] = self.simulate_ftpt(comp, data, base, start, end, u) - initial
        reconstructed = sum(parts.values())
        observed = data.loc[dates, self.FTPT_STATE]
        direct = self.simulate_ftpt(comp, data, x, start, end, U)
        noresid = self.simulate_ftpt(comp, data, x, start, end)
        error = float(np.max(np.abs(reconstructed.to_numpy() - observed.to_numpy())))
        direct_error = float(np.max(np.abs(direct.to_numpy() - observed.to_numpy())))
        if max(error, direct_error) > 1e-08:
            raise AssertionError(f'FT/PT加法性または実績再現に失敗: {error}, {direct_error}')
        channels = {}
        if split_channels:
            dx = pd.DataFrame(0.0, index=data.index, columns=self.EXOG)
            dx.loc[dates, 'vu'] = x.loc[dates, 'vu'] - base.loc[dates, 'vu']
            for leg in ['FT', 'PT']:
                channels[leg] = self.simulate_ftpt(self.ftpt_labor_entry(comp, leg), data, dx, start, end, zero_history=True, constant=False)
            if not np.allclose(channels['FT'] + channels['PT'], parts['labor'], atol=1e-08):
                raise AssertionError('FT/PT労働需給伝播寄与の和が全体に一致しません')
        tables = {}
        for eq in self.FTPT_STATE:
            t = pd.DataFrame({k: v[eq] for k, v in parts.items()}, index=dates)
            t['actual'] = observed[eq]
            t['conditional_no_residual'] = noresid[eq]
            t['reconstructed'] = reconstructed[eq]
            if split_channels:
                t['labor_via_FT'] = channels['FT'][eq]
                t['labor_via_PT'] = channels['PT'][eq]
            tables[eq] = t
        metadata = {'start': start, 'end': end, 'anchor': str(anchor), 'mean_start': mean_start, 'weight_path': 'observed lagged employment weights identical in every counterfactual', 'weight_exogeneity': 'No employment composition equation; not a causal employment-share attribution', 'additive_components': self.FTPT_PARTS, 'auxiliary_nonadditive_columns': ['actual', 'conditional_no_residual', 'reconstructed', 'labor_via_FT', 'labor_via_PT'], 'max_reconstruction_error': error, 'direct_simulation_error': direct_error, 'relative_prices': 'unchanged exogenous supplied relative price growth rates', 'labor_channels': 'artificial split of a common V/U input by wage-equation entry, full feedback retained; not independent shocks', 'residuals': '5 equation innovations; no aggregate wage residual; not orthogonalized'}
        return (tables, metadata)

    # Reused from source cell 36: ftpt_news
    def ftpt_news(self, hd, gdp, expectation_source):
        compatible = hd.copy()
        compatible['residual_gw'] = hd.residual_gw_ft + hd.residual_gw_pt
        nw = self.make_news(compatible, gdp, expectation_source)
        if 'labor_via_FT' in hd:
            nw['news_labor_ft'] = self.change_only(hd.labor_via_FT)
            nw['news_labor_pt'] = self.change_only(hd.labor_via_PT)
        nw['news_residual_gw_ft'] = self.change_only(hd.residual_gw_ft)
        nw['news_residual_gw_pt'] = self.change_only(hd.residual_gw_pt)
        nw['wage_system'] = 'FTPT_5equations_observed_lagged_weights'
        return nw

    # Reused from source cell 38: ftpt_frozen_roots
    def ftpt_frozen_roots(self, comp, w):
        A, B, c, imp = self.ftpt_matrices(comp, w)
        n = len(self.FTPT_STATE)
        companion = np.zeros((n * 8, n * 8))
        companion[:n] = np.concatenate(list(A), axis=1)
        companion[n:, :-n] = np.eye(n * 7)
        roots = np.linalg.eigvals(companion)
        return {'weight_ft': float(w), 'spectral_radius': float(np.abs(roots).max()), 'roots_above_one': int((np.abs(roots) > 1 + 1e-06).sum()), 'unit_roots': int((np.abs(roots - 1) < 1e-06).sum())}


def load_macro_inputs(config):
    """Load original BB raw inputs; no writes to the input directory."""
    return BBFTPTMacroEngine(config)


def build_macro_version(engine, cutoff_q):
    """Return a cached, prefix-refitted FT/PT news version."""
    if not isinstance(engine, BBFTPTMacroEngine):
        raise TypeError("engine must be created by load_macro_inputs(config)")
    return engine.build_version(cutoff_q)


## 5. 株式入力・感応度推定

入力監査、時点照会、元15節の統計関数、個別株への拡張です。

In [ ]:
"""Local-file inputs for the TOPIX500 BB-news factor notebook.

All prices/returns are read without forward filling.  Membership histories and
market-cap availability dates are queried as of each formation date.  This cell
contains no data downloads and does not modify source files.
"""
from pathlib import Path
import hashlib
import json
import re
import warnings
import numpy as np
import pandas as pd


SF_IO_EXAMPLE_CONFIG = {
    "as_of": None,                         # None: run date, Asia/Tokyo
    "returns": {
        "path": "stock_inputs/returns.csv", "format": "auto",
        "layout": "long", "frequency": "monthly", "kind": "returns",
        "unit": "decimal", "encoding": "utf-8-sig", "sheet_name": 0,
        "columns": {"date": "date", "security_id": "security_id", "value": "return"},
    },
    "caps": {
        "path": "stock_inputs/market_caps.csv", "format": "auto", "layout": "long",
        "columns": {"date": "date", "security_id": "security_id", "value": "market_cap"},
        "max_age_days": 62, "availability_lag_days": 0,
        "definition": "total_market_cap",   # e.g. total_market_cap / free_float_market_cap
        "currency": "JPY",
    },
    "membership": {
        "path": "stock_inputs/membership.csv", "format": "auto", "mode": "intervals",
        "columns": {"security_id": "security_id", "effective_from": "effective_from",
                    "effective_to": "effective_to"},
    },
    "sectors": None,                       # Optional static or dated long table
    "market": None,                        # Optional monthly returns/TR-index series
    "cash": None,
    "cash_missing_policy": "zero_explicit", # Explicit research assumption, recorded in audit
    "trading_calendar": None,              # Optional file {path, columns:{date:...}}
    "id_pad_length": None,
}


def sf_io_security_id(value, pad_length=None):
    """Keep identifiers as strings; remove Excel's integer-valued '.0' only."""
    if pd.isna(value):
        raise ValueError("security_id が欠損しています。")
    out = str(value).strip()
    if out.lower() in {"", "nan", "none", "null", "<na>"}:
        raise ValueError("security_id が空欄です。")
    if re.fullmatch(r"[0-9]+\.0+", out):
        out = out.split(".")[0]
    if pad_length is not None and out.isdigit():
        out = out.zfill(int(pad_length))
    return out


def sf_io_timestamp(value, month_label_end=True):
    """Date-only inputs are local Japanese dates; explicit zones become JST."""
    if value is None or pd.isna(value) or str(value).strip() == "":
        return pd.NaT
    if isinstance(value, pd.Period):
        stamp = value.to_timestamp(how="end").normalize()
    elif month_label_end and re.fullmatch(r"\d{4}-\d{2}", str(value).strip()):
        stamp = pd.Period(str(value).strip(), freq="M").to_timestamp(how="end").normalize()
    else:
        stamp = pd.Timestamp(value)
    if stamp.tzinfo is not None:
        stamp = stamp.tz_convert("Asia/Tokyo").tz_localize(None)
    return stamp


def sf_io_dates(values, label, allow_missing=False):
    values = pd.Series(values, copy=False)
    # Parsing unique date values is much cheaper than per-row parsing daily panels.
    lookup = {}
    for raw in values.dropna().unique():
        try:
            lookup[raw] = sf_io_timestamp(raw)
        except Exception as exc:
            raise ValueError(f"{label}: 日付を解釈できません: {raw!r}") from exc
    result = pd.to_datetime(values.map(lookup))
    if not allow_missing and result.isna().any():
        raise ValueError(f"{label}: 欠損日付があります。")
    return result


def sf_io_asof_stamp(date):
    """Formation date denotes end of day unless an explicit time is supplied."""
    stamp = sf_io_timestamp(date)
    if pd.isna(stamp):
        raise ValueError("as-of 日付が欠損です。")
    if stamp == stamp.normalize():
        stamp = stamp + pd.Timedelta(days=1) - pd.Timedelta(nanoseconds=1)
    return stamp


def sf_io_last_complete_month(as_of=None):
    stamp = (pd.Timestamp.now(tz="Asia/Tokyo").tz_localize(None)
             if as_of is None else sf_io_timestamp(as_of))
    period = stamp.to_period("M")
    return period if stamp.day == stamp.days_in_month else period - 1


def sf_io_input_status(cfg):
    """Return all configured paths together so missing inputs are reported once."""
    records = []
    for role in ("returns", "caps", "membership", "sectors", "market", "cash", "trading_calendar"):
        spec = cfg.get(role)
        required = role in {"returns", "caps", "membership"}
        path = spec.get("path") if isinstance(spec, dict) else None
        configured = path is not None and str(path).strip() != ""
        records.append({"role": role, "path": str(path) if configured else "",
                        "required": required, "configured": configured,
                        "exists": bool(configured and Path(path).expanduser().is_file())})
    return pd.DataFrame(records)


def sf_io_read_table(spec, role, audit=None):
    path = Path(spec["path"]).expanduser()
    if not path.is_file():
        raise FileNotFoundError(f"{role}: {path}")
    fmt = str(spec.get("format", "auto")).lower()
    if fmt == "auto":
        fmt = path.suffix.lower().lstrip(".")
    cols = spec.get("columns", {})
    id_col = cols.get("security_id", "security_id")
    dtype = {id_col: "string"} if spec.get("layout", "long") != "wide" else None
    try:
        if fmt in {"csv", "tsv", "txt"}:
            sep = spec.get("sep", "\t" if fmt == "tsv" else ",")
            table = pd.read_csv(path, encoding=spec.get("encoding", "utf-8-sig"), sep=sep,
                                dtype=dtype, keep_default_na=True)
        elif fmt in {"xlsx", "xls", "excel"}:
            table = pd.read_excel(path, sheet_name=spec.get("sheet_name", 0), dtype=dtype,
                                  engine=spec.get("engine"))
        elif fmt in {"parquet", "pq"}:
            table = pd.read_parquet(path)
        else:
            raise ValueError(f"{role}: format={fmt!r} は未対応です。CSV/Excel/Parquetを指定してください。")
    except ImportError as exc:
        package = "openpyxl（古いxlsはxlrd）" if fmt in {"xlsx", "xls", "excel"} else "pyarrow または fastparquet"
        raise ImportError(f"{role}: このファイル形式の読み取りには {package} が必要です。CSVなら追加依存不要です。") from exc
    if not isinstance(table, pd.DataFrame) or table.empty:
        raise ValueError(f"{role}: 入力が空です。テンプレートのヘッダーだけでは分析できません。")
    if table.columns.duplicated().any():
        raise ValueError(f"{role}: 重複列があります。")
    if audit is not None:
        digest = hashlib.sha256()
        with path.open("rb") as handle:
            for block in iter(lambda: handle.read(1024 * 1024), b""):
                digest.update(block)
        audit.setdefault("files", []).append({"role": role, "path": str(path.resolve()),
                "bytes": path.stat().st_size, "sha256": digest.hexdigest(),
                "rows": int(len(table)), "columns": [str(c) for c in table.columns]})
    return table


def sf_io_rename(table, spec, required, role):
    mapping = {canonical: spec.get("columns", {}).get(canonical, canonical)
               for canonical in set(required) | {"available_at", "member", "sector"}}
    missing = [source for canonical, source in mapping.items()
               if canonical in required and source not in table.columns]
    if missing:
        raise ValueError(f"{role}: 必須列がありません: {missing}。columns設定を確認してください。")
    actual = {source: canonical for canonical, source in mapping.items() if source in table.columns}
    if len(actual) != len([s for s in mapping.values() if s in table.columns]):
        raise ValueError(f"{role}: columns設定で同じ入力列を複数の意味へ割り当てています。")
    result = table.rename(columns=actual).copy()
    if result.columns.duplicated().any():
        raise ValueError(f"{role}: columns変換後の列名が重複しました。")
    return result


def sf_io_panel_long(spec, role, audit=None, pad_length=None, single=False):
    table = sf_io_read_table(spec, role, audit)
    layout = str(spec.get("layout", "long")).lower()
    if layout not in {"long", "wide"}:
        raise ValueError(f"{role}: layoutはlong/wideを指定してください。")
    if layout == "wide":
        date_col = spec.get("columns", {}).get("date", "date")
        if date_col not in table:
            raise ValueError(f"{role}: wide形式のdate列 {date_col!r} がありません。")
        ignore = set(spec.get("ignore_columns", []))
        value_cols = [c for c in table.columns if c != date_col and c not in ignore]
        if not value_cols:
            raise ValueError(f"{role}: wide形式の銘柄列がありません。")
        if single and len(value_cols) != 1:
            raise ValueError(f"{role}: 単一のbenchmark系列を指定してください。不要列はignore_columnsへ: {value_cols}")
        table = table[[date_col] + value_cols].melt(id_vars=[date_col], var_name="security_id", value_name="value")
        table = table.rename(columns={date_col: "date"})
    else:
        required = ["date", "value"] + ([] if single else ["security_id"])
        table = sf_io_rename(table, spec, required, role)
        if single:
            table["security_id"] = "__SERIES__"
    table["date"] = sf_io_dates(table["date"], role + ".date")
    table["security_id"] = table["security_id"].map(lambda x: sf_io_security_id(x, pad_length))
    table["value"] = pd.to_numeric(table["value"], errors="raise").astype(float)
    if np.isinf(table["value"]).any():
        raise ValueError(f"{role}: 無限大の値があります。")
    if table.duplicated(["date", "security_id"]).any():
        dup = table.loc[table.duplicated(["date", "security_id"], keep=False), ["date", "security_id"]].head()
        raise ValueError(f"{role}: 日付・銘柄の重複があります:\n{dup}")
    return table.sort_values(["date", "security_id"]).reset_index(drop=True)


def sf_io_to_monthly(table, spec, role, cutoff, trading_calendar=None, audit=None):
    """Build complete monthly returns; no implicit price or return filling."""
    frequency = str(spec.get("frequency", "monthly")).lower()
    if frequency not in {"monthly", "daily"}:
        raise ValueError(f"{role}: frequencyはmonthly/dailyを指定してください。")
    kind = spec.get("kind")
    if kind not in {"returns", "tr_levels"}:
        raise ValueError(f"{role}: kind='returns' または 'tr_levels' を明示してください。")
    data = table.copy()
    if kind == "returns":
        unit = spec.get("unit")
        if unit not in {"decimal", "percent"}:
            raise ValueError(f"{role}: returnsのunitをdecimal/percentで明示してください。")
        if unit == "percent":
            data["value"] /= 100.0
        if (data.value.dropna() < -1.0 - 1e-12).any():
            raise ValueError(f"{role}: -100%未満の単純リターンがあります。単位を確認してください。")
        data.loc[data.value.between(-1.0 - 1e-12, -1.0), "value"] = -1.0
    elif (data.value.dropna() < 0).any():
        raise ValueError(f"{role}: TR指数の水準が負です。")
    if frequency == "monthly":
        data["month"] = data.date.dt.to_period("M")
        if data.duplicated(["month", "security_id"]).any():
            raise ValueError(f"{role}: 同じ月に複数行があります。dailyデータならfrequency='daily'を指定してください。")
        wide = data.pivot(index="month", columns="security_id", values="value").sort_index()
        wide = wide.reindex(pd.period_range(wide.index.min(), wide.index.max(), freq="M", name="month"))
        if kind == "tr_levels":
            previous = wide.shift(1)
            out = wide.divide(previous).subtract(1).where(previous > 0)
        else:
            out = wide
    else:
        data["date"] = data.date.dt.normalize()
        if data.duplicated(["date", "security_id"]).any():
            raise ValueError(f"{role}: 日次入力に同じ日付の複数観測があります。")
        wide = data.pivot(index="date", columns="security_id", values="value").sort_index()
        calendar = (pd.DatetimeIndex(trading_calendar).normalize().unique().sort_values()
                    if trading_calendar is not None else pd.DatetimeIndex(wide.index))
        if trading_calendar is not None:
            # Keep expected dates around both endpoint months.  Otherwise a
            # truncated input file could turn its partial last month into a
            # superficially complete monthly return.
            first_boundary = wide.index.min().to_period("M").start_time.normalize()
            last_boundary = min(wide.index.max().to_period("M"), cutoff).end_time.normalize()
            calendar = calendar[(calendar >= first_boundary) & (calendar <= last_boundary)]
        else:
            calendar = calendar[calendar.to_period("M") <= cutoff]
        if len(calendar) == 0:
            raise ValueError(f"{role}: as_of以前の取引日がありません。")
        wide = wide.loc[wide.index.to_period("M") <= cutoff]
        unknown = wide.index.difference(calendar)
        if len(unknown):
            raise ValueError(f"{role}: trading_calendarにない観測日があります: {unknown[:5].tolist()}")
        wide = wide.reindex(calendar)
        month_index = calendar.to_period("M")
        periods = pd.period_range(month_index.min(), month_index.max(), freq="M", name="month")
        out = pd.DataFrame(np.nan, index=periods, columns=wide.columns, dtype=float)
        if kind == "returns":
            for month in periods:
                block = wide.loc[month_index == month]
                if block.empty:
                    continue
                complete = block.notna().all(axis=0)
                compounded = (1.0 + block).prod(axis=0, min_count=len(block)) - 1.0
                # A verified terminal -100% return makes end wealth zero, even
                # when later observations stop following a delisting.
                terminal = block.eq(-1.0).any(axis=0)
                out.loc[month, complete] = compounded.loc[complete]
                out.loc[month, terminal] = -1.0
        else:
            # Each boundary is an actual last-calendar-day observation, not the
            # last nonmissing value for each security.
            boundaries = wide.groupby(month_index).tail(1).copy()
            boundaries.index = boundaries.index.to_period("M")
            boundaries = boundaries.reindex(periods)
            prev = boundaries.shift(1)
            out = boundaries.divide(prev).subtract(1).where(prev > 0)
            gaps = wide.isna().groupby(month_index).any().reindex(periods, fill_value=True)
            out = out.mask(gaps)
        if audit is not None:
            audit.setdefault("notes", []).append(
                f"{role}: dailyデータを月次化。calendar=" +
                ("provided" if trading_calendar is not None else "observed-date union; exchange calendar completeness is not certified") +
                "; 欠損を前方補完しない。")
    out = out.loc[out.index <= cutoff].astype(float)
    out.index = pd.PeriodIndex(out.index, freq="M", name="month")
    out.columns = pd.Index(out.columns.astype(str), name="security_id")
    if out.empty:
        raise ValueError(f"{role}: as_of以前の完了月データがありません。")
    if audit is not None:
        audit.setdefault("series", {})[role] = {
            "start": str(out.index.min()), "end": str(out.index.max()),
            "months": len(out), "securities": len(out.columns),
            "nonmissing_returns": int(out.notna().sum().sum()),
            "large_abs_monthly_moves_over_50pct": int((out.abs() > .5).sum().sum()),
            "terminal_minus_100pct": int(out.eq(-1).sum().sum()),
            "large_moves_removed": False,
        }
    return out


def sf_io_bool(value):
    if pd.isna(value):
        raise ValueError("membership.memberが欠損しています。")
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    text = str(value).strip().lower()
    if text in {"1", "1.0", "true", "t", "yes", "y"}:
        return True
    if text in {"0", "0.0", "false", "f", "no", "n"}:
        return False
    raise ValueError(f"membership.memberを真偽値に変換できません: {value!r}")


def sf_io_load_membership(spec, audit, pad_length=None):
    data = sf_io_read_table(spec, "membership", audit)
    mode = spec.get("mode", "intervals")
    if mode == "intervals":
        data = sf_io_rename(data, spec, ["security_id", "effective_from", "effective_to"], "membership")
        data["security_id"] = data.security_id.map(lambda x: sf_io_security_id(x, pad_length))
        data["effective_from"] = sf_io_dates(data.effective_from, "membership.effective_from")
        data["effective_to"] = sf_io_dates(data.effective_to, "membership.effective_to", allow_missing=True)
        if ((data.effective_to.notna()) & (data.effective_to <= data.effective_from)).any():
            raise ValueError("membership: effective_toはeffective_fromより後（排他的終了日）にしてください。")
        if "available_at" in data:
            data["available_at"] = sf_io_dates(data.available_at, "membership.available_at")
        else:
            data["available_at"] = data.effective_from
            audit["notes"].append("membership.available_atなし: 組入履歴は各effective_from時点で既知だったと仮定。")
        data = data.sort_values(["security_id", "effective_from"])
        for sid, block in data.groupby("security_id", sort=False):
            starts = block.effective_from.to_numpy()
            ends = block.effective_to.to_numpy()
            for idx in range(len(block) - 1):
                if pd.isna(ends[idx]) or ends[idx] > starts[idx + 1]:
                    raise ValueError(f"membership: {sid}の組入期間が重複しています。")
        return {"mode": mode, "data": data.reset_index(drop=True)}
    if mode == "snapshots":
        if spec.get("complete_snapshots") is not True:
            raise ValueError("membership snapshotsは各dateの全構成銘柄が必要です。complete_snapshots=Trueを明示してください。")
        data = sf_io_rename(data, spec, ["date", "security_id"], "membership")
        data["date"] = sf_io_dates(data.date, "membership.date")
        data["security_id"] = data.security_id.map(lambda x: sf_io_security_id(x, pad_length))
        data["member"] = data.member.map(sf_io_bool) if "member" in data else True
        data["available_at"] = (sf_io_dates(data.available_at, "membership.available_at")
                                if "available_at" in data else data.date)
        if data.duplicated(["date", "security_id"]).any():
            raise ValueError("membership: 同じsnapshot内で銘柄IDが重複しています。")
        # A complete snapshot is usable only when every row is available.
        snapshots = data.groupby("date").available_at.max().rename("available_at")
        return {"mode": mode, "data": data.sort_values(["date", "security_id"]), "snapshots": snapshots}
    raise ValueError("membership.modeはintervals/snapshotsを指定してください。")


def sf_io_load_caps(spec, audit, pad_length=None):
    data = sf_io_panel_long(spec, "caps", audit, pad_length).rename(columns={"value": "market_cap"})
    if (data.market_cap.dropna() < 0).any():
        raise ValueError("caps: 時価総額が負です。")
    if "available_at" in data:
        data["available_at"] = sf_io_dates(data.available_at, "caps.available_at")
    else:
        lag = int(spec.get("availability_lag_days", 0))
        if lag < 0:
            raise ValueError("caps.availability_lag_daysは0以上にしてください。")
        data["available_at"] = data.date + pd.Timedelta(days=lag)
        audit["notes"].append(f"caps.available_atなし: observation date + {lag}暦日の利用可能日を仮定。")
    if (data.available_at < data.date).any():
        raise ValueError("caps: available_atが時価総額の観測日より前です。")
    return data[["date", "security_id", "market_cap", "available_at"]].sort_values(["date", "security_id"])


def sf_io_load_sectors(spec, audit, pad_length=None):
    if not isinstance(spec, dict) or not spec.get("path"):
        return None
    data = sf_io_read_table(spec, "sectors", audit)
    mode = spec.get("mode", "static")
    required = ["security_id", "sector"] + (["date"] if mode == "dated" else [])
    data = sf_io_rename(data, spec, required, "sectors")
    data["security_id"] = data.security_id.map(lambda x: sf_io_security_id(x, pad_length))
    data["sector"] = data.sector.astype("string")
    if mode == "static":
        if data.security_id.duplicated().any():
            raise ValueError("sectors static: 銘柄IDが重複しています。")
        data["date"] = pd.Timestamp("1900-01-01")
        data["available_at"] = data.date
        audit["notes"].append("sector分類はstatic入力。歴史的な業種変更を再現しない。")
    elif mode == "dated":
        data["date"] = sf_io_dates(data.date, "sectors.date")
        data["available_at"] = (sf_io_dates(data.available_at, "sectors.available_at")
                                if "available_at" in data else data.date)
        if data.duplicated(["date", "security_id"]).any():
            raise ValueError("sectors: 同じ日付・銘柄に複数の業種があります。")
    else:
        raise ValueError("sectors.modeはstatic/datedを指定してください。")
    return data[["date", "security_id", "sector", "available_at"]].sort_values(["date", "security_id"])


def load_stock_bundle(cfg):
    """Load and validate local data.  All monthly returns use decimal units."""
    audit = {"files": [], "notes": [], "series": {}, "return_unit": "decimal",
             "definitions": {
                 "returns_kind": cfg.get("returns", {}).get("kind"),
                 "source_return_basis": cfg.get("returns", {}).get("return_basis", "not_recorded_in_local_file_config"),
                 "source_currency": cfg.get("returns", {}).get("currency", "not_recorded_in_local_file_config"),
                 "market_cap_definition": cfg.get("caps", {}).get("definition", "not_recorded_in_local_file_config"),
                 "market_cap_currency": cfg.get("caps", {}).get("currency", "not_recorded_in_local_file_config"),
             }}
    status = sf_io_input_status(cfg)
    bad = status.loc[(status.required | status.configured) & ~status.exists]
    if not bad.empty:
        message = "\n".join(f"- {r.role}: {r.path or '(path未設定)'}" for r in bad.itertuples())
        raise FileNotFoundError("次の入力ファイルを指定してください。空テンプレートは分析データとして使えません。\n" + message)
    cutoff = sf_io_last_complete_month(cfg.get("as_of"))
    audit["last_complete_month"] = str(cutoff)
    pad = cfg.get("id_pad_length")
    calendar = None
    cal_spec = cfg.get("trading_calendar")
    if isinstance(cal_spec, dict) and cal_spec.get("path"):
        frame = sf_io_read_table(cal_spec, "trading_calendar", audit)
        frame = sf_io_rename(frame, cal_spec, ["date"], "trading_calendar")
        calendar = pd.DatetimeIndex(sf_io_dates(frame.date, "trading_calendar.date")).normalize()
        if calendar.has_duplicates:
            raise ValueError("trading_calendar: 重複日があります。")
        calendar = calendar.sort_values()
    returns_long = sf_io_panel_long(cfg["returns"], "returns", audit, pad)
    monthly = sf_io_to_monthly(returns_long, cfg["returns"], "returns", cutoff, calendar, audit)
    bundle = {"monthly_returns": monthly, "caps": sf_io_load_caps(cfg["caps"], audit, pad),
              "membership": sf_io_load_membership(cfg["membership"], audit, pad),
              "sectors": sf_io_load_sectors(cfg.get("sectors"), audit, pad),
              "audit": audit, "config": cfg, "input_status": status,
              "trading_calendar": calendar}
    for role in ("market", "cash"):
        spec = cfg.get(role)
        if isinstance(spec, dict) and spec.get("path"):
            long = sf_io_panel_long(spec, role, audit, single=True)
            series = sf_io_to_monthly(long, spec, role, cutoff, calendar, audit).iloc[:, 0]
            bundle[role + "_monthly"] = series.reindex(monthly.index).rename(role)
        elif role == "cash" and cfg.get("cash_missing_policy", "nan") == "zero_explicit":
            bundle["cash_monthly"] = pd.Series(0.0, index=monthly.index, name="cash")
            audit["cash_assumption"] = "zero_explicit: every monthly cash return is assumed 0; not observed data"
        elif role == "cash" and cfg.get("cash_missing_policy", "nan") == "error":
            raise ValueError("cash入力がありません。cash_missing_policyでerror/nan/zero_explicitを選んでください。")
        else:
            if role == "cash" and cfg.get("cash_missing_policy", "nan") not in {"nan", "zero_explicit", "error"}:
                raise ValueError("cash_missing_policyはerror/nan/zero_explicitを指定してください。")
            bundle[role + "_monthly"] = pd.Series(np.nan, index=monthly.index, name=role)
            audit[role + "_assumption"] = "not supplied; missing, not replaced with zero"
    bundle["security_ids"] = pd.Index(monthly.columns, name="security_id")
    # Coverage is reported rather than removing dates shared by any missing stock.
    audit["coverage_monthly"] = monthly.notna().sum(axis=1).rename("stocks_with_return").reset_index().astype({"month": str}).to_dict("records")
    return bundle


def universe_at(bundle, date):
    """Historical TOPIX500 members known and effective at the formation date."""
    stamp = sf_io_asof_stamp(date)
    obj = bundle["membership"]
    data = obj["data"]
    if obj["mode"] == "intervals":
        mask = ((data.effective_from <= stamp) &
                (data.effective_to.isna() | (data.effective_to > stamp)) &
                (data.available_at <= stamp))
        selected = data.loc[mask, "security_id"]
    else:
        snapshots = obj["snapshots"]
        usable = snapshots[(snapshots.index <= stamp) & (snapshots <= stamp)]
        if usable.empty:
            raise ValueError(f"{stamp.date()}: この時点で利用できる完全な構成銘柄snapshotがありません。")
        snapshot_date = usable.index.max()
        selected = data.loc[(data.date == snapshot_date) & data.member, "security_id"]
    if selected.empty:
        raise ValueError(f"{stamp.date()}: 構成履歴からTOPIX500ユニバースを確定できません。現在の構成で代替しません。")
    if selected.duplicated().any():
        raise ValueError(f"{stamp.date()}: 有効な構成履歴が重複しています。")
    return pd.Index(sorted(selected.astype(str).tolist()), name="security_id")


def caps_at(bundle, date, max_age_days=None):
    """Latest available positive capitalization, with an explicit staleness cap."""
    stamp = sf_io_asof_stamp(date)
    data = bundle["caps"]
    allowed = data.loc[(data.date <= stamp) & (data.available_at <= stamp)]
    allowed = allowed.sort_values(["security_id", "date", "available_at"]).groupby("security_id", sort=False).tail(1).copy()
    allowed["age_days"] = (stamp.normalize() - allowed.date.dt.normalize()).dt.days
    limit = (bundle.get("config", {}).get("caps", {}).get("max_age_days", 62)
             if max_age_days is None else max_age_days)
    if limit is not None:
        if int(limit) < 0:
            raise ValueError("caps max_age_daysは0以上かNoneにしてください。")
        allowed = allowed.loc[allowed.age_days <= int(limit)]
    allowed = allowed.loc[allowed.market_cap.notna() & (allowed.market_cap > 0)].set_index("security_id")
    result = allowed.market_cap.astype(float).sort_index().rename("market_cap")
    result.attrs["age_days"] = allowed.age_days.to_dict()
    result.attrs["observed_dates"] = allowed.date.astype(str).to_dict()
    result.attrs["available_at"] = allowed.available_at.astype(str).to_dict()
    return result


def sectors_at(bundle, date):
    data = bundle.get("sectors")
    if data is None:
        return pd.Series(dtype="string", name="sector", index=pd.Index([], name="security_id"))
    stamp = sf_io_asof_stamp(date)
    allowed = data.loc[(data.date <= stamp) & (data.available_at <= stamp)]
    allowed = allowed.sort_values(["security_id", "date", "available_at"]).groupby("security_id", sort=False).tail(1)
    return allowed.set_index("security_id").sector.sort_index().rename("sector")


def input_schema_frames():
    """Empty, canonical templates only; no fabricated observations."""
    return {
        "returns.csv": pd.DataFrame(columns=["date", "security_id", "return"]),
        "market_caps.csv": pd.DataFrame(columns=["date", "security_id", "market_cap", "available_at"]),
        "membership.csv": pd.DataFrame(columns=["security_id", "effective_from", "effective_to", "available_at"]),
        "membership_snapshots.csv": pd.DataFrame(columns=["date", "security_id", "member", "available_at"]),
        "sectors.csv": pd.DataFrame(columns=["date", "security_id", "sector", "available_at"]),
        "market.csv": pd.DataFrame(columns=["date", "return"]),
        "cash.csv": pd.DataFrame(columns=["date", "return"]),
    }


def write_input_templates(directory, enabled=False):
    """Opt-in header-only CSV templates.  Existing files are never overwritten."""
    if not enabled:
        raise ValueError("テンプレートを作る場合のみenabled=Trueを明示してください。")
    directory = Path(directory).expanduser()
    frames = input_schema_frames()
    existing = [str(directory / name) for name in frames if (directory / name).exists()]
    if existing:
        raise FileExistsError("既存ファイルを上書きしません: " + ", ".join(existing))
    directory.mkdir(parents=True, exist_ok=True)
    for name, frame in frames.items():
        frame.to_csv(directory / name, index=False, encoding="utf-8-sig")
    return [str(directory / name) for name in frames]


In [ ]:
def sf_bh_adjust(pvals):
    x=np.asarray(pvals,float);result=np.full(x.shape,np.nan);valid=np.isfinite(x)
    p=x[valid];m=len(p)
    if m==0:return result
    order=np.argsort(p);q=np.minimum.accumulate((p[order]*m/np.arange(1,m+1))[::-1])[::-1]
    tmp=np.empty(m);tmp[order]=np.minimum(q,1.0);result[valid]=tmp
    return result

def sf_ols_hac(y,X,lags=4,time_step=1):
    a=np.asarray(X,float);b=np.asarray(y,float);n,k=a.shape
    if n<=k+2: raise ValueError('OLSの残差自由度が不足')
    if np.linalg.matrix_rank(a)<k: raise ValueError('資産回帰の説明変数がランク落ち')
    pinv=np.linalg.pinv(a);beta=pinv@b;resid=b-a@beta;bread=pinv@pinv.T
    score=a*resid[:,None];meat=score.T@score
    # 欠損で四半期間隔が飛んでも隣接行を隣接四半期とみなさない。
    times=y.index.asi8 if isinstance(y,pd.Series) and isinstance(y.index,pd.PeriodIndex) else np.arange(n)*time_step
    pos={int(t):i for i,t in enumerate(times)}
    for j in range(1,min(lags,n-1)+1):
        pairs=[(i,pos[int(t)-j*time_step]) for i,t in enumerate(times) if int(t)-j*time_step in pos]
        if pairs:
            now,prev=map(np.array,zip(*pairs))
            cross=score[now].T@score[prev];meat+=(1-j/(lags+1))*(cross+cross.T)
    cov=(n/(n-k))*bread@meat@bread
    cov=(cov+cov.T)/2;se=np.sqrt(np.maximum(np.diag(cov),0))
    t=np.divide(beta,se,out=np.full_like(beta,np.nan),where=se>0)
    p=2*stats.t.sf(np.abs(t),n-k);crit=stats.t.ppf(.975,n-k)
    return {'beta':beta,'se':se,'p':p,'low':beta-crit*se,'high':beta+crit*se,
            'resid':resid,'n':n,'df':n-k,'cov':cov,'r2':1-(resid@resid)/np.sum((b-b.mean())**2)}

def sf_partial_corr(y,x,Z):
    z=np.asarray(Z,float);yr=np.asarray(y)-z@np.linalg.lstsq(z,np.asarray(y),rcond=None)[0]
    xr=np.asarray(x)-z@np.linalg.lstsq(z,np.asarray(x),rcond=None)[0]
    if np.std(xr)<1e-12 or np.std(yr)<1e-12: return np.nan
    return float(np.corrcoef(xr,yr)[0,1])


In [ ]:
"""Stock sensitivities; embedded verbatim in the delivered notebook."""
import json
import numpy as np
import pandas as pd
from scipy import stats


SF_PRIMARY_SPECS = {
    "total": {"target": "news_total", "controls": ["news_growth"]},
    "cost_push": {"target": "news_cost_push", "controls": ["news_labor", "news_growth"]},
    "labor": {"target": "news_labor", "controls": ["news_cost_push", "news_growth"]},
}
SF_DESCRIPTIVE_SPECS = {
    name.removeprefix("news_"): {"target": name, "controls": ["news_growth"]}
    for name in ["news_total", "news_cost_push", "news_labor", "news_energy", "news_food", "news_shortage"]
}


def sf_rolling_compound(monthly, months=12):
    """Simple compounded return; requires every month and permits a true -100%."""
    if not isinstance(monthly.index, pd.PeriodIndex) or monthly.index.freqstr != "M":
        raise TypeError("monthly must have a monthly PeriodIndex")
    if monthly.index.has_duplicates:
        raise ValueError("Duplicate return months")
    if not isinstance(months, (int, np.integer)) or months < 1:
        raise ValueError("months must be a positive integer")
    if monthly.empty:
        return monthly.copy()
    regular = monthly.sort_index().reindex(pd.period_range(monthly.index.min(), monthly.index.max(), freq="M"))
    values = regular.to_numpy(dtype=float)
    if np.isinf(values).any():
        raise ValueError("Infinite returns are invalid")
    if (values[np.isfinite(values)] < -1).any():
        raise ValueError("A simple return below -100% is invalid")
    return (1.0 + regular).rolling(months, min_periods=months).apply(np.prod, raw=True) - 1.0


def sf_quarter_end_returns12(monthly, cash=None, return_basis="total"):
    """Same 12-month compounded percentage-return target as source notebook §15."""
    annual = 100.0 * sf_rolling_compound(monthly, 12)
    if return_basis == "excess_cash":
        if cash is None:
            raise ValueError("excess_cash requires an explicit cash series")
        cash12 = 100.0 * sf_rolling_compound(cash.reindex(monthly.index), 12)
        annual = annual.sub(cash12, axis=0) if isinstance(annual, pd.DataFrame) else annual - cash12
    elif return_basis != "total":
        raise ValueError("return_basis must be 'total' or 'excess_cash'")
    annual = annual.loc[annual.index.month.isin([3, 6, 9, 12])].copy()
    annual.index = annual.index.asfreq("Q")
    annual.index.name = "quarter"
    return annual


def sf_fit_sensitivities(returns12, news, universe, cfg, specs=None, sample_end=None, market12=None):
    """Pairwise stock samples, common calendar window, exact calendar-lag HAC.

    Returns (estimates, coverage). No p-value screening, return winsorizing,
    cross-stock complete-case requirement, or implicit missing-value imputation.
    """
    specs = specs or SF_PRIMARY_SPECS
    for name, obj in [("returns12", returns12), ("news", news)]:
        if not isinstance(obj.index, pd.PeriodIndex) or obj.index.freqstr != "Q-DEC":
            raise TypeError(f"{name} must use a calendar-quarter PeriodIndex (Q-DEC)")
        if obj.index.has_duplicates or obj.index.isna().any():
            raise ValueError(f"{name} has duplicate or missing quarters")
        if not obj.index.is_monotonic_increasing:
            raise ValueError(f"{name} must be sorted by quarter")
    if news.empty:
        raise ValueError("news is empty")
    universe = list(map(str, universe))
    if len(set(universe)) != len(universe):
        raise ValueError("Duplicate security IDs in the estimation universe")
    end = pd.Period(sample_end, freq="Q") if sample_end is not None else news.index.max()
    nw = news.loc[:end].copy()
    yall = returns12.loc[:end].copy()
    window = cfg.get("window_quarters")
    if window is not None:
        start = end - int(window) + 1
        nw = nw.loc[start:]
        yall = yall.loc[start:]
    if cfg.get("sample_start"):
        start = pd.Period(cfg["sample_start"], freq="Q")
        nw, yall = nw.loc[start:], yall.loc[start:]
    if cfg.get("control_market", False):
        if market12 is None or market12.dropna().empty:
            raise ValueError("control_market=True requires a valid market return series")
        nw["market_return12"] = market12.reindex(nw.index)
    method = cfg.get("estimator", "ols")
    if method not in {"ols", "wls"}:
        raise ValueError("estimator must be ols or wls")
    min_n = int(cfg.get("min_quarters", 40))
    if min_n < 12:
        raise ValueError("min_quarters must be >=12; very short samples are not supported")
    half_life = float(cfg.get("half_life_quarters", 20))
    if not np.isfinite(half_life) or half_life <= 0:
        raise ValueError("half_life_quarters must be positive")
    rows, coverage = [], []
    for factor, spec in specs.items():
        target = spec["target"]
        controls = list(spec["controls"])
        if cfg.get("control_market", False) and "market_return12" not in controls:
            controls.append("market_return12")
        names = [target] + controls
        if len(set(names)) != len(names):
            raise ValueError(f"Duplicated regressors for {factor}: {names}")
        missing = set(names).difference(nw.columns)
        if missing:
            raise ValueError(f"Missing macro columns for {factor}: {sorted(missing)}")
        for sid in map(str, universe):
            audit = {"security_id": sid, "factor": factor, "target": target, "sample_end_requested": str(end)}
            if sid not in yall.columns:
                coverage.append({**audit, "n": 0, "status": "no_return_series"})
                continue
            d = pd.concat([yall[sid].rename("y"), nw[names]], axis=1).replace([np.inf, -np.inf], np.nan).dropna()
            audit.update(n=len(d), sample_start=str(d.index.min()) if len(d) else None,
                         sample_end=str(d.index.max()) if len(d) else None)
            if len(d) < min_n:
                coverage.append({**audit, "status": "insufficient_quarters"})
                continue
            max_gap = int(cfg.get("max_last_observation_gap_quarters", 1))
            if end.ordinal - d.index.max().ordinal > max_gap:
                coverage.append({**audit, "status": "stale_return_history"})
                continue
            if d.y.std(ddof=1) < 1e-12 or any(d[c].std(ddof=1) < 1e-12 for c in names):
                coverage.append({**audit, "status": "constant_response_or_regressor"})
                continue
            X = np.column_stack([np.ones(len(d)), d[names].to_numpy()])
            if np.linalg.matrix_rank(X) < X.shape[1]:
                coverage.append({**audit, "status": "rank_deficient_regressors"})
                continue
            zscaled = np.column_stack([np.ones(len(d)), ((d[names]-d[names].mean()) / d[names].std()).to_numpy()])
            cond = float(np.linalg.cond(zscaled))
            Z = X[:, [0] + list(range(2, X.shape[1]))]
            xr = d[target].to_numpy() - Z @ np.linalg.lstsq(Z, d[target].to_numpy(), rcond=None)[0]
            if np.std(xr, ddof=1) < float(cfg.get("min_residual_news_sd_pp", 1e-8)):
                coverage.append({**audit, "status": "unidentified_target_after_controls"})
                continue
            # HAC applied to transformed WLS scores if recency weighting is requested.
            if method == "wls":
                ages = end.ordinal - d.index.asi8
                weights = np.exp(-np.log(2.0) * ages / half_life)
                weights /= weights.mean()
                rootw = np.sqrt(weights)
                yfit = pd.Series(d.y.to_numpy() * rootw, index=d.index)
                xfit = X * rootw[:, None]
                weighted_Z = Z * rootw[:, None]
                yrw = yfit.to_numpy() - weighted_Z @ np.linalg.lstsq(weighted_Z, yfit.to_numpy(), rcond=None)[0]
                xw = d[target].to_numpy() * rootw
                xrw = xw - weighted_Z @ np.linalg.lstsq(weighted_Z, xw, rcond=None)[0]
                # Weighted residual correlation uses the residual inner product;
                # residuals are orthogonal to sqrt(weight), not the unweighted constant.
                denom = np.linalg.norm(yrw) * np.linalg.norm(xrw)
                pcor = float(yrw @ xrw / denom) if denom > 0 else np.nan
                weighted_denom = weights.sum() - np.sum(weights**2)/weights.sum()
                weighted_residual_sd = float(np.linalg.norm(xrw)/np.sqrt(weighted_denom))
                if weighted_residual_sd < float(cfg.get("min_residual_news_sd_pp", 1e-8)):
                    coverage.append({**audit, "status": "unidentified_recent_target_after_controls"})
                    continue
            else:
                weights = np.ones(len(d))
                yfit, xfit = d.y, X
                pcor = sf_partial_corr(d.y, d[target], Z)
                weighted_residual_sd = float(np.std(xr, ddof=1))
            fits = {lag: sf_ols_hac(yfit, xfit, lags=lag) for lag in (4, 8)}
            b = fits[8]["beta"]
            fitted = X @ b
            rss = float(np.sum(weights * (d.y.to_numpy()-fitted)**2))
            mean_y = np.average(d.y, weights=weights)
            tss = float(np.sum(weights * (d.y.to_numpy()-mean_y)**2))
            row = {**audit, "beta_raw": float(b[1]), "beta_per_1pp": float(b[1]),
                   "partial_corr": pcor, "partial_corr_growth": pcor,
                   "news_sd_pp": float(d[target].std(ddof=1)),
                   "residual_news_sd_pp": float(np.std(xr, ddof=1)),
                   "residual_news_sd_weighted_pp": weighted_residual_sd,
                   "beta_per_1sd": float(b[1] * d[target].std(ddof=1)),
                   "se": float(fits[8]["se"][1]), "r2": 1-rss/tss if tss else np.nan,
                   "intercept": float(b[0]), "condition_standardized_X": cond,
                   "high_collinearity_flag": cond > float(cfg.get("condition_warning", 100)),
                   "controls": json.dumps(controls), "control_coefficients": json.dumps(dict(zip(controls, map(float, b[2:])))),
                   "estimator": method, "weighted_observation_count": float(weights.sum()**2 / np.sum(weights**2)),
                   "return_basis": cfg.get("return_basis", "total"), "status": "eligible"}
            for lag, fit in fits.items():
                row.update({f"p_hac{lag}": float(fit["p"][1]),
                            f"se_hac{lag}": float(fit["se"][1]),
                            f"ci_low_hac{lag}": float(fit["low"][1]),
                            f"ci_high_hac{lag}": float(fit["high"][1])})
            rows.append(row)
            coverage.append({**audit, "status": "eligible"})
    out = pd.DataFrame(rows)
    if not out.empty:
        for lag in (4, 8):
            out[f"q_bh_within_factor_hac{lag}"] = out.groupby("factor")[f"p_hac{lag}"].transform(sf_bh_adjust)
            out[f"q_bh_all_factors_hac{lag}"] = sf_bh_adjust(out[f"p_hac{lag}"])
    return out, pd.DataFrame(coverage)


def sf_shrink_betas(estimates, min_cross_section=5):
    """Vasicek-style cross-sectional prior; raw-beta HAC inference stays separate.

    Prior variance = sample variance of raw estimated betas (the simple empirical
    prior used in the cited approach), not a claim to de-noise correlated errors.
    The prior is reestimated on the current eligible cross-section only.
    """
    if estimates.empty:
        return estimates.copy()
    required = {"factor", "beta_raw", "se"}
    if not required.issubset(estimates):
        raise ValueError(f"Missing shrinkage inputs: {sorted(required-set(estimates))}")
    if not isinstance(min_cross_section, (int, np.integer)) or min_cross_section < 2:
        raise ValueError("min_cross_section must be an integer >=2")
    # A concatenated history may have repeated original row labels.  Work on
    # a unique row index so that .loc never mixes different formation vintages.
    out = estimates.copy().reset_index(drop=True)
    for col in ["prior_mean", "prior_variance", "shrinkage_weight", "beta_shrunk"]:
        out[col] = np.nan
    out["shrinkage_status"] = pd.Series("invalid_beta_or_se", index=out.index, dtype=object)
    keys = [x for x in ("formation_month", "factor") if x in out]
    grouping = keys[0] if len(keys) == 1 else keys
    for _, ids in out.groupby(grouping, sort=False).groups.items():
        part = out.loc[ids]
        good = np.isfinite(part.beta_raw) & np.isfinite(part.se) & (part.se >= 0)
        good_ids = part.index[good]
        values = part.loc[good, "beta_raw"]
        if len(values) < min_cross_section:
            out.loc[good_ids, "shrinkage_status"] = "insufficient_cross_section"
            continue
        mu, tau2 = float(values.mean()), float(values.var(ddof=1))
        variance = out.loc[good_ids, "se"] ** 2
        if not np.isfinite(tau2) or tau2 <= 1e-18:
            lam = pd.Series(0.0, index=good_ids)
            status = "flat_cross_section"
        else:
            lam = tau2 / (tau2 + variance)
            status = "ok"
        out.loc[good_ids, "prior_mean"] = mu
        out.loc[good_ids, "prior_variance"] = tau2
        out.loc[good_ids, "shrinkage_weight"] = lam
        out.loc[good_ids, "beta_shrunk"] = mu + lam * (out.loc[good_ids, "beta_raw"] - mu)
        out.loc[good_ids, "shrinkage_status"] = status
    return out


def sf_project_neutral_weights(raw_weights, sectors=None, market_betas=None):
    """Optional minimum-distance projection; it is not the original 2x3 factor.

    Constraints are imposed on estimated/recorded exposures. Rescale to gross 2;
    do not rescale near-zero residual weights. All inputs must be known at signal.
    """
    w = raw_weights.astype(float).copy()
    if w.empty or not np.isfinite(w).all():
        raise ValueError("Invalid starting weights for neutrality projection")
    columns = [np.ones(len(w))]
    if sectors is not None:
        sec = sectors.reindex(w.index)
        if sec.isna().any():
            raise ValueError("Sector-neutral variant requires sectors for every eligible stock")
        columns.extend(pd.get_dummies(sec.astype(str), dtype=float).to_numpy().T)
    if market_betas is not None:
        mb = market_betas.reindex(w.index)
        if not np.isfinite(mb).all():
            raise ValueError("Market-neutral variant requires valid ex-ante market betas")
        columns.append(mb.to_numpy())
    C = np.column_stack(columns)
    # Project using the design itself, avoiding squaring its condition number.
    # Redundant sector dummies plus the intercept are handled by least squares.
    result = w.to_numpy() - C @ np.linalg.lstsq(C, w.to_numpy(), rcond=None)[0]
    gross = float(np.abs(result).sum())
    if gross < 1e-8:
        raise ValueError("Neutrality constraints remove nearly all factor exposure")
    result *= 2.0 / gross
    if np.max(np.abs(C.T @ result)) > 1e-8:
        raise ArithmeticError("Neutrality constraints failed numerical verification")
    return pd.Series(result, index=w.index)


def sf_exante_market_betas(monthly_returns, market_monthly, as_of_month, window=60, min_months=36):
    """Backward-only monthly market betas used solely by optional neutrality."""
    end = pd.Period(as_of_month, freq="M")
    start = end-int(window)+1
    r = monthly_returns.loc[start:end]
    m = market_monthly.reindex(r.index)
    out = {}
    for sid in r:
        d = pd.concat([r[sid].rename("y"), m.rename("m")], axis=1).dropna()
        if len(d) >= min_months and d.m.var(ddof=1) > 1e-14:
            out[sid] = float(np.cov(d.y, d.m, ddof=1)[0, 1] / d.m.var(ddof=1))
    return pd.Series(out, dtype=float)


def sf_postranking_block_ci(d, names, reps=0, block_length=8, seed=20261111):
    """Paired contiguous-quarter block bootstrap, conditional on fixed inputs.

    Each block contains the observed factor return and *all* macro regressors.
    Blocks cannot cross a missing calendar quarter.  This resamples the supplied
    realized strategy, not the original securities, BB model or sorting process.
    """
    if not isinstance(reps, (int, np.integer)) or reps < 0 or (0 < reps < 20):
        raise ValueError("bootstrap reps must be 0 or an integer >=20")
    if not isinstance(block_length, (int, np.integer)) or block_length < 1:
        raise ValueError("bootstrap block_length must be an integer >=1")
    result = {"bootstrap_ci_low": np.nan, "bootstrap_ci_high": np.nan,
              "bootstrap_reps_requested": int(reps), "bootstrap_accepted": 0,
              "bootstrap_valid_blocks": 0, "bootstrap_block_length": int(block_length),
              "bootstrap_status": "disabled",
              "bootstrap_scope": "fixed_generated_news_and_realized_strategy_not_BB_or_sort_reestimation"}
    if reps == 0:
        return result
    if not isinstance(d.index, pd.PeriodIndex) or d.index.freqstr != "Q-DEC":
        raise TypeError("Bootstrap inputs must use calendar-quarter PeriodIndex")
    if d.index.has_duplicates or not d.index.is_monotonic_increasing:
        raise ValueError("Bootstrap inputs need unique sorted quarters")
    values = d[["y"]+list(names)].to_numpy(float)
    if not np.isfinite(values).all():
        raise ValueError("Bootstrap inputs must be finite complete-case rows")
    n = len(d)
    if n < block_length:
        result["bootstrap_status"] = "fewer_observations_than_requested_block_length"
        return result
    ordinals = d.index.asi8
    starts = np.asarray([i for i in range(n-block_length+1)
                         if np.all(np.diff(ordinals[i:i+block_length]) == 1)], dtype=int)
    result["bootstrap_valid_blocks"] = int(len(starts))
    if len(starts) < 2:
        result["bootstrap_status"] = "insufficient_contiguous_blocks"
        return result
    rng = np.random.default_rng(seed)
    estimates = []
    offsets = np.arange(block_length)
    for _ in range(reps):
        selected = rng.choice(starts, size=int(np.ceil(n/block_length)), replace=True)
        take = (selected[:, None]+offsets[None, :]).ravel()[:n]
        sample = values[take]
        X = np.column_stack([np.ones(n), sample[:, 1:]])
        if np.linalg.matrix_rank(X) != X.shape[1]:
            continue
        beta = np.linalg.lstsq(X, sample[:, 0], rcond=None)[0][1]
        if np.isfinite(beta):
            estimates.append(float(beta))
    result["bootstrap_accepted"] = len(estimates)
    if len(estimates) < max(20, int(np.ceil(.90*reps))):
        result["bootstrap_status"] = "insufficient_full_rank_bootstrap_draws"
        return result
    low, high = np.quantile(estimates, [.025, .975])
    result.update(bootstrap_ci_low=float(low), bootstrap_ci_high=float(high),
                  bootstrap_status="ok_conditional_percentile")
    return result


def sf_postranking_diagnostics(backtest, news, specs=None, min_quarters=12,
                                market_monthly=None, control_market=False,
                                bootstrap_reps=0, block_length=8, seed=20261111):
    """Horizon-matched post-ranking exposure, using realized funded returns.

    The first valid 12-month observation starts after 12 *post-formation*
    months.  ``control_market=True`` adds a matching 12-month market excess
    return using the same cash series as the collateral accounting.  HAC
    intervals, and the optional paired block-bootstrap intervals, condition on
    the supplied generated macro series and selected trading strategy; they do
    not include BB or sorting estimation uncertainty.  The bootstrap uses OLS
    percentile intervals, while HAC uses the source notebook's t approximation.
    """
    specs = specs or SF_PRIMARY_SPECS
    if not isinstance(news.index, pd.PeriodIndex) or news.index.freqstr != "Q-DEC":
        raise TypeError("news must use a calendar-quarter PeriodIndex")
    if news.index.has_duplicates:
        raise ValueError("news has duplicate quarters")
    ret12 = backtest["returns12"].copy()
    bases = ["excess_return12_gross", "excess_return12_net"]
    required = {"strategy", "month", *bases}
    if not required.issubset(ret12):
        raise ValueError(f"Accounting output is missing columns: {sorted(required-set(ret12))}")
    if not isinstance(min_quarters, (int, np.integer)) or min_quarters < 8:
        raise ValueError("min_quarters must be an integer >=8")
    nw = news.copy()
    if control_market:
        if market_monthly is None or market_monthly.dropna().empty:
            raise ValueError("Market-controlled diagnostics need market_monthly")
        history = backtest["monthly"][["month", "cash_return"]].drop_duplicates()
        if history.month.duplicated().any():
            raise ValueError("Different strategies have inconsistent cash returns")
        cash_monthly = history.set_index("month").cash_return.sort_index()
        cash_monthly.index = pd.PeriodIndex(cash_monthly.index, freq="M")
        nw["market_return12"] = sf_quarter_end_returns12(
            market_monthly, cash=cash_monthly, return_basis="excess_cash").reindex(nw.index)
    factor_map = {}
    history = backtest.get("monthly")
    if isinstance(history, pd.DataFrame) and "factor" in history:
        for strategy, group in history.groupby("strategy"):
            mapped = group.factor.drop_duplicates()
            if len(mapped) != 1:
                raise ValueError(f"{strategy}: inconsistent factor names in accounting output")
            factor_map[str(strategy)] = str(mapped.iloc[0])
    rows = []
    for strategy, g in ret12.groupby("strategy"):
        if g.month.duplicated().any():
            raise ValueError(f"{strategy}: duplicate post-ranking months")
        g = g.set_index("month").sort_index()
        g.index = pd.PeriodIndex(g.index, freq="M")
        g = g.loc[g.index.month.isin([3, 6, 9, 12])]
        g.index = g.index.asfreq("Q")
        target_name = factor_map.get(str(strategy), str(strategy).split("|")[0])
        if target_name not in specs:
            raise ValueError(f"No exposure specification for strategy {strategy!r}; factor={target_name!r}")
        spec = specs[target_name]
        names = [spec["target"]] + list(spec["controls"])
        if control_market and "market_return12" not in names:
            names.append("market_return12")
        if len(names) != len(set(names)):
            raise ValueError(f"Duplicated post-ranking regressors: {names}")
        missing = set(names).difference(nw.columns)
        if missing:
            raise ValueError(f"Missing post-ranking news columns: {sorted(missing)}")
        for basis in bases:
            d = pd.concat([(100*g[basis]).rename("y"), nw[names]], axis=1)
            d = d.replace([np.inf, -np.inf], np.nan).dropna()
            context = {"strategy": strategy, "factor": target_name,
                       "return_kind": basis, "n": len(d), "target": names[0],
                       "market_controlled": bool(control_market),
                       "controls": json.dumps(names[1:]),
                       "inference": "conditional_on_generated_news_and_realized_selection"}
            if len(d) < max(min_quarters, len(names)+4):
                rows.append({**context, "status": "insufficient_postranking_quarters"})
                continue
            X = np.column_stack([np.ones(len(d)), d[names]])
            if np.linalg.matrix_rank(X) < X.shape[1] or d.y.std() < 1e-12:
                rows.append({**context, "status": "singular_postranking_regression"})
                continue
            fits = {lag: sf_ols_hac(d.y, X, lag) for lag in (4, 8)}
            fit = fits[8]
            Z = X[:, [0] + list(range(2, X.shape[1]))]
            row = {**context, "start": str(d.index.min()), "end": str(d.index.max()),
                   "beta": float(fit["beta"][1]),
                   "partial_corr": sf_partial_corr(d.y, d[names[0]], Z),
                   "r2": float(fit["r2"]), "status": "ok"}
            for lag, fitted in fits.items():
                row.update({f"p_hac{lag}": float(fitted["p"][1]),
                            f"se_hac{lag}": float(fitted["se"][1]),
                            f"ci_low_hac{lag}": float(fitted["low"][1]),
                            f"ci_high_hac{lag}": float(fitted["high"][1])})
            for j, name in enumerate(names, 1):
                row["exposure_"+name] = float(fit["beta"][j])
            row.update(sf_postranking_block_ci(d, names, reps=bootstrap_reps,
                                                block_length=block_length, seed=seed))
            rows.append(row)
    out = pd.DataFrame(rows)
    if not out.empty and "p_hac8" in out:
        out["q_bh_hac8"] = sf_bh_adjust(out.p_hac8)
    return out


## 6. 因子形成・会計・研究フロー

In [ ]:
"""TOPIX500 inflation-factor formation and monthly collateral accounting.

This source is embedded directly in the delivered notebook.  All returns are
decimal returns.  A target dated ``month`` is held during that month and must
have been formed using information available by the preceding month end.

The monthly convention is deliberately explicit: targets are expressed as a
fraction of beginning-of-month NAV; trading and stock-borrow costs are paid
from cash at month end.  They therefore do not reduce the return-earning stock
notional within that month.  This is a research accounting convention, not a
claim to reproduce next-day executions or broker margin/financing terms.
"""

import numpy as np
import pandas as pd


SF_CELLS = ("SL", "SN", "SH", "BL", "BN", "BH")


class FactorFormationError(ValueError):
    """An eligible, diversified 2x3 factor could not be formed."""


def _fa_security_id(values):
    ids = pd.Index([str(v).strip() for v in values], name="security_id")
    if ids.isin(["", "nan", "None", "<NA>"]).any():
        raise ValueError("security_id に空欄があります。入力時に銘柄IDを正規化してください。")
    if ids.has_duplicates:
        raise ValueError("同一スナップショット内で security_id が重複しています。")
    return ids


def _fa_series(value, column, name):
    if isinstance(value, pd.Series):
        out = value.copy()
    elif isinstance(value, pd.DataFrame):
        if column not in value:
            raise ValueError(f"{name}: {column!r} 列がありません。")
        out = value.set_index("security_id")[column] if "security_id" in value else value[column]
    else:
        raise TypeError(f"{name} は Series または DataFrame で指定してください。")
    out.index = _fa_security_id(out.index)
    out = pd.to_numeric(out, errors="raise").astype(float)
    out.name = column
    return out


def _fa_members(membership, default_index):
    if membership is None:
        return pd.Index(default_index, name="security_id")
    if isinstance(membership, pd.DataFrame):
        if "security_id" not in membership:
            raise ValueError("membership の security_id 列がありません。")
        m = membership
        if "member" in m:
            if not m.member.dropna().isin([True, False, 0, 1]).all():
                raise ValueError("membership.member は真偽値にしてください。")
            m = m.loc[m.member.fillna(False).astype(bool)]
        return _fa_security_id(m.security_id)
    if isinstance(membership, pd.Series):
        if not membership.dropna().isin([True, False, 0, 1]).all():
            raise ValueError("membership Series は security_id index の真偽値にしてください。")
        return _fa_security_id(membership.index[membership.fillna(False).astype(bool)])
    return _fa_security_id(list(membership))


def _fa_caps_for_labels(labels, caps, membership):
    cap = _fa_series(caps, "market_cap", "caps")
    labs = labels.copy()
    if "security_id" in labs:
        labs = labs.set_index("security_id")
    labs.index = _fa_security_id(labs.index)
    required = {"size_group", "score_group", "cell"}
    if not required.issubset(labs.columns):
        raise ValueError(f"labels に必要な列がありません: {sorted(required-set(labs.columns))}")
    members = _fa_members(membership, labs.index)
    labs = labs.drop(columns=["market_cap", "cell_weight", "factor_weight"], errors="ignore")
    labs["market_cap"] = cap.reindex(labs.index)
    reason = pd.Series("", index=labs.index, dtype=object)
    reason.loc[~labs.index.isin(members)] = "not_current_member"
    reason.loc[(reason == "") & ~np.isfinite(labs.market_cap)] = "missing_or_nonfinite_cap"
    reason.loc[(reason == "") & (labs.market_cap <= 0)] = "nonpositive_cap"
    exclusions = reason[reason != ""].rename("reason").reset_index()
    return labs.loc[reason == ""].copy(), exclusions


def sf_reweight_2x3(labels, caps, membership=None, min_cell_size=5, gross_target=2.0):
    """Reweight previously formed labels without using new scores or re-sorting.

    Membership removals are applied as of the decision date.  New members are
    included only at the next score formation.  Missing caps are explicitly
    returned in ``exclusions``.  Every cell must retain ``min_cell_size`` names;
    no cell, side or factor is replaced with cash when that condition fails.
    """
    if not isinstance(min_cell_size, (int, np.integer)) or min_cell_size < 1:
        raise ValueError("min_cell_size は1以上の整数です。")
    if not np.isfinite(gross_target) or gross_target <= 0:
        raise ValueError("gross_target は正の有限値です。")
    labs, exclusions = _fa_caps_for_labels(labels, caps, membership)
    if not labs.cell.isin(SF_CELLS).all():
        raise ValueError("不正な2x3セルがあります。")
    if not (labs.cell == labs.size_group.astype(str) + labs.score_group.astype(str)).all():
        raise ValueError("cell と size_group/score_group が一致しません。")
    counts = labs.groupby("cell").size().reindex(SF_CELLS, fill_value=0)
    if (counts < min_cell_size).any():
        bad = counts[counts < min_cell_size].to_dict()
        raise FactorFormationError(
            f"2x3セルの銘柄数不足: {bad}; min_cell_size={min_cell_size}。"
            "不足期間を0収益や除外期間として扱わないでください。"
        )
    totals = labs.groupby("cell").market_cap.transform("sum")
    if (~np.isfinite(totals) | (totals <= 0)).any():
        raise FactorFormationError("セル時価総額合計が正の有限値ではありません。")
    labs["cell_weight"] = labs.market_cap / totals
    coefficients = {"SL": -0.5, "SN": 0.0, "SH": 0.5,
                    "BL": -0.5, "BN": 0.0, "BH": 0.5}
    labs["factor_weight"] = labs.cell.map(coefficients) * labs.cell_weight * (gross_target / 2.0)
    sums = labs.groupby("cell").cell_weight.sum().reindex(SF_CELLS)
    if not np.allclose(sums, 1.0, atol=1e-12):
        raise AssertionError("セル内ウェイトが1に合計されません。")
    fw = labs.factor_weight.copy().rename("weight")
    if not (np.isclose(fw.sum(), 0, atol=1e-12)
            and np.isclose(fw.abs().sum(), gross_target, atol=1e-12)):
        raise AssertionError("形成ウェイトのnet/grossが指定と一致しません。")
    return {
        "labels": labs.sort_index().reset_index(),
        "cell_weights": labs.reset_index()[["security_id", "cell", "cell_weight"]]
            .rename(columns={"cell_weight": "weight"}),
        "factor_weights": fw.sort_index(),
        "cell_counts": counts,
        "exclusions": exclusions,
    }


def sf_form_2x3(scores, caps, membership=None, score_column="beta_shrunk",
                min_cell_size=5, score_quantiles=(0.30, 0.70),
                flat_tolerance=1e-12, gross_target=2.0):
    """Independent size-median x common-score-quantile 2x3 formation.

    Equal scores remain in the same group; no security-ID tie-break fabricates
    exposure differences.  Size uses the eligible universe's market-cap median.
    Score boundaries use that same universe, not size-group-specific quantiles.
    This is a TOPIX500 adaptation, not the official developed-market FF breaks.
    """
    low_q, high_q = score_quantiles
    if not (0 < low_q < high_q < 1):
        raise ValueError("score_quantiles は 0 < low < high < 1 にしてください。")
    s = _fa_series(scores, score_column, "scores")
    cap = _fa_series(caps, "market_cap", "caps")
    members = _fa_members(membership, s.index)
    frame = pd.DataFrame({"score": s, "market_cap": cap.reindex(s.index)})
    reason = pd.Series("", index=frame.index, dtype=object)
    reason.loc[~frame.index.isin(members)] = "not_current_member"
    reason.loc[(reason == "") & ~np.isfinite(frame.score)] = "missing_or_nonfinite_score"
    reason.loc[(reason == "") & ~np.isfinite(frame.market_cap)] = "missing_or_nonfinite_cap"
    reason.loc[(reason == "") & (frame.market_cap <= 0)] = "nonpositive_cap"
    excluded = reason[reason != ""].rename("reason").reset_index()
    frame = frame.loc[reason == ""].copy()
    if frame.empty:
        raise FactorFormationError("感応度と時価総額のある適格銘柄がありません。")
    lo, hi = frame.score.quantile([low_q, high_q]).to_numpy()
    if not np.isfinite([lo, hi]).all() or hi - lo <= flat_tolerance:
        raise FactorFormationError(
            "スコアの30/70分位点に十分な差がありません。"
            "同値スコアを銘柄ID順で人工的にHigh/Lowへ分けません。"
        )
    size_cut = float(frame.market_cap.median())
    frame["size_group"] = np.where(frame.market_cap <= size_cut, "S", "B")
    frame["score_group"] = np.select(
        [frame.score <= lo, frame.score > hi], ["L", "H"], default="N")
    frame["cell"] = frame.size_group + frame.score_group
    result = sf_reweight_2x3(frame, cap, members,
                             min_cell_size=min_cell_size, gross_target=gross_target)
    result["exclusions"] = excluded
    result["breakpoints"] = {
        "size_method": "eligible_universe_market_cap_median",
        "size_median": size_cut,
        "score_column": score_column,
        "low_quantile": float(low_q), "high_quantile": float(high_q),
        "low_breakpoint": float(lo), "high_breakpoint": float(hi),
        "n_eligible": int(len(frame)), "n_excluded": int(len(excluded)),
        "gross_target": float(gross_target),
        "tie_policy": "equal_scores_same_group; L<=q30, N<=q70, H>q70",
    }
    return result


def _fa_months(values):
    if isinstance(values, pd.PeriodIndex):
        if values.freqstr != "M":
            raise ValueError("月次会計のindexはPeriodIndex(freq='M')にしてください。")
        return values.copy()
    try:
        return pd.PeriodIndex(values, freq="M")
    except Exception as exc:
        raise ValueError("年月を月次Periodへ変換できません。") from exc


def _fa_timestamp_jst(value):
    stamp = pd.Timestamp(value)
    if pd.isna(stamp):
        return pd.NaT
    return stamp.tz_convert("Asia/Tokyo").tz_localize(None) if stamp.tzinfo else stamp


def _fa_target_table(target_weights_long):
    t = target_weights_long.copy()
    required = {"month", "strategy", "security_id", "weight"}
    if not required.issubset(t):
        raise ValueError(f"target_weights_long の必要列: {sorted(required-set(t))}")
    if t.empty:
        raise ValueError("対象ウェイトが空です。未形成を0収益へ置き換えません。")
    t["month"] = _fa_months(t.month)
    if t.month.isna().any() or t.strategy.isna().any() or t.security_id.isna().any():
        raise ValueError("ウェイトのmonth/strategy/security_idに欠損があります。")
    t["strategy"] = t.strategy.astype(str)
    t["security_id"] = t.security_id.astype(str).str.strip()
    if t.security_id.isin(["", "nan", "None", "<NA>"]).any():
        raise ValueError("ウェイトのsecurity_idが不正です。")
    if t.duplicated(["month", "strategy", "security_id"]).any():
        raise ValueError("ウェイトの month/strategy/security_id が重複しています。")
    t["weight"] = pd.to_numeric(t.weight, errors="raise").astype(float)
    if not np.isfinite(t.weight).all():
        raise ValueError("ウェイトに非有限値があります。")
    if "signal_month" in t:
        t["signal_month"] = _fa_months(t.signal_month)
        if t.signal_month.isna().any() or (t.signal_month >= t.month).any():
            raise ValueError("signal_month は必ず保有monthより前にしてください。")
    if "available_at" in t:
        availability = t.available_at.map(_fa_timestamp_jst)
        if availability.isna().any():
            raise ValueError("available_atに欠損があります。")
        cutoffs = t.month.map(lambda m: (m - 1).end_time)
        if (availability > cutoffs).any():
            raise ValueError("前月末より後に利用可能となる情報がウェイトに混入しています。")
    return t.sort_values(["strategy", "month", "security_id"]).reset_index(drop=True)


def _fa_monthly_inputs(monthly_returns, cash_returns):
    r = monthly_returns.copy()
    if not isinstance(r, pd.DataFrame) or r.empty:
        raise ValueError("monthly_returns は空でないDataFrameにしてください。")
    r.index = _fa_months(r.index)
    if r.index.has_duplicates or r.index.isna().any():
        raise ValueError("monthly_returns の月が重複または欠損しています。")
    r.columns = _fa_security_id(r.columns)
    r = r.sort_index().apply(pd.to_numeric, errors="raise").astype(float)
    if np.isinf(r.to_numpy()).any():
        raise ValueError("リターンに無限大があります。")
    if (r < -1).any().any():
        raise ValueError("単純リターンが-100%を下回っています。-100%自体は許容します。")
    if not isinstance(cash_returns, pd.Series):
        raise ValueError("cash_returns Seriesが必要です。0仮定も明示的なSeriesで指定してください。")
    cash = cash_returns.copy()
    cash.index = _fa_months(cash.index)
    if cash.index.has_duplicates or cash.index.isna().any():
        raise ValueError("cash_returnsの月が重複または欠損しています。")
    cash = pd.to_numeric(cash, errors="raise").astype(float).sort_index()
    if np.isinf(cash).any() or (cash.dropna() <= -1).any():
        raise ValueError("cash_returns は-100%を超える有限の単純リターンにしてください。")
    return r, cash


def _fa_held_returns(r, month, weights, strategy):
    held = weights[weights.abs() > 1e-14]
    if month not in r.index:
        raise ValueError(f"{strategy} {month}: 保有月リターンがありません。")
    missing_ids = held.index.difference(r.columns)
    if len(missing_ids):
        raise ValueError(f"{strategy} {month}: 保有銘柄の列がありません: {list(missing_ids[:10])}")
    realized = r.loc[month].reindex(held.index)
    missing = realized.index[realized.isna()]
    if len(missing):
        raise ValueError(
            f"{strategy} {month}: 保有銘柄のリターン欠損: {list(missing[:10])}。"
            "上場廃止・企業行動・リターンを修正してください。欠損銘柄の除外や0補完は行いません。"
        )
    return held, realized


def sf_cell_returns(monthly_returns, cell_weights_long):
    """Gross long-only returns of all six cells, requiring every held return."""
    if cell_weights_long is None or len(cell_weights_long) == 0:
        return pd.DataFrame(columns=["month", "strategy", "cell", "return", "n_stocks"])
    cells = cell_weights_long.copy()
    required = {"month", "strategy", "cell", "security_id", "weight"}
    if not required.issubset(cells):
        raise ValueError(f"cell_weights_long の必要列: {sorted(required-set(cells))}")
    cells["month"] = _fa_months(cells.month)
    cells["security_id"] = cells.security_id.astype(str)
    cells["weight"] = pd.to_numeric(cells.weight, errors="raise").astype(float)
    if (cells.weight < 0).any() or not np.isfinite(cells.weight).all():
        raise ValueError("セルウェイトは非負の有限値にしてください。")
    if cells.duplicated(["month", "strategy", "cell", "security_id"]).any():
        raise ValueError("セルウェイトが重複しています。")
    if not cells.cell.isin(SF_CELLS).all():
        raise ValueError("cellはSL/SN/SH/BL/BN/BHのいずれかです。")
    r = monthly_returns.copy()
    r.index = _fa_months(r.index)
    r.columns = _fa_security_id(r.columns)
    if r.index.has_duplicates:
        raise ValueError("monthly_returnsの月が重複しています。")
    out = []
    for (month, strategy), grp in cells.groupby(["month", "strategy"], sort=True):
        if set(grp.cell) != set(SF_CELLS):
            raise ValueError(f"{strategy} {month}: 6セルがそろっていません。")
        for cell, c in grp.groupby("cell"):
            weights = c.set_index("security_id").weight
            if not np.isclose(weights.sum(), 1, atol=1e-10):
                raise ValueError(f"{strategy} {month} {cell}: セルウェイト合計が1ではありません。")
            held, rr = _fa_held_returns(r, month, weights, f"{strategy}/{cell}")
            if (rr < -1).any() or not np.isfinite(rr).all():
                raise ValueError(f"{strategy} {month} {cell}: リターンが不正です。")
            out.append(dict(month=month, strategy=str(strategy), cell=cell,
                            **{"return": float(held @ rr)}, n_stocks=int(len(held))))
    return pd.DataFrame(out).sort_values(["strategy", "month", "cell"]).reset_index(drop=True)


def sf_factor_returns12(monthly, window=12):
    """Compounded collateralized factor total return less compounded cash.

    This is NOT prod(1 + monthly high-minus-low) - 1, and is NOT the difference
    between independently compounded long and short leg portfolios.
    """
    required = {"month", "strategy", "total_return_gross", "total_return_net", "cash_return"}
    if not required.issubset(monthly):
        raise ValueError(f"monthly の必要列: {sorted(required-set(monthly))}")
    if not isinstance(window, int) or window < 1:
        raise ValueError("windowは1以上の整数です。")
    records = []
    for strategy, g in monthly.groupby("strategy", sort=True):
        g = g.copy()
        g["month"] = _fa_months(g.month)
        g = g.sort_values("month").set_index("month")
        expected = pd.period_range(g.index.min(), g.index.max(), freq="M")
        if g.index.has_duplicates or not g.index.equals(expected):
            raise ValueError(f"{strategy}: 12か月集計の月次系列に重複・期間欠落があります。")
        cols = ["total_return_gross", "total_return_net", "cash_return"]
        if g[cols].isna().any().any() or not np.isfinite(g[cols].to_numpy()).all():
            raise ValueError(f"{strategy}: 12か月集計に欠損・非有限リターンがあります。")
        if (g[cols] <= -1).any().any():
            raise ValueError(f"{strategy}: 非正のwealthを複利集計できません。")
        z = np.expm1(np.log1p(g[cols]).rolling(window, min_periods=window).sum())
        z = z.rename(columns={"total_return_gross": "total_return12_gross",
                              "total_return_net": "total_return12_net",
                              "cash_return": "cash_return12"})
        z["excess_return12_gross"] = z.total_return12_gross - z.cash_return12
        z["excess_return12_net"] = z.total_return12_net - z.cash_return12
        z["strategy"] = strategy
        z["window_months"] = window
        records.append(z.reset_index())
    return pd.concat(records, ignore_index=True) if records else pd.DataFrame()


def sf_factor_performance(monthly):
    """Descriptive net/gross funded performance; annualization assumes months."""
    out = []
    for strategy, g in monthly.groupby("strategy", sort=True):
        g = g.sort_values("month")
        n = len(g)
        for basis in ("gross", "net"):
            total = g[f"total_return_{basis}"].to_numpy(float)
            excess = total - g.cash_return.to_numpy(float)
            wealth = np.r_[1.0, np.cumprod(1.0 + total)]
            vol = float(np.std(excess, ddof=1) * np.sqrt(12)) if n > 1 else np.nan
            mean = float(np.mean(excess) * 12)
            out.append(dict(strategy=strategy, basis=basis, n_months=n,
                            start=str(g.month.iloc[0]), end=str(g.month.iloc[-1]),
                            cumulative_total_return=float(wealth[-1]-1),
                            cagr_total=float(wealth[-1]**(12/n)-1),
                            annualized_mean_excess=mean, annualized_excess_vol=vol,
                            sharpe=mean/vol if np.isfinite(vol) and vol > 0 else np.nan,
                            max_drawdown=float(np.min(wealth/np.maximum.accumulate(wealth)-1)),
                            annualized_turnover=float(g.turnover_net.mean()*12),
                            annualized_trading_cost=float(g.trading_cost.mean()*12),
                            annualized_borrow_cost=float(g.borrow_cost.mean()*12)))
    return pd.DataFrame(out)


def sf_run_factor_backtest(monthly_returns, target_weights_long, cash_returns,
                           cost_bps=10.0, borrow_bps=0.0, cell_weights_long=None,
                           require_dollar_neutral=True, expected_gross=2.0):
    """Monthly close-to-close research accounting of fully specified targets.

    Required target columns: month (holding month), strategy, security_id,
    weight.  Optional signal_month and available_at are checked for look-ahead.
    Every month after a strategy starts must have a complete target.  Missing
    held returns, missing cash returns, gaps and non-positive NAV stop execution.

    Gross funded return = rf + sum_i w_i * (r_i - rf).
    Net funded return = gross funded return - trading cost - borrow cost.
    Turnover is sum_i abs(target_i - prior_end_weight_i), including initial
    establishment and closures.  Prior-end weights use the respective actual
    gross/net NAV.  Trading costs use net turnover and are paid at month end.
    Borrow is the annual rate / 12 times beginning-of-month short notional.
    Gross returns exclude both cost types.  Loan financing, collateral rebates,
    margin haircuts and intra-month execution slippage are not modeled.
    """
    for name, rate in [("cost_bps", cost_bps), ("borrow_bps", borrow_bps)]:
        if not np.isfinite(rate) or rate < 0:
            raise ValueError(f"{name}は非負の有限値にしてください。")
    r, cash = _fa_monthly_inputs(monthly_returns, cash_returns)
    targets = _fa_target_table(target_weights_long)
    monthly_rows, holding_rows = [], []
    for strategy, s in targets.groupby("strategy", sort=True):
        observed = pd.PeriodIndex(sorted(s.month.unique()), freq="M")
        expected = pd.period_range(observed.min(), observed.max(), freq="M")
        if not observed.equals(expected):
            missing = expected.difference(observed)
            raise ValueError(f"{strategy}: 開始後のtargetが欠けています: {list(map(str,missing))}")
        prev_net = pd.Series(dtype=float)
        prev_gross = pd.Series(dtype=float)
        nav_net = nav_gross = nav_cash = 1.0
        for month in expected:
            block = s.loc[s.month == month]
            weights = block.set_index("security_id").weight.astype(float)
            weights = weights[weights.abs() > 1e-14]
            if weights.empty:
                raise ValueError(f"{strategy} {month}: targetが実質的に空です。")
            net_exposure, gross_exposure = float(weights.sum()), float(weights.abs().sum())
            if require_dollar_neutral and not np.isclose(net_exposure, 0, atol=1e-10):
                raise ValueError(f"{strategy} {month}: net exposureが0ではありません: {net_exposure}")
            if expected_gross is not None and not np.isclose(gross_exposure, expected_gross, atol=1e-10):
                raise ValueError(f"{strategy} {month}: gross exposureが{expected_gross}ではありません: {gross_exposure}")
            held, rr = _fa_held_returns(r, month, weights, strategy)
            if month not in cash.index or not np.isfinite(cash.loc[month]):
                raise ValueError(f"{strategy} {month}: cash returnがありません。暗黙の0補完はしません。")
            rf = float(cash.loc[month])
            ids = weights.index.union(prev_net.index).union(prev_gross.index)
            target_aligned = weights.reindex(ids, fill_value=0.0)
            drift_net = prev_net.reindex(ids, fill_value=0.0)
            drift_gross = prev_gross.reindex(ids, fill_value=0.0)
            turnover_net = float((target_aligned-drift_net).abs().sum())
            turnover_gross = float((target_aligned-drift_gross).abs().sum())
            fee = float(cost_bps / 10000.0 * turnover_net)
            short_notional = float(-weights[weights < 0].sum())
            borrow = float(borrow_bps / 10000.0 / 12.0 * short_notional)
            stock_pnl = float(held @ rr)
            cash_pnl = float((1.0-net_exposure) * rf)
            total_gross = stock_pnl + cash_pnl
            total_net = total_gross - fee - borrow
            if not np.isfinite(total_net) or total_net <= -1 or total_gross <= -1:
                raise ValueError(
                    f"{strategy} {month}: NAVが非正になります (gross={total_gross}, net={total_net})。"
                    "破産後のリターンを継続・リセットしません。")
            old_nav_net, old_nav_gross = nav_net, nav_gross
            nav_gross *= 1+total_gross
            nav_net *= 1+total_net
            nav_cash *= 1+rf
            if not np.isfinite([nav_gross, nav_net, nav_cash]).all():
                raise ValueError(f"{strategy} {month}: NAVが非有限値になりました。")
            prev_net = held*(1+rr)/(1+total_net)
            prev_gross = held*(1+rr)/(1+total_gross)
            end_cash_weight_net = float(((1-net_exposure)*(1+rf)-fee-borrow)/(1+total_net))
            end_cash_weight_gross = float((1-net_exposure)*(1+rf)/(1+total_gross))
            if not (np.isclose(prev_net.sum()+end_cash_weight_net, 1, atol=1e-10)
                    and np.isclose(prev_gross.sum()+end_cash_weight_gross, 1, atol=1e-10)):
                raise AssertionError("保有株と担保現金の期末NAV恒等式が成立しません。")
            meta = {}
            for col in ("factor", "variant", "model_version", "signal_month", "available_at"):
                if col in block:
                    unique = block[col].drop_duplicates()
                    if len(unique) != 1:
                        raise ValueError(f"{strategy} {month}: {col} がtarget内で一致しません。")
                    meta[col] = unique.iloc[0]
            monthly_rows.append(dict(
                month=month, strategy=str(strategy), cash_return=rf,
                stock_pnl=stock_pnl, cash_pnl=cash_pnl,
                excess_return_gross=total_gross-rf, excess_return_net=total_net-rf,
                total_return_gross=total_gross, total_return_net=total_net,
                turnover_net=turnover_net, turnover_gross=turnover_gross,
                trading_cost=fee, borrow_cost=borrow,
                gross_exposure=gross_exposure, net_exposure=net_exposure,
                n_long=int((weights > 0).sum()), n_short=int((weights < 0).sum()),
                nav_gross=nav_gross, nav_net=nav_net, nav_cash=nav_cash,
                start_nav_gross=old_nav_gross, start_nav_net=old_nav_net,
                end_cash_weight_net=end_cash_weight_net,
                end_cash_weight_gross=end_cash_weight_gross,
                **meta))
            for sid in ids:
                weight = float(target_aligned.loc[sid])
                holding_rows.append(dict(
                    month=month, strategy=str(strategy), security_id=sid,
                    target_weight=weight, pretrade_weight_net=float(drift_net.loc[sid]),
                    trade_weight_net=weight-float(drift_net.loc[sid]),
                    security_return=float(rr.loc[sid]) if sid in rr.index else np.nan,
                    end_weight_net=float(prev_net.get(sid, 0.0)),
                    end_weight_gross=float(prev_gross.get(sid, 0.0))))
    monthly = pd.DataFrame(monthly_rows).sort_values(["strategy", "month"]).reset_index(drop=True)
    cells = sf_cell_returns(r, cell_weights_long)
    if not cells.empty:
        keys_monthly = set(zip(monthly.month, monthly.strategy))
        keys_cells = set(zip(cells.month, cells.strategy))
        if not keys_cells.issubset(keys_monthly):
            raise ValueError("セル収益に、対応するファクター月・strategyのない行があります。")
        # Neutrality variants are not the original six-cell FF combination.
        # The caller can provide cell diagnostics only for original strategies;
        # once a strategy is provided, however, its entire monthly history must
        # be present (no selective omission of a problematic month).
        provided_strategies = set(cells.strategy)
        required_keys = {(m, s) for m, s in keys_monthly if s in provided_strategies}
        if required_keys != keys_cells:
            raise ValueError("セル診断を指定したstrategy内に対象月の欠落があります。")
        for (month, strategy), c in cells.groupby(["month", "strategy"], sort=True):
            c = c.set_index("cell")["return"]
            spread = 0.5*(c.SH-c.SL+c.BH-c.BL)
            row = monthly.loc[(monthly.month==month)&(monthly.strategy==strategy)].iloc[0]
            if expected_gross is not None:
                spread *= expected_gross/2.0
                if not np.isclose(row.excess_return_gross, spread, atol=1e-10):
                    raise AssertionError(f"{strategy} {month}: セルH-Lと個別株ウェイト会計が一致しません。")
    return {
        "monthly": monthly,
        "holdings": pd.DataFrame(holding_rows),
        "cell_returns": cells,
        "returns12": sf_factor_returns12(monthly),
        "annualized_summary": sf_factor_performance(monthly),
        "accounting": {
            "return_unit": "decimal", "frequency": "monthly",
            "target_timing": "preceding_month_end_information; close_to_close_research",
            "nav_basis": "collateral_capital_1; stock_weights_relative_to_beginning_NAV",
            "cost_timing": "month_end_cash_payment_no_intra_month_notional_reduction",
            "borrow_timing": "month_end; annual_rate_divided_by_12_times_initial_short_notional",
            "cost_bps_one_way": float(cost_bps), "borrow_bps_annual": float(borrow_bps),
            "terminal_liquidation_included": False,
            "missing_returns_policy": "raise_for_every_held_stock",
            "missing_target_month_policy": "raise_after_strategy_start",
        },
    }


In [ ]:
"""Research orchestration, release scheduling, reports and artifact export."""
from pathlib import Path
import copy
import json
import hashlib
import platform
import time
import numpy as np
import pandas as pd


def sf_make_release_schedule(engine, macro_cfg):
    """When all inputs required for a quarterly model were available.

    An actual calendar, if supplied, is authoritative: unlisted quarters are
    unavailable, not filled with an assumed release. Without a calendar every
    row is explicitly marked as an assumed lag; this is never real-time data.
    """
    quarters = pd.period_range(engine.earliest_quarter, engine.latest_quarter, freq="Q")
    spec = macro_cfg.get("release_calendar")
    if spec and spec.get("path"):
        d = sf_io_read_table(spec, "macro_release_calendar")
        mapping = spec.get("columns", {"quarter": "quarter", "available_at": "available_at"})
        d = d.rename(columns={v: k for k, v in mapping.items()})
        if not {"quarter", "available_at"}.issubset(d):
            raise ValueError("macro release calendar needs quarter and available_at")
        d["quarter"] = pd.PeriodIndex(d.quarter.astype(str), freq="Q")
        d["available_at"] = d.available_at.map(lambda v: sf_io_timestamp(v, month_label_end=False))
        if d.quarter.duplicated().any() or d.available_at.isna().any():
            raise ValueError("Release calendar has duplicate quarters or missing availability")
        d = d.loc[d.quarter.isin(quarters), ["quarter", "available_at"]].sort_values("quarter")
        if d.empty:
            raise ValueError("No release-calendar quarter is inside the available macro history")
        d["release_source"] = "user_provided_model_bundle_calendar"
        d["release_is_assumed"] = False
    else:
        lag = int(macro_cfg.get("assumed_release_lag_months", 3))
        if lag < 1:
            raise ValueError("An assumed macro release lag must be at least one month")
        d = pd.DataFrame({"quarter": quarters})
        d["available_at"] = [(q.asfreq("M", "end")+lag).end_time for q in quarters]
        d["release_source"] = f"assumed_quarter_end_plus_{lag}_month_end"
        d["release_is_assumed"] = True
    qend = d.quarter.map(lambda q: q.end_time.normalize())
    if (d.available_at < qend).any():
        raise ValueError("A full-quarter model cannot be available before that quarter ends")
    return d.reset_index(drop=True)


def sf_available_model(schedule, date):
    usable = schedule.loc[schedule.available_at <= sf_io_asof_stamp(date)]
    if usable.empty:
        return None
    # A later publication of an older quarter cannot replace a newer model.
    return usable.sort_values(["quarter", "available_at"]).iloc[-1]


def sf_input_inventory(config):
    stock = sf_io_input_status(config["stocks"]).copy()
    stock["group"] = "stocks"
    raw = pd.DataFrame([{"group": "macro", "role": str(p.relative_to(Path(config["macro"]["input_dir"]).resolve())),
                         "path": str(p), "required": True, "configured": True, "exists": p.is_file()}
                        for p in macro_required_paths(config["macro"])])
    release_spec = config["macro"].get("release_calendar")
    if release_spec and release_spec.get("path"):
        p = Path(release_spec["path"]).expanduser()
        extra = pd.DataFrame([{"group": "macro", "role": "model_release_calendar",
                               "path": str(p), "required": True, "configured": True,
                               "exists": p.is_file()}])
        raw = pd.concat([raw, extra], ignore_index=True)
    return pd.concat([raw, stock], ignore_index=True, sort=False)


def sf_add_timing(frame, month, strategy, state, model_row):
    out = frame.copy()
    out["month"] = month+1
    out["strategy"] = strategy
    out["signal_month"] = month
    out["score_formation_month"] = state["formation_month"]
    out["model_version"] = state["model_version"]
    out["macro_end"] = str(model_row.quarter)
    out["available_at"] = model_row.available_at
    return out


def sf_run_research(config, stock_bundle=None, macro_engine=None, verbose=True):
    """Complete standalone raw-input → macro → stock → factor research pipeline.

    Optional injected objects are for tests or an interactive notebook session;
    normal operation reads all data from config and computes every model itself.
    """
    cfg = copy.deepcopy(config)
    started = time.perf_counter()
    stocks = stock_bundle if stock_bundle is not None else load_stock_bundle(cfg["stocks"])
    engine = macro_engine if macro_engine is not None else load_macro_inputs(cfg["macro"])
    releases = sf_make_release_schedule(engine, cfg["macro"])
    scfg, fcfg = cfg["sensitivity"], cfg["factor"]
    specs = cfg.get("factor_specs", SF_PRIMARY_SPECS)
    monthly = stocks["monthly_returns"].copy().sort_index()
    if monthly.empty:
        raise ValueError("No stock returns were loaded")
    cash = stocks["cash_monthly"].reindex(monthly.index)
    if cash.isna().any():
        raise ValueError("Collateral accounting needs cash for every stock-data month. Supply cash or explicitly select zero_explicit.")
    market = stocks["market_monthly"].reindex(monthly.index)
    annual = sf_quarter_end_returns12(monthly, cash, scfg.get("return_basis", "total"))
    # The market control has one fixed definition across formation and
    # post-ranking regressions, independent of the stock response basis.
    market12 = sf_quarter_end_returns12(market, cash, "excess_cash") if market.notna().any() else None
    first_hold = pd.Period(fcfg.get("start_month") or monthly.index.min(), freq="M")
    last_hold = min(pd.Period(fcfg.get("end_month") or monthly.index.max(), freq="M"), monthly.index.max())
    if last_hold < first_hold:
        raise ValueError("Factor end precedes start")
    methods = list(fcfg.get("score_methods", ["beta_shrunk", "beta_raw", "partial_corr"]))
    if not methods or len(set(methods)) != len(methods):
        raise ValueError("score_methods must be a nonempty list without duplicates")
    if not set(methods).issubset({"beta_shrunk", "beta_raw", "partial_corr"}):
        raise ValueError("Unknown score method")
    beta_rows, coverage_rows, audits, breakpoints = [], [], [], []
    labels_rows, target_rows, cell_rows, exclusions = [], [], [], []
    version_rows, version_news, version_metadata = [], [], []
    states, score_cache, used_versions = {}, {}, {}
    min_cell = int(fcfg.get("min_cell_size", 5))
    gross_target = float(fcfg.get("gross_target", 2.0))
    for held in pd.period_range(first_hold, last_hold, freq="M"):
        signal = held-1
        date = signal.end_time
        model_row = sf_available_model(releases, date)
        if model_row is None:
            audits.append({"month": held, "signal_month": signal, "status": "warmup_no_available_macro"})
            continue
        age = signal.ordinal-model_row.quarter.asfreq("M", "end").ordinal
        if age > int(cfg["macro"].get("max_model_age_months", 6)):
            message = f"{signal}: latest available model {model_row.quarter} is {age} months old"
            if states:
                raise ValueError(message+"; factor history is not silently bridged")
            audits.append({"month": held, "status": "warmup_stale_model", "detail": message})
            continue
        # Missing early universe history is a warmup limitation; after inception it is fatal.
        try:
            members = universe_at(stocks, date)
        except ValueError as exc:
            if states:
                raise
            audits.append({"month": held, "status": "warmup_no_universe", "detail": str(exc)})
            continue
        caps = caps_at(stocks, date)
        eligible = members.intersection(caps.index).intersection(monthly.columns)
        # Exclusions BEFORE beta estimation must remain visible too.  Otherwise
        # a missing cap/return column could disappear from every downstream table.
        absent_returns = members.difference(monthly.columns)
        absent_caps = members.difference(caps.index).difference(absent_returns)
        for ids, reason in [(absent_returns, "no_stock_return_column"),
                            (absent_caps, "no_available_positive_cap_within_max_age")]:
            if len(ids):
                exclusions.append(pd.DataFrame({"security_id": ids, "reason": reason,
                                                 "signal_month": signal, "strategy": "all_strategies",
                                                 "stage": "pre_estimation_eligibility"}))
        q = model_row.quarter
        if q not in used_versions:
            version = build_macro_version(engine, q)
            used_versions[q] = version
            meta = {**version["provenance"], "model_version": version["model_version"],
                    "macro_end": str(q), "available_at": str(model_row.available_at),
                    "release_source": model_row.release_source, "release_is_assumed": bool(model_row.release_is_assumed)}
            version_metadata.append(meta)
            nw = version["news"].copy()
            nw.index.name = "quarter"
            nw = nw.reset_index()
            nw["model_version"] = version["model_version"]
            version_news.append(nw)
            version_rows.append({"macro_end": str(q), "model_version": version["model_version"],
                                 "available_at": model_row.available_at, "first_signal_month": signal,
                                 "history_additivity_error": version["diagnostics"]["history_additivity_error"],
                                 "release_is_assumed": bool(model_row.release_is_assumed)})
            if verbose:
                print(f"Macro {q} available {str(model_row.available_at)[:10]} → formation {signal}; universe={len(members)}, eligible caps/returns={len(eligible)}")
        version = used_versions[q]
        if q not in score_cache:
            scores, coverage = sf_fit_sensitivities(annual, version["news"], eligible,
                                                    scfg, specs, q, market12)
            if not coverage.empty:
                coverage["formation_month"], coverage["model_version"] = signal, version["model_version"]
                coverage_rows.append(coverage)
            if not scores.empty:
                scores["formation_month"] = signal
                scores["model_version"] = version["model_version"]
                scores["macro_available_at"] = model_row.available_at
                scores = sf_shrink_betas(scores, int(scfg.get("min_cross_section", 5)))
                beta_rows.append(scores)
            score_cache[q] = {"scores": scores, "formation_month": signal}
        scores = score_cache[q]["scores"]
        for factor in specs:
            selected = scores.loc[scores.factor == factor] if not scores.empty else pd.DataFrame()
            for method in methods:
                strategy = f"{factor}|{method}"
                old = states.get(strategy)
                new = old is None or old["model_version"] != version["model_version"]
                try:
                    if selected.empty:
                        raise FactorFormationError(f"No eligible estimated betas for {factor} at {signal}")
                    if new:
                        formed = sf_form_2x3(selected, caps, members, score_column=method,
                                             min_cell_size=min_cell,
                                             score_quantiles=tuple(fcfg.get("score_quantiles", [0.30, 0.70])),
                                             gross_target=gross_target)
                        state = {"labels": formed["labels"], "formation_month": signal,
                                 "model_version": version["model_version"]}
                        breakpoints.append({"signal_month": signal, "strategy": strategy, **formed["breakpoints"]})
                    else:
                        state = old
                        formed = sf_reweight_2x3(state["labels"], caps, members,
                                                min_cell_size=min_cell, gross_target=gross_target)
                    if not formed["exclusions"].empty:
                        ex = formed["exclusions"].copy()
                        ex["signal_month"], ex["strategy"] = signal, strategy
                        exclusions.append(ex)
                    states[strategy] = state
                    fw = formed["factor_weights"]
                    base_weights = fw.rename("weight").rename_axis("security_id").reset_index()
                    target_rows.append(sf_add_timing(base_weights, signal, strategy, state, model_row))
                    cell_rows.append(sf_add_timing(formed["cell_weights"], signal, strategy, state, model_row))
                    labels_rows.append(sf_add_timing(formed["labels"], signal, strategy, state, model_row))
                    audits.append({"month": held, "signal_month": signal, "strategy": strategy, "status": "formed" if new else "reweighted",
                                   "macro_end": str(q), "n_universe": len(members), "n_cap_eligible": len(eligible),
                                   "n_factor_eligible": len(formed["labels"]), "release_is_assumed": bool(model_row.release_is_assumed),
                                   "min_cell_n": int(formed["cell_counts"].min())})
                    # Optional neutrality overlays use the same point-in-time universe.
                    # These are minimum-distance projections, not Fama-French sorts.
                    if method == "beta_shrunk":
                        for neutral in fcfg.get("neutral_variants", []):
                            if neutral not in {"sector", "market", "sector_market"}:
                                raise ValueError("neutral_variants must contain sector/market/sector_market")
                            sectors = sectors_at(stocks, date) if "sector" in neutral else None
                            mb = sf_exante_market_betas(monthly, market, signal,
                                                        fcfg.get("market_beta_window_months", 60),
                                                        fcfg.get("market_beta_min_months", 36)) if "market" in neutral else None
                            nw = sf_project_neutral_weights(fw, sectors, mb) * (gross_target/2.0)
                            nt = nw.rename("weight").rename_axis("security_id").reset_index()
                            target_rows.append(sf_add_timing(nt, signal, strategy+"|"+neutral, state, model_row))
                except FactorFormationError as exc:
                    if old is not None:
                        raise FactorFormationError(f"{held} {strategy}: an already-started factor cannot be formed; {exc}") from exc
                    audits.append({"month": held, "signal_month": signal, "strategy": strategy,
                                   "status": "warmup_insufficient_formation", "detail": str(exc)})
    concat = lambda xs: pd.concat(xs, ignore_index=True) if xs else pd.DataFrame()
    targets, cells = concat(target_rows), concat(cell_rows)
    # Latest descriptive table uses the current vintage, and is never backfilled
    # into historical formations. Its macro quarter can postdate public availability.
    latest = build_macro_version(engine, engine.latest_quarter)
    latest_month = monthly.index.max()
    latest_members = universe_at(stocks, latest_month.end_time)
    latest_eligible = latest_members.intersection(caps_at(stocks, latest_month.end_time).index).intersection(monthly.columns)
    descriptive, descriptive_coverage = sf_fit_sensitivities(annual, latest["news"], latest_eligible,
                                                            scfg, SF_DESCRIPTIVE_SPECS, engine.latest_quarter, market12)
    joint, joint_coverage = sf_fit_sensitivities(annual, latest["news"], latest_eligible,
                                               scfg, specs, engine.latest_quarter, market12)
    joint = sf_shrink_betas(joint, int(scfg.get("min_cross_section", 5)))
    backtest = None
    diagnostics = pd.DataFrame()
    if not targets.empty:
        backtest = sf_run_factor_backtest(monthly, targets, cash,
                                          cost_bps=float(fcfg.get("cost_bps_one_way", 10)),
                                          borrow_bps=float(fcfg.get("borrow_bps_annual", 0)),
                                          cell_weights_long=cells,
                                          expected_gross=gross_target)
        diagnostics = sf_postranking_diagnostics(backtest, latest["news"], specs,
                                                  int(cfg.get("diagnostics", {}).get("min_quarters", 12)),
                                                  market_monthly=market,
                                                  control_market=bool(scfg.get("control_market", False)),
                                                  bootstrap_reps=int(cfg.get("diagnostics", {}).get("bootstrap_reps", 399)),
                                                  block_length=int(cfg.get("diagnostics", {}).get("block_length", 8)),
                                                  seed=int(cfg.get("diagnostics", {}).get("seed", 20261111)))
    result = {
        "config": cfg, "stocks": stocks, "macro_engine": engine, "latest_macro": latest,
        "release_schedule": releases, "macro_versions": pd.DataFrame(version_rows),
        "macro_version_metadata": version_metadata, "macro_news_history": concat(version_news),
        "stock_sensitivity_main": descriptive, "stock_sensitivity_joint": joint,
        "latest_coverage_main": descriptive_coverage, "latest_coverage_joint": joint_coverage,
        "stock_beta_history": concat(beta_rows), "stock_beta_coverage": concat(coverage_rows),
        "factor_breakpoints": pd.DataFrame(breakpoints), "factor_membership": concat(labels_rows),
        "factor_weights": targets, "factor_cell_weights": cells,
        "formation_audit": pd.DataFrame(audits), "formation_exclusions": concat(exclusions),
        "backtest": backtest, "postranking_sensitivity": diagnostics,
        "elapsed_seconds": time.perf_counter()-started,
        "research_status": "recursive_current_vintage_not_real_time",
        "empirical_data": cfg.get("data_label", "user_provided") != "synthetic_validation",
        "limitations": [
            "Macro raw data are the supplied current vintage; prefix refitting does not reconstruct historical statistical revisions.",
            "Without a user release calendar, macro bundle availability is an explicit lag assumption.",
            "Monthly close-to-close accounting does not implement next-day or two-business-day execution.",
            "Trading costs are paid at month-end; lending, margin and financing conventions are simplified.",
            "Post-ranking HAC inference is conditional on generated macro news and the realized selection history.",
            "Real input files were not supplied at notebook delivery; delivery validation used synthetic fixtures only.",
        ],
    }
    if backtest is not None:
        result.update(sf_additional_diagnostics(result))
    if verbose:
        print(f"Completed in {result['elapsed_seconds']:.1f}s; beta rows={len(result['stock_beta_history'])}, monthly target rows={len(targets)}")
    return result


def sf_additional_diagnostics(result):
    """Post-ranking cell monotonicity, market loading and same-period comparisons."""
    bt = result["backtest"]
    specs = result["config"].get("factor_specs", SF_PRIMARY_SPECS)
    news = result["latest_macro"]["news"]
    monthly = bt["monthly"]
    market = result["stocks"]["market_monthly"]
    cash = result["stocks"]["cash_monthly"]
    control_market = bool(result["config"].get("sensitivity", {}).get("control_market", False))
    market12 = (sf_quarter_end_returns12(market, cash, "excess_cash")
                if control_market else None)
    rows = []
    for strategy, g in monthly.groupby("strategy"):
        s = g.set_index("month").sort_index()
        x = market.reindex(s.index)-cash.reindex(s.index)
        d = pd.concat([s.excess_return_net.rename("y"), x.rename("m")], axis=1).dropna()
        if len(d) >= 24 and d.m.std() > 1e-12:
            X = np.column_stack([np.ones(len(d)), d.m])
            fit = sf_ols_hac(d.y, X, lags=6)
            rows.append({"strategy": strategy, "n_months": len(d), "market_beta": float(fit["beta"][1]),
                         "market_beta_low": float(fit["low"][1]), "market_beta_high": float(fit["high"][1]),
                         "alpha_annual_arithmetic": float(fit["beta"][0]*12),
                         "alpha_p_hac6": float(fit["p"][0])})
    # Compare strategies on exactly the same realized monthly interval.
    firsts = monthly.groupby("strategy").month.min()
    lasts = monthly.groupby("strategy").month.max()
    common_start, common_end = firsts.max(), lasts.min()
    common = monthly.loc[monthly.month.between(common_start, common_end)].copy()
    common_summary = sf_factor_performance(common) if len(common) else pd.DataFrame()
    if not common_summary.empty:
        common_summary["common_start"] = str(common_start)
        common_summary["common_end"] = str(common_end)
    # Six long-only cells plus size-balanced L/N/H. Fees not applied to these
    # diagnostic leg series; they are explicitly gross-return diagnostics.
    cell = bt.get("cell_returns", pd.DataFrame())
    mono = []
    if not cell.empty:
        value_col = next((x for x in ["return", "cell_return", "gross_return"] if x in cell), None)
        if value_col is None:
            raise ValueError("Accounting cell return schema changed")
        for strategy, g in cell.groupby("strategy"):
            wide = g.pivot(index="month", columns="cell", values=value_col).sort_index()
            wide.index = pd.PeriodIndex(wide.index, freq="M")
            for group in ["L", "N", "H"]:
                wide[group] = 0.5*(wide["S"+group]+wide["B"+group])
            r12 = sf_quarter_end_returns12(wide, cash, "excess_cash")
            factor = strategy.split("|")[0]
            if factor not in specs:
                continue
            fitted, cov = sf_fit_sensitivities(r12, news, list(wide.columns),
                                               {"min_quarters": max(12, result["config"].get("diagnostics", {}).get("min_quarters", 12)),
                                                "estimator": "ols", "return_basis": "excess_cash", "control_market": control_market,
                                                "max_last_observation_gap_quarters": 1000},
                                               {factor: specs[factor]}, news.index.max(), market12)
            if not fitted.empty:
                fitted = fitted.rename(columns={"security_id": "cell"})
                fitted["strategy"] = strategy
                fitted["accounting_basis"] = "gross_long_only_cell_excess_cash_12m"
                mono.append(fitted)
    # Factor-factor correlation does not imply macro-factor orthogonality.
    fp = monthly.pivot(index="month", columns="strategy", values="excess_return_net")
    correlations = fp.loc[common_start:common_end].corr(min_periods=12)
    return {"market_diagnostics": pd.DataFrame(rows), "common_period_performance": common_summary,
            "cell_postranking_sensitivity": pd.concat(mono, ignore_index=True) if mono else pd.DataFrame(),
            "factor_correlation_common_period": correlations}


def sf_export_results(result, output_root):
    """New run directory only. Never writes to the original macro inputs."""
    root = Path(output_root).expanduser().resolve()
    inputs = Path(result["config"]["macro"]["input_dir"]).expanduser().resolve()
    if root == inputs or inputs in root.parents:
        raise ValueError("Output directory must not be inside the original inputs")
    stamp = pd.Timestamp.now(tz="Asia/Tokyo").strftime("%Y%m%dT%H%M%S_%f")
    dest = root/stamp
    dest.mkdir(parents=True, exist_ok=False)
    written = []
    for key, value in result.items():
        if isinstance(value, pd.DataFrame):
            value.to_csv(dest/(key+".csv"), index=(key=="factor_correlation_common_period"), encoding="utf-8-sig")
            written.append(key+".csv")
    if result["backtest"] is not None:
        for key, value in result["backtest"].items():
            if isinstance(value, pd.DataFrame):
                name = "factor_"+key+".csv"
                value.to_csv(dest/name, index=False, encoding="utf-8-sig")
                written.append(name)
    latest = result["latest_macro"]
    for key in ["news", "coefficients", "bb_input", "inflation_contributions"]:
        if isinstance(latest.get(key), pd.DataFrame):
            latest[key].to_csv(dest/("latest_macro_"+key+".csv"), encoding="utf-8-sig")
            written.append("latest_macro_"+key+".csv")
    metadata = {"status": "completed", "research_status": result["research_status"],
                "empirical_data": bool(result.get("empirical_data", True)),
                "data_label": result["config"].get("data_label", "user_provided"),
                "config": result["config"], "elapsed_seconds": result["elapsed_seconds"],
                "macro_input_manifest": result["macro_engine"].input_manifest,
                "stock_input_audit": result["stocks"]["audit"],
                "macro_versions": result["macro_version_metadata"],
                "latest_macro_diagnostics": latest["diagnostics"],
                "limitations": result["limitations"],
                "accounting": result["backtest"].get("accounting") if result["backtest"] is not None else None,
                "environment": {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__},
                "files": written}
    (dest/"run_metadata.json").write_text(json.dumps(metadata, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
    return dest


In [ ]:
"""Notebook-only tables and figures; no fabricated empirical output."""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
from IPython.display import display, Markdown


def sf_configure_figures():
    fonts = {f.name for f in font_manager.fontManager.ttflist}
    for name in ["Noto Sans CJK JP", "Hiragino Sans", "Yu Gothic", "Meiryo", "IPAexGothic"]:
        if name in fonts:
            plt.rcParams["font.family"] = name
            break
    plt.rcParams.update({"axes.unicode_minus": False, "figure.dpi": 110,
                         "axes.spines.top": False, "axes.spines.right": False,
                         "axes.grid": True, "grid.alpha": 0.2, "font.size": 10})


def sf_show_results(result, top_n=25):
    """Core findings stay inside the notebook; full tables remain in RESULT."""
    sf_configure_figures()
    stocks, bt = result["stocks"], result["backtest"]
    r = stocks["monthly_returns"]
    empirical = result.get("empirical_data", True)
    title = "TOPIX500 インフレ感応度・ファクター分析" if empirical else "合成データによる統合検証（投資結果ではありません）"
    display(Markdown("## "+title))
    display(pd.DataFrame([
        {"項目": "株式データ期間", "値": f"{r.index.min()} – {r.index.max()}"},
        {"項目": "履歴全体の銘柄ID数", "値": r.shape[1]},
        {"項目": "BBの最新完全四半期", "値": str(result["latest_macro"]["macro_end"])},
        {"項目": "構築に使ったBBモデル版数", "値": len(result["macro_versions"])},
        {"項目": "マクロの時点管理", "値": "現在ビンテージを各過去時点で再推定（統計改定は再現しない）"},
        {"項目": "公表日時", "値": "仮定ラグ" if result["release_schedule"].release_is_assumed.any() else "ユーザー指定カレンダー"},
        {"項目": "現金収益", "値": stocks["audit"].get("cash_assumption", "入力された現金収益率")},
        {"項目": "会計", "値": "月次・担保資本1・グロス"+str(result["config"]["factor"].get("gross_target", 2))+"、費用は月末控除"},
    ]))
    display(Markdown("### 1. 最新の銘柄別感応度\n数値は現行ビンテージによる記述分析です。歴史的な銘柄選定にこの表を遡及利用していません。"))
    main = result["stock_sensitivity_main"]
    joint = result["stock_sensitivity_joint"]
    if main.empty:
        display(Markdown("最低観測数を満たす銘柄がありません。`RESULT['latest_coverage_main']`を確認してください。"))
    else:
        cols = [x for x in ["security_id", "factor", "beta_raw", "partial_corr", "beta_per_1sd", "se_hac8", "q_bh_all_factors_hac8", "n"] if x in main]
        display(main[cols].sort_values(["factor", "beta_raw"], ascending=[True, False]).groupby("factor", group_keys=False).head(min(top_n, 10)).round(4))
    if not joint.empty:
        factors = list(result["config"].get("factor_specs", SF_PRIMARY_SPECS))
        fig, axes = plt.subplots(1, len(factors), figsize=(4.2*len(factors), 3.3), squeeze=False)
        for ax, factor in zip(axes.ravel(), factors):
            s = joint.loc[joint.factor == factor]
            ax.hist(s.beta_raw.dropna(), bins=25, alpha=.45, label="Raw beta", color="#8293A8")
            ax.hist(s.beta_shrunk.dropna(), bins=25, alpha=.65, label="Shrunk beta", color="#236B8E")
            ax.axvline(0, color="#334155", lw=.8)
            ax.set(title=factor, xlabel="12M return % / 1pp news", ylabel="Stocks")
        axes.ravel()[0].legend(fontsize=8)
        fig.suptitle("Latest conditional inflation sensitivity", y=1.02)
        fig.tight_layout(); display(fig); plt.close(fig)
    display(Markdown("### 2. 形成履歴・データ充足状況"))
    audit = result["formation_audit"]
    if not audit.empty:
        display(audit.groupby("status").size().rename("rows").to_frame())
        available = [x for x in ["month", "strategy", "macro_end", "n_universe", "n_factor_eligible", "min_cell_n", "release_is_assumed", "status"] if x in audit]
        display(audit[available].tail(18))
    if bt is None:
        display(Markdown("**形成可能な保有期間がまだありません。** 0収益で埋めていません。βの最低40四半期、6セルの最低銘柄数、入力履歴を確認してください。"))
        return
    display(Markdown("### 3. ファクターの損益・リスク\n正のインフレ感応度と正の平均収益は別に評価します。以下の富系列は現金担保を含む定義です。"))
    display(bt["annualized_summary"].round(4))
    if not result.get("common_period_performance", pd.DataFrame()).empty:
        display(Markdown("**全ファクターの共通評価期間**"))
        display(result["common_period_performance"].round(4))
    m = bt["monthly"]
    primary = m.loc[m.strategy.str.endswith("|beta_shrunk")]
    if not primary.empty:
        fig, ax = plt.subplots(figsize=(10, 4))
        colors = ["#236B8E", "#C77D37", "#31856D"]
        for (strategy, g), color in zip(primary.groupby("strategy", sort=False), colors):
            ax.plot(pd.PeriodIndex(g.month, freq="M").to_timestamp("M"), g.nav_net,
                    label=strategy.split("|")[0], color=color, lw=1.7)
        ax.axhline(1, color="#64748B", lw=.8, ls="--")
        ax.set(title="Collateral wealth after trading and borrow costs", ylabel="Initial capital = 1", xlabel="Holding month")
        ax.legend(); fig.tight_layout(); display(fig); plt.close(fig)
    display(Markdown("### 4. 形成後のニュース感応度\n12か月のファクター超過収益を同時期のニュースに回帰します。HAC・ブートストラップは生成済みニュースと形成結果に条件付けた推論です。"))
    d = result["postranking_sensitivity"]
    if d.empty:
        display(Markdown("形成後の12か月感応度を計算できる期間がありません。"))
    else:
        columns = [c for c in ["strategy", "return_kind", "n", "beta", "partial_corr", "ci_low_hac8", "ci_high_hac8", "q_bh_hac8", "bootstrap_ci_low", "bootstrap_ci_high", "bootstrap_status", "status"] if c in d]
        display(d[columns].round(4))
        good = d.loc[(d.status == "ok") & d.return_kind.eq("excess_return12_net") & d.strategy.str.endswith("|beta_shrunk")]
        if not good.empty:
            fig, ax = plt.subplots(figsize=(8, 3.4))
            x = np.arange(len(good))
            low = np.maximum(0, good.beta-good.ci_low_hac8)
            high = np.maximum(0, good.ci_high_hac8-good.beta)
            ax.errorbar(x, good.beta, yerr=np.vstack([low, high]), fmt="o", capsize=5, color="#236B8E")
            ax.axhline(0, color="#64748B", lw=.9)
            ax.set(xticks=x, xticklabels=[s.split("|")[0] for s in good.strategy],
                   ylabel="12M excess return % / 1pp news", title="Post-ranking target beta and HAC 95% interval")
            fig.tight_layout(); display(fig); plt.close(fig)
    mono = result.get("cell_postranking_sensitivity", pd.DataFrame())
    if not mono.empty:
        display(Markdown("### 5. Low → Neutral → High の単調性\n各セルは費用前のロングオンリー系列。High群自体が正の感応度かどうかも確認できます。"))
        z = mono.loc[mono.cell.isin(["L", "N", "H"]) & mono.strategy.str.endswith("|beta_shrunk")]
        display(z[["strategy", "cell", "beta_raw", "partial_corr", "n", "ci_low_hac8", "ci_high_hac8"]].round(4))
    if not result.get("market_diagnostics", pd.DataFrame()).empty:
        display(Markdown("### 6. 市場への感応度\nネット投資額ゼロやサイズ調整は、市場βゼロを保証しません。"))
        display(result["market_diagnostics"].round(4))
    display(Markdown("### 全結果へのアクセス\n`RESULT`に全DataFrame、`RESULT['backtest']`に会計・損益、`RESULT['macro_engine']`にBB入力監査が残ります。CSV出力を有効化すると、入力を変更せず新しいrunディレクトリへ保存します。"))


def sf_run_self_checks():
    """Fast deterministic math/clock checks; synthetic data never enter RESULT."""
    rng = np.random.default_rng(81204)
    records = []
    def ok(name, value):
        if not bool(value):
            raise AssertionError(name)
        records.append({"check": name, "status": "PASS", "data": "synthetic_only"})
    idx = pd.period_range("2000Q1", periods=64, freq="Q")
    x, g = rng.normal(size=(2, 64))
    X = np.column_stack([np.ones(64), x, g])
    y = pd.Series(.3+1.7*x-.6*g, index=idx)
    fit = sf_ols_hac(y, X, 8)
    ok("Reused OLS/HAC coefficient", np.allclose(fit["beta"], [.3, 1.7, -.6], atol=1e-10))
    r = pd.Series([.02]*11+[-1.0], index=pd.period_range("2020-01", periods=12, freq="M"))
    ok("True -100% return is retained", np.isclose(sf_rolling_compound(r).iloc[-1], -1))
    r.iloc[2] = np.nan
    ok("Incomplete 12M windows remain missing", pd.isna(sf_rolling_compound(r).iloc[-1]))
    ids = [f"S{i:03d}" for i in range(60)]
    caps = pd.Series(np.repeat([1., 10.], 30), index=ids)
    scores = pd.DataFrame({"security_id": ids, "beta_shrunk": np.tile(np.arange(30), 2)})
    formed = sf_form_2x3(scores, caps, min_cell_size=3)
    w = formed["factor_weights"]
    ok("2x3 factor net 0 and gross 2", np.isclose(w.sum(), 0) and np.isclose(w.abs().sum(), 2))
    score = scores.set_index("security_id").beta_shrunk
    ok("High-minus-low score direction", float(w @ score.reindex(w.index)) > 0)
    target = pd.DataFrame({"security_id": ids, "weight": w.reindex(ids).to_numpy(),
                           "month": pd.Period("2020-02", "M"), "strategy": "test",
                           "signal_month": pd.Period("2020-01", "M")})
    ret = pd.DataFrame([np.linspace(-.02, .03, 60)], index=pd.PeriodIndex(["2020-02"], freq="M"), columns=ids)
    cash = pd.Series(.001, index=ret.index)
    bt = sf_run_factor_backtest(ret, target, cash, cost_bps=10)
    expected = .001+float(w @ ret.iloc[0])-0.001*2
    ok("Collateral cash and bilateral trading cost", np.isclose(bt["monthly"].total_return_net.iloc[0], expected))
    future = target.copy(); future["signal_month"] = pd.Period("2020-02", "M")
    rejected = False
    try:
        sf_run_factor_backtest(ret, future, cash)
    except ValueError:
        rejected = True
    ok("Future signal rejected", rejected)
    return pd.DataFrame(records)


## 7. 入力確認と実行

最初に短い合成データの自己検査を行います。そのデータは実分析へ混ぜません。不足ファイルはまとめて表示し、架空データで置換せず停止します。

In [ ]:
if RUN_SELF_CHECKS:
    SELF_CHECK_RESULTS = sf_run_self_checks()
    display(SELF_CHECK_RESULTS)
if WRITE_EMPTY_INPUT_TEMPLATES:
    template_paths = write_input_templates(STOCK_DIR, enabled=True)
    display(pd.DataFrame({'created_empty_template': template_paths}))


In [ ]:
INPUT_STATUS = sf_input_inventory(CONFIG)
display(INPUT_STATUS[['group','role','required','configured','exists','path']])
if RUN_ANALYSIS:
    bad = INPUT_STATUS.loc[(INPUT_STATUS.required | INPUT_STATUS.configured) & ~INPUT_STATUS.exists]
    if not bad.empty:
        raise FileNotFoundError('設定セルのパスを修正し、必要な実データを配置してください。\n' + '\n'.join(bad.path.astype(str)))


In [ ]:
RESULT = None
if RUN_ANALYSIS:
    RESULT = sf_run_research(CONFIG, verbose=True)
    if SAVE_RESULTS:
        RUN_OUTPUT_DIR = sf_export_results(RESULT, OUTPUT_ROOT)
        print('結果保存先:', RUN_OUTPUT_DIR)


## 8. 分析結果

表・図は入力データから生成します。最新の記述分析と、過去時点で形成したファクターの成績を区別して表示します。

In [ ]:
if RESULT is not None:
    sf_show_results(RESULT, top_n=25)
else:
    display(Markdown('`RUN_ANALYSIS=True`で実分析を実行すると、ここに結果を表示します。'))


### 詳細テーブルの確認例

必要な行のコメントを外して利用してください。`RESULT`には全銘柄・全時点が残ります。

In [ ]:
# display(RESULT['stock_sensitivity_joint'].query("factor == 'cost_push'").sort_values('beta_shrunk', ascending=False).head(30))
# display(RESULT['stock_beta_history'].tail(30))
# display(RESULT['factor_breakpoints'].tail(18))
# display(RESULT['formation_exclusions'].tail(30))
# display(RESULT['factor_weights'].tail(30))
# display(RESULT['factor_correlation_common_period'])
# display(RESULT['backtest']['monthly'].tail(18))
# display(RESULT['macro_engine'].coverage)
# display(RESULT['latest_macro']['coefficients'])


## 9. 検証結果と実データで確認する範囲

納品前に次の検証を行っています。各検査は合成データまたは独立した数値計算との照合であり、TOPIX500の実証結果ではありません。

| 検証対象 | 確認内容 |
|---|---|
| BB/FTPT移植 | 合成原系列から再構築したBB入力・Newsが元関数と数値許容誤差内で一致。寄与の加法性 |
| マクロ時点管理 | 過去cutoff以降の原系列変更が過去版へ影響しない。原入力のハッシュが不変 |
| 入力端点 | 未完成月・四半期、Google完了月、内部欠損、CPI基準不一致を検査 |
| 株式I/O | CSV/Excel/Parquet、ID、縦横形式、caps利用可能日、構成区間・snapshot、日次・月次欠損 |
| 回帰・推論 | OLS/WLS、HAC、BHを独立計算・statsmodelsと照合。将来リターン変更への不変性 |
| 因子構築 | 2×3のnet/gross・同値・不足セル・月次更新・セルとH−L会計の一致 |
| 損益会計 | 現金担保、−100%銘柄損失、ドリフト、売買費用、借株料、欠損保有リターンの拒否 |
| 形成後検証 | 実際の会計出力から正しい12か月感応度を回収。連続ブロックのみbootstrap |
| 統合実行 | 合成canonical BB入力と株式ファイルから全14コードセルを順次実行し、12戦略・25マクロモデル版・図表・CSV出力を確認 |

実行検証は、全コードセルを同一の新規Python名前空間で順に実行し、表・図の出力を取得する方法で実施しました。

実ファイルの列名・シート・単位・銘柄ID・配当調整定義、実際の公表ビンテージ、銘柄入替・上場廃止処理は、入力後に監査表で確認してください。配当込みかどうかを数値だけから自動認定する処理はありません。金融分析の「有意性」だけでデータの意味を保証しません。

## 10. 参考文献・元コードの出典

- 添付 `main(1).ipynb`：原系列読み込み、BB制約回帰、FT/PTの5方程式・歴史分解・NewsOnly、15節のHAC・偏相関・BH補正。元NotebookのSHA256を `BB_SOURCE_NOTEBOOK_SHA256` に記録しています。
- AQR (2026), *Inflation Redux?*, pp.5–6, 19. [原資料](https://www.aqr.com/Insights/Research/Alternative-Thinking/Inflation-Redux)。AQRの予想サプライズとの合成指標を、そのまま日本BBニュースと同一視していません。
- Boons, Duarte, de Roon and Szymanowska (2020), *Time-Varying Inflation Risk and Stock Returns*, JFE 136(2), 444–470. [著者版](https://www.newyorkfed.org/medialibrary/media/research/economists/duarte/inflation_NRC_JFE2019.pdf)。インフレβ、縮小、サイズ調整によるH−Lの先例。本Notebookの四半期BB・2×3は適用設計です。
- Esakia and Goltz (2023), *Targeting Macroeconomic Exposures in Equity Portfolios*, FAJ 79(1), 37–57. [CFA Institute](https://rpc.cfainstitute.org/research/financial-analysts-journal/2022/targeting-macroeconomic-exposures-in-equity-portfolios)。個別株の推定マクロ露出と形成後の頑健性。四半期ニュースの前方補完による擬似的な週次観測増加は行いません。
- Ang, Brière and Signori (2012), *Inflation and Individual Equities*, FAJ 68(4), 36–55. [論文紹介](https://rpc.cfainstitute.org/research/financial-analysts-journal/2012/inflation-and-individual-equities)。個別株インフレβの時間変動とOOSの難しさ。
- Kenneth French Data Library, [Developed-market factors](https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/Data_Library/f-f_developed.html)。独立2×3・セル内時価総額加重・サイズ間平均の構成。公開日本系列のUSD/RF基準をJPY分析へ無調整で混ぜません。

### 今回の実装範囲

同時感応度・形成後の感応度ファクター・市場/業種の追加診断までを実装しています。インフレそのものの予測、動的なレジーム配分、リスクプレミアムの因果的識別、500銘柄を直接使う最適追随ポートフォリオ、全段階を再推定するbootstrapは別の研究課題です。